# Train the DALL-E 2 Diffusion Prior

Antonio Esteves @ UMinho, June 2025

## Acknowledgements

* https://github.com/lucidrains, the dall-e 2 model source code.

* https://github.com/Veldrovive, for providing pretrarained prior and decoder

* https://github.com/rom1504, for multi-node training contribution.

This model was trained using the aesthetic subset of the [LAION2B dataset](https://laion.ai/blog/laion-5b/), details of the run can be found on [wandb](https://wandb.ai/nousr_laion/dalle2_train_decoder/reports/Decoder-Training--VmlldzoyMjEyMjcw).

It is missing the upsamplers so it can only produce 64x64 images and has only be trained for 0.5% of what OpenAI did for their DALLE2. It can still produce some impressive results, but is not nearly at the level you might see on the news yet.

While the upsamplers have yet to be trained, we have substituted [SwinIR](https://github.com/JingyunLiang/SwinIR) to produce 256x256 images. This tends to flatten details, but still generally improves the quality of the output.

In [ ]:
import sys
import os
import json
import shutil
from   shutil                   import rmtree
import torch
import importlib
import math
from   math                     import ceil, sqrt
import random
from   random                   import choice
import ftfy
import urllib.request
import time
import copy
import fsspec

from   datetime                 import timedelta
from   itertools                import zip_longest
from   typing                   import Any, Dict, Optional, List, Tuple, TypeVar, Union
from   pydantic                 import BaseModel, validator, model_validator
from   packaging                import version
import regex                    as     re
import ipywidgets               as     widgets
from   tqdm.auto                import tqdm
from   functools                import lru_cache, partial, wraps
from   contextlib               import contextmanager, nullcontext
from   collections              import namedtuple
from   collections.abc          import Iterable
from   pathlib                  import Path
import numpy                    as     np
from   PIL                      import Image
import matplotlib.pyplot        as     plt

import torch
import torchvision
from   torch                    import nn, einsum
import torch.nn.functional      as     F
from   torchvision.utils        import make_grid, save_image
from   torch.utils.checkpoint   import checkpoint
from   torch                    import nn, einsum
import torchvision.transforms   as     T
from   torch.utils              import data
from   torch.utils.data         import Dataset, DataLoader, IterableDataset, random_split
from   torchvision.datasets     import ImageFolder
from   torch.optim              import AdamW, Adam
from   torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR
from   torch.cuda.amp           import autocast, GradScaler
from   torch.autograd           import grad as torch_grad

from   torchmetrics.image.fid   import FrechetInceptionDistance
from   torchmetrics.image.inception import InceptionScore
from   torchmetrics.image.kid   import KernelInceptionDistance
from   torchmetrics.image.lpip  import LearnedPerceptualImagePatchSimilarity

from   einops                   import rearrange, repeat, reduce, pack, unpack
from   einops.layers.torch      import Rearrange
from   kornia.filters           import gaussian_blur2d
import kornia.augmentation      as     K
from   resize_right             import resize
import webdataset               as     wds

from   accelerate               import Accelerator, DistributedType, \
                                       DistributedDataParallelKwargs, \
                                       InitProcessGroupKwargs
from   accelerate.utils         import dataclasses as accelerate_dataclasses
from   accelerate.utils         import set_seed

import pytorch_warmup           as     warmup
from   ema_pytorch              import EMA
from   vector_quantize_pytorch  import VectorQuantize as VQ
from   embedding_reader         import EmbeddingReader

# rotary embeddings

from   rotary_embedding_torch   import RotaryEmbedding

# if using x-clip version of CLIP

from   x_clip                   import CLIP
#from   x_clip                  import CLIP as XCLIP
from   open_clip                import list_pretrained
from   coca_pytorch             import CoCa

from   clip                     import tokenize


In [ ]:
!nvidia-smi

print(f'GPU 0: {torch.cuda.get_device_name(0)}')
print(f'GPU 1: {torch.cuda.get_device_name(1)}')

In [ ]:
device           = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
inference_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

config_file = "../config/dalle2_train_prior_config_04.json"

os.environ["os.PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]    = "max_split_size_mb:512"

# constants

NAT               = 1. / math.log(2.)
DEFAULT_DATA_PATH = './.tracker-data'
__version__       = '1.15.6'

# types

UnetOutput         = namedtuple('UnetOutput', ['pred', 'var_interp_frac_unnormalized'])
InnerType          = TypeVar('InnerType')
ListOrTuple        = Union[List[InnerType], Tuple[InnerType]]
SingularOrIterable = Union[InnerType, ListOrTuple[InnerType]]
MList              = nn.ModuleList

In [ ]:
cos = nn.CosineSimilarity(dim=1, eps=1e-6)

def exists(val):
    return val is not None

def all_between(values: list, lower_bound, upper_bound):
    for value in values:
        if value < lower_bound or value > upper_bound:
            return False

    return True

def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d


def default2(val, d):
    return val if exists(val) else d


def identity(t, *args, **kwargs):
    return t


def first(arr, d = None):
    if len(arr) == 0:
        return d
    return arr[0]


def maybe(fn):
    @wraps(fn)
    def inner(x, *args, **kwargs):
        if not exists(x):
            return x
        return fn(x, *args, **kwargs)
    return inner


def cast_tuple(val, length = None, validate = True):
    if isinstance(val, list):
        val = tuple(val)

    out = val if isinstance(val, tuple) else ((val,) * default(length, 1))

    if exists(length) and validate:
        assert len(out) == length

    return out


def cast_tuple3(val, length = 1):
    return val if isinstance(val, tuple) else ((val,) * length)


def module_device(module):
    if isinstance(module, nn.Identity):
        return 'cpu' # It doesn't matter
    return next(module.parameters()).device


def zero_init_(m):
    nn.init.zeros_(m.weight)
    if exists(m.bias):
        nn.init.zeros_(m.bias)

@contextmanager
def null_context(*args, **kwargs):
    yield


def eval_decorator(fn):
    def inner(model, *args, **kwargs):
        was_training = model.training
        model.eval()
        out = fn(model, *args, **kwargs)
        model.train(was_training)
        return out
    return inner


def is_float_dtype(dtype):
    return any([dtype == float_dtype for float_dtype in (torch.float64, torch.float32, torch.float16, torch.bfloat16)])

def is_list_str(x):
    if not isinstance(x, (list, tuple)):
        return False
    return all([type(el) == str for el in x])


def pad_tuple_to_length(t, length, fillvalue = None):
    remain_length = length - len(t)
    if remain_length <= 0:
        return t
    return (*t, *((fillvalue,) * remain_length))

def noop(*args, **kwargs):
    pass


def cycle(dl):
    while True:
        for data in dl:
            yield data


def yes_or_no(question):
    answer = input(f'{question} (y/n) ')
    return answer.lower() in ('yes', 'y')


def accum_log(log, new_logs):
    for key, new_value in new_logs.items():
        old_value = log.get(key, 0.)
        log[key] = old_value + new_value
    return log

## Helper functions

## Checkpointing helper function

In [ ]:
def make_checkpointable(fn, **kwargs):
    if isinstance(fn, nn.ModuleList):
        return [maybe(make_checkpointable)(el, **kwargs) for el in fn]

    condition = kwargs.pop('condition', None)

    if exists(condition) and not condition(fn):
        return fn

    @wraps(fn)
    def inner(*args):
        input_needs_grad = any([isinstance(el, torch.Tensor) and el.requires_grad for el in args])

        if not input_needs_grad:
            return fn(*args)

        return checkpoint(fn, *args)

    return inner

## Control freezing of CLIP

In [ ]:
def set_module_requires_grad_(module, requires_grad):
    for param in module.parameters():
        param.requires_grad = requires_grad

def freeze_all_layers_(module):
    set_module_requires_grad_(module, False)

def unfreeze_all_layers_(module):
    set_module_requires_grad_(module, True)

def freeze_model_and_make_eval_(model):
    model.eval()
    freeze_all_layers_(model)

## Tensor helpers

In [ ]:
def log(t, eps = 1e-12):
    return torch.log(t.clamp(min = eps))

def l2norm(t):
    return F.normalize(t, dim = -1)

def resize_image_to(
        image,
        target_image_size,
        clamp_range = None,
        nearest = False,
        **kwargs
    ):
    orig_image_size = image.shape[-1]

    if orig_image_size == target_image_size:
        return image

    if not nearest:
        scale_factors = target_image_size / orig_image_size
        out           = resize(image, scale_factors = scale_factors, **kwargs)
    else:
        out = F.interpolate(image, target_image_size, mode = 'nearest')

    if exists(clamp_range):
        out = out.clamp(*clamp_range)

    return out

## Image normalization functions

In [ ]:
# DDPM expects images to be in the range -1 to 1
# but CLIP may expect otherwise

def normalize_neg_one_to_one(img):
    return img * 2 - 1

def unnormalize_zero_to_one(normed_img):
    return (normed_img + 1) * 0.5

## CLIP adapters

In [ ]:
EmbeddedText  = namedtuple('EmbedTextReturn',  ['text_embed',  'text_encodings'])
EmbeddedImage = namedtuple('EmbedImageReturn', ['image_embed', 'image_encodings'])

class BaseClipAdapter(nn.Module):
    def __init__(self, clip, **kwargs):
        super().__init__()
        self.clip = clip
        self.overrides = kwargs

    def validate_and_resize_image(self, image):
        image_size = image.shape[-1]
        assert image_size >= self.image_size, f'you are passing in an image of size {image_size} but CLIP requires the image size to be at least {self.image_size}'
        return resize_image_to(image, self.image_size)

    @property
    def dim_latent(self):
        raise NotImplementedError

    @property
    def image_size(self):
        raise NotImplementedError

    @property
    def image_channels(self):
        raise NotImplementedError

    @property
    def max_text_len(self):
        raise NotImplementedError

    def embed_text(self, text):
        raise NotImplementedError

    def embed_image(self, image):
        raise NotImplementedError


class XClipAdapter(BaseClipAdapter):
    @property
    def dim_latent(self):
        return self.clip.dim_latent

    @property
    def image_size(self):
        return self.clip.image_size

    @property
    def image_channels(self):
        return self.clip.image_channels

    @property
    def max_text_len(self):
        return self.clip.text_seq_len

    @torch.no_grad()
    def embed_text(self, text):
        text = text[..., :self.max_text_len]
        text_mask = text != 0
        encoder_output = self.clip.text_transformer(text)

        encoder_output_is_cls = encoder_output.ndim == 3

        text_cls, text_encodings = (encoder_output[:, 0], encoder_output[:, 1:]) if encoder_output_is_cls else (encoder_output, None)
        text_embed = self.clip.to_text_latent(text_cls)

        if exists(text_encodings):
            text_encodings = text_encodings.masked_fill(~text_mask[..., None], 0.)

        return EmbeddedText(l2norm(text_embed), text_encodings)

    @torch.no_grad()
    def embed_image(self, image):
        image = self.validate_and_resize_image(image)
        encoder_output = self.clip.visual_transformer(image)
        image_cls, image_encodings = encoder_output[:, 0], encoder_output[:, 1:]
        image_embed = self.clip.to_visual_latent(image_cls)
        return EmbeddedImage(l2norm(image_embed), image_encodings)


class CoCaAdapter(BaseClipAdapter):
    @property
    def dim_latent(self):
        return self.clip.dim

    @property
    def image_size(self):
        assert 'image_size' in self.overrides
        return self.overrides['image_size']

    @property
    def image_channels(self):
        assert 'image_channels' in self.overrides
        return self.overrides['image_channels']

    @property
    def max_text_len(self):
        assert 'max_text_len' in self.overrides
        return self.overrides['max_text_len']

    @torch.no_grad()
    def embed_text(self, text):
        text = text[..., :self.max_text_len]
        text_mask = text != 0
        text_embed, text_encodings = self.clip.embed_text(text)
        text_encodings = text_encodings.masked_fill(~text_mask[..., None], 0.)
        return EmbeddedText(text_embed, text_encodings)

    @torch.no_grad()
    def embed_image(self, image):
        image = self.validate_and_resize_image(image)
        image_embed, image_encodings = self.clip.embed_image(image)
        return EmbeddedImage(image_embed, image_encodings)


class OpenAIClipAdapter(BaseClipAdapter):
    def __init__(
        self,
        name = 'ViT-B/32'
    ):
        import clip
        openai_clip, preprocess = clip.load(name)
        super().__init__(openai_clip)
        self.eos_id = 49407 # for handling 0 being also '!'

        text_attention_final = self.find_layer('ln_final')

        self.dim_latent_ = text_attention_final.weight.shape[0]
        self.handle      = text_attention_final.register_forward_hook(self._hook)

        self.clip_normalize = preprocess.transforms[-1]
        self.cleared = False

    def find_layer(self,  layer):
        modules = dict([*self.clip.named_modules()])
        return modules.get(layer, None)

    def clear(self):
        if self.cleared:
            return

        self.handle()

    def _hook(self, _, inputs, outputs):
        self.text_encodings = outputs

    @property
    def dim_latent(self):
        return self.dim_latent_

    @property
    def image_size(self):
        return self.clip.visual.input_resolution

    @property
    def image_channels(self):
        return 3

    @property
    def max_text_len(self):
        return self.clip.context_length

    @torch.no_grad()
    def embed_text(self, text):
        text = text[..., :self.max_text_len]

        is_eos_id = (text == self.eos_id)
        text_mask_excluding_eos = is_eos_id.cumsum(dim = -1) == 0
        text_mask = F.pad(text_mask_excluding_eos, (1, -1), value = True)
        text_mask = text_mask & (text != 0)
        assert not self.cleared

        text_embed     = self.clip.encode_text(text)
        text_encodings = self.text_encodings
        text_encodings = text_encodings.masked_fill(~text_mask[..., None], 0.)
        del self.text_encodings
        return EmbeddedText(l2norm(text_embed.float()), text_encodings.float())

    @torch.no_grad()
    def embed_image(self, image):
        assert not self.cleared
        image       = self.validate_and_resize_image(image)
        image       = self.clip_normalize(image)
        image_embed = self.clip.encode_image(image)
        return EmbeddedImage(l2norm(image_embed.float()), None)


class OpenClipAdapter(BaseClipAdapter):
    def __init__(
        self,
        name       = 'ViT-B-32',
        pretrained = 'laion400m_e32'
    ):
        import open_clip
        clip, _, preprocess = open_clip.create_model_and_transforms(name, pretrained = pretrained)

        super().__init__(clip)
        self.eos_id = 49407

        text_attention_final = self.find_layer('ln_final')
        self._dim_latent     = text_attention_final.weight.shape[0]

        self.handle  = text_attention_final.register_forward_hook(self._hook)
        self.clip_normalize = preprocess.transforms[-1]
        self.cleared        = False

    def find_layer(self,  layer):
        modules = dict([*self.clip.named_modules()])
        return modules.get(layer, None)

    def clear(self):
        if self.cleared:
            return

        self.handle()

    def _hook(self, _, inputs, outputs):
        self.text_encodings = outputs

    @property
    def dim_latent(self):
        return self._dim_latent

    @property
    def image_size(self):
        image_size = self.clip.visual.image_size
        if isinstance(image_size, tuple):
            return max(image_size)
        return image_size

    @property
    def image_channels(self):
        return 3

    @property
    def max_text_len(self):
        return self.clip.context_length

    @torch.no_grad()
    def embed_text(self, text):
        text = text[..., :self.max_text_len]

        is_eos_id = (text == self.eos_id)
        text_mask_excluding_eos = is_eos_id.cumsum(dim = -1) == 0
        text_mask = F.pad(text_mask_excluding_eos, (1, -1), value = True)
        text_mask = text_mask & (text != 0)
        assert not self.cleared

        text_embed     = self.clip.encode_text(text)
        text_encodings = self.text_encodings
        text_encodings = text_encodings.masked_fill(~text_mask[..., None], 0.)
        del self.text_encodings
        return EmbeddedText(l2norm(text_embed.float()), text_encodings.float())

    @torch.no_grad()
    def embed_image(self, image):
        assert not self.cleared
        image       = self.validate_and_resize_image(image)
        image       = self.clip_normalize(image)
        image_embed = self.clip.encode_image(image)
        return EmbeddedImage(l2norm(image_embed.float()), None)

## Classifier-free guidance functions

In [ ]:
def prob_mask_like(shape, prob, device):
    if prob == 1:
        return torch.ones(shape, device = device, dtype = torch.bool)
    elif prob == 0:
        return torch.zeros(shape, device = device, dtype = torch.bool)
    else:
        return torch.zeros(shape, device = device).float().uniform_(0, 1) < prob

## Gaussian diffusion helper functions

In [ ]:
def extract(a, t, x_shape):
    b, *_ = t.shape
    out   = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

def meanflat(x):
    return x.mean(dim = tuple(range(1, len(x.shape))))

def normal_kl(mean1, logvar1, mean2, logvar2):
    return 0.5 * (-1.0 + logvar2 - logvar1 + torch.exp(logvar1 - logvar2) + ((mean1 - mean2) ** 2) * torch.exp(-logvar2))

def approx_standard_normal_cdf(x):
    return 0.5 * (1.0 + torch.tanh(((2.0 / math.pi) ** 0.5) * (x + 0.044715 * (x ** 3))))

def discretized_gaussian_log_likelihood(x, *, means, log_scales, thres = 0.999):
    assert x.shape == means.shape == log_scales.shape

    # attempting to correct nan gradients when learned variance is turned on
    # in the setting of deepspeed fp16
    eps = 1e-12 if x.dtype == torch.float32 else 1e-3

    centered_x   = x - means
    inv_stdv     = torch.exp(-log_scales)
    plus_in      = inv_stdv * (centered_x + 1. / 255.)
    cdf_plus     = approx_standard_normal_cdf(plus_in)
    min_in       = inv_stdv * (centered_x - 1. / 255.)
    cdf_min      = approx_standard_normal_cdf(min_in)
    log_cdf_plus = log(cdf_plus, eps = eps)
    log_one_minus_cdf_min = log(1. - cdf_min, eps = eps)
    cdf_delta    = cdf_plus - cdf_min

    log_probs = torch.where(x < -thres,
        log_cdf_plus,
        torch.where(x > thres,
            log_one_minus_cdf_min,
            log(cdf_delta, eps = eps)))

    return log_probs

def cosine_beta_schedule(timesteps, s = 0.008):
    """
    cosine schedule
    as proposed in https://openreview.net/forum?id=-NEXDKk8gZ
    """
    steps = timesteps + 1
    x     = torch.linspace(0, timesteps, steps, dtype = torch.float64)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / first(alphas_cumprod)
    betas          = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)


def linear_beta_schedule(timesteps):
    scale      = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end   = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype = torch.float64)


def quadratic_beta_schedule(timesteps):
    scale      = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end   = scale * 0.02
    return torch.linspace(beta_start**0.5, beta_end**0.5, timesteps, dtype = torch.float64) ** 2


def sigmoid_beta_schedule(timesteps):
    scale      = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end   = scale * 0.02
    betas      = torch.linspace(-6, 6, timesteps, dtype = torch.float64)
    return torch.sigmoid(betas) * (beta_end - beta_start) + beta_start


class NoiseScheduler(nn.Module):
    def __init__(self, *, beta_schedule, timesteps, loss_type, p2_loss_weight_gamma = 0., p2_loss_weight_k = 1):
        super().__init__()

        if beta_schedule == "cosine":
            betas = cosine_beta_schedule(timesteps)
        elif beta_schedule == "linear":
            betas = linear_beta_schedule(timesteps)
        elif beta_schedule == "quadratic":
            betas = quadratic_beta_schedule(timesteps)
        elif beta_schedule == "jsd":
            betas = 1.0 / torch.linspace(timesteps, 1, timesteps)
        elif beta_schedule == "sigmoid":
            betas = sigmoid_beta_schedule(timesteps)
        else:
            raise NotImplementedError()

        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, axis = 0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value = 1.)

        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)

        if loss_type == 'l1':
            loss_fn = F.l1_loss
        elif loss_type == 'l2':
            loss_fn = F.mse_loss
        elif loss_type == 'huber':
            loss_fn = F.smooth_l1_loss
        else:
            raise NotImplementedError()

        self.loss_type = loss_type
        self.loss_fn   = loss_fn

        # register buffer helper function to cast double back to float

        register_buffer = lambda name, val: self.register_buffer(name, val.to(torch.float32))

        register_buffer('betas', betas)
        register_buffer('alphas_cumprod', alphas_cumprod)
        register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)

        # calculations for diffusion q(x_t | x_{t-1}) and others

        register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1. - alphas_cumprod))
        register_buffer('log_one_minus_alphas_cumprod', torch.log(1. - alphas_cumprod))
        register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1. / alphas_cumprod))
        register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1. / alphas_cumprod - 1))

        # calculations for posterior q(x_{t-1} | x_t, x_0)

        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)

        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)

        register_buffer('posterior_variance', posterior_variance)

        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain

        register_buffer('posterior_log_variance_clipped', torch.log(posterior_variance.clamp(min =1e-20)))
        register_buffer('posterior_mean_coef1', betas * torch.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))
        register_buffer('posterior_mean_coef2', (1. - alphas_cumprod_prev) * torch.sqrt(alphas) / (1. - alphas_cumprod))

        # p2 loss reweighting

        self.has_p2_loss_reweighting = p2_loss_weight_gamma > 0.
        register_buffer('p2_loss_weight', (p2_loss_weight_k + alphas_cumprod / (1 - alphas_cumprod)) ** -p2_loss_weight_gamma)

    def sample_random_times(self, batch):
        return torch.randint(0, self.num_timesteps, (batch,), device = self.betas.device, dtype = torch.long)

    def q_posterior(self, x_start, x_t, t):
        posterior_mean = (
            extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +
            extract(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        posterior_variance             = extract(self.posterior_variance, t, x_t.shape)
        posterior_log_variance_clipped = extract(self.posterior_log_variance_clipped, t, x_t.shape)
        return posterior_mean, posterior_variance, posterior_log_variance_clipped

    def q_sample(self, x_start, t, noise = None):
        noise = default(noise, lambda: torch.randn_like(x_start))

        return (
            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start +
            extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise
        )

    def calculate_v(self, x_start, t, noise = None):
        return (
            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * noise -
            extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * x_start
        )

    def q_sample_from_to(self, x_from, from_t, to_t, noise = None):
        shape = x_from.shape
        noise = default(noise, lambda: torch.randn_like(x_from))

        alpha      = extract(self.sqrt_alphas_cumprod, from_t, shape)
        sigma      = extract(self.sqrt_one_minus_alphas_cumprod, from_t, shape)
        alpha_next = extract(self.sqrt_alphas_cumprod, to_t, shape)
        sigma_next = extract(self.sqrt_one_minus_alphas_cumprod, to_t, shape)

        return x_from * (alpha_next / alpha) + noise * (sigma_next * alpha - sigma * alpha_next) / alpha

    def predict_start_from_v(self, x_t, t, v):
        return (
            extract(self.sqrt_alphas_cumprod, t, x_t.shape) * x_t -
            extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape) * v
        )

    def predict_start_from_noise(self, x_t, t, noise):
        return (
            extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
            extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise
        )

    def predict_noise_from_start(self, x_t, t, x0):
        return (
            (extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - x0) / \
            extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape)
        )

    def p2_reweigh_loss(self, loss, times):
        if not self.has_p2_loss_reweighting:
            return loss
        return loss * extract(self.p2_loss_weight, times, loss.shape)

## Rearrange image to sequence

In [ ]:
class RearrangeToSequence(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x):
        x     = rearrange(x, 'b c ... -> b ... c')
        x, ps = pack([x], 'b * c')

        x     = self.fn(x)

        x,    = unpack(x, ps, 'b * c')
        x     = rearrange(x, 'b ... c -> b c ...')
        return x

## Diffusion prior

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps = 1e-5, fp16_eps = 1e-3, stable = False):
        super().__init__()
        self.eps      = eps
        self.fp16_eps = fp16_eps
        self.stable   = stable
        self.g        = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        eps = self.eps if x.dtype == torch.float32 else self.fp16_eps

        if self.stable:
            x = x / x.amax(dim = -1, keepdim = True).detach()

        var   = torch.var(x, dim = -1, unbiased = False, keepdim = True)
        mean  = torch.mean(x, dim = -1, keepdim = True)
        return (x - mean) * (var + eps).rsqrt() * self.g

class ChanLayerNorm(nn.Module):
    def __init__(self, dim, eps = 1e-5, fp16_eps = 1e-3, stable = False):
        super().__init__()
        self.eps      = eps
        self.fp16_eps = fp16_eps
        self.stable   = stable
        self.g        = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        eps = self.eps if x.dtype == torch.float32 else self.fp16_eps

        if self.stable:
            x = x / x.amax(dim = 1, keepdim = True).detach()

        var  = torch.var(x, dim = 1, unbiased = False, keepdim = True)
        mean = torch.mean(x, dim = 1, keepdim = True)
        return (x - mean) * (var + eps).rsqrt() * self.g

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


### MLP

In [ ]:
class MLP(nn.Module):
    def __init__(
            self,
            dim_in,
            dim_out,
            *,
            expansion_factor = 2.,
            depth            = 2,
            norm             = False,
        ):
        super().__init__()
        hidden_dim = int(expansion_factor * dim_out)
        norm_fn    = lambda: nn.LayerNorm(hidden_dim) if norm else nn.Identity()

        layers = [nn.Sequential(
            nn.Linear(dim_in, hidden_dim),
            nn.SiLU(),
            norm_fn()
        )]

        for _ in range(depth - 1):
            layers.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
                norm_fn()
            ))

        layers.append(nn.Linear(hidden_dim, dim_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.float())


### Relative positional bias for causal transformer

In [ ]:
class RelPosBias(nn.Module):
    def __init__(
            self,
            heads        = 8,
            num_buckets  = 32,
            max_distance = 128,
        ):
        super().__init__()
        self.num_buckets             = num_buckets
        self.max_distance            = max_distance
        self.relative_attention_bias = nn.Embedding(num_buckets, heads)

    @staticmethod
    def _relative_position_bucket(
            relative_position,
            num_buckets  = 32,
            max_distance = 128
        ):
        n_neg = -relative_position # invert the sign of the relative position matrix
        n     = torch.max(n_neg, torch.zeros_like(n_neg)) # remove the negative values

        #       0   1   2   ...  79  80  81  <- j
        #     ------------------------------
        # n = [ 0,  0,  0,  ...,  0,  0,  0]  0  <- i
        #     [ 1,  0,  0,  ...,  0,  0,  0]  1  
        #     [ 2,  1,  0,  ...,  0,  0,  0]  2  
        #       ...,
        #     [14, 13, 12,  ...,  0,  0,  0]  14 
        #     [15, 14, 13,  ...,  0,  0,  0]  15 
        #     [16, 15, 14,  ...,  0,  0,  0]  16 
        #     [17, 16, 15,  ...,  0,  0,  0]  17 
        #      ...,
        #     [78, 77, 76,  ...,  0,  0,  0]  78 
        #     [79, 78, 77,  ...,  0,  0,  0]  79 
        #     [80, 79, 78,  ...,  1,  0,  0]  80 <- i

        # Determine which relative positions are less than 16 to be represented exactly
        max_exact = num_buckets // 2 # 16
        # is_small is True where n < 16 amd False otherwise
        is_small  = n < max_exact

        #              0      1      2     ...   79    80    81  <- j
        #            --------------------------------------------
        # is_small = [True,  True,  True,  ..., True, True, True] 0  <- i
        #            [True,  True,  True,  ..., True, True, True] 1
        #            [True,  True,  True,  ..., True, True, True] 2
        #             ...,
        #            [True,  True,  True,  ..., True, True, True] 14
        #            [True,  True,  True,  ..., True, True, True] 15
        #            [False, True,  True,  ..., True, True, True] 16
        #            [False, False, True,  ..., True, True, True] 17
        #             ...
        #            [False, False, False, ..., True, True, True] 78
        #            [False, False, False, ..., True, True, True] 79
        #            [False, False, False, ..., True, True, True] 80 <- i
        
        # minFP = -9223372036854775792
        # val_if_large = minFP on the main diagonal an above, since n=0
        # val_if_large1 = 16 + (log(n/16) / log(128 / 16) * (32 - 16)), below the main diagonal
        val_if_large1 = max_exact + (torch.log(n.float() / max_exact) / math.log(max_distance / max_exact) * (num_buckets - max_exact)).long()
        
        # minFP = log(0) = -9223372036854775792
        #
        # if n=0  -> val_if_large1 = minFP
        #    n=1  -> val_if_large1 = -5
        #    n=2  -> val_if_large1 = 0
        #    n=3  -> val_if_large1 = 3
        #    n=4  -> val_if_large1 = 5
        #    n=5  -> val_if_large1 = 7
        #    ...
        #    n=78 -> val_if_large1 = 28
        #    n=79 -> val_if_large1 = 28
        #    n=80 -> val_if_large1 = 28

        # clip val_if_large1 to be at most 'num_buckets-1'=31
        # val_if_large = min(val_if_large1, 31)
        # val_if_large = val_if_large1, since all values of val_if_large1 are less than 31
        val_if_large = torch.min(val_if_large1, torch.full_like(val_if_large1, num_buckets - 1))

        # For small relative positions (n<16), take the exact value (rel_pos = n)
        # For large relative positions (n>=16), take the computed bucketed value (rel_pos = val_if_large)        
        rel_pos = torch.where(is_small, n, val_if_large)

        # rel_pos = [ 0,  0,  0,  ...,  0,  0,  0]
        #           [ 1,  0,  0,  ...,  0,  0,  0]
        #           [ 2,  1,  0,  ...,  0,  0,  0]
        #            ...
        #           [28, 28, 27,  ...,  0,  0,  0]
        #           [28, 28, 28,  ...,  0,  0,  0]
        #           [28, 28, 28,  ...,  1,  0,  0]

        return rel_pos

    def forward(self, i, j, *, device):
        q_pos     = torch.arange(i, dtype = torch.long, device = device) # [0, 1, 2, 3, ..., i-1=80]
        k_pos     = torch.arange(j, dtype = torch.long, device = device) # [0, 1, 2, 3, ..., j-1=81]
        # Calculate the relative position matrix for j-i, with a shape of [i, j]
        rel_pos   = rearrange(k_pos, 'j -> 1 j') - rearrange(q_pos, 'i -> i 1')

        # [[  0,   1,   2,  ...,  79,  80,  81],
        #  [ -1,   0,   1,  ...,  78,  79,  80],
        #  [ -2,  -1,   0,  ...,  77,  78,  79],
        #   ...,
        #  [-78, -77, -76,  ...,   1,   2,   3],
        #  [-79, -78, -77,  ...,   0,   1,   2],
        #  [-80, -79, -78,  ...,  -1,   0,   1]]
    
        # Calculate a bucketed relative position matrix, which is the exact value of j-i when (j-i < 16)
        # and a logarithmically scaled value when (j-i >= 16)
        rp_bucket = self._relative_position_bucket(
            rel_pos, 
            num_buckets  = self.num_buckets, 
            max_distance = self.max_distance
            )
        
        # Get the embeddings for the bucketed relative position matrix, which are our relative position biases
        values    = self.relative_attention_bias(rp_bucket)

        # Rearrange the embeddings to go from shape [i, j, heads] to [heads, i, j]
        bias      = rearrange(values, 'i j h -> h i j')

        return bias


### Feedforward

In [ ]:
class SwiGLU(nn.Module):
    """
    Used successfully in https://arxiv.org/abs/2204.0231
    """
    def forward(self, x):
        x, gate = x.chunk(2, dim = -1)
        return x * F.silu(gate)

def FeedForwardPrior(
    dim,
    mult                 = 4,
    dropout              = 0.,
    post_activation_norm = False
    ):
    """
    Post-activation norm: https://arxiv.org/abs/2110.09456
    """

    inner_dim = int(mult * dim)
    return nn.Sequential(
        LayerNorm(dim),
        nn.Linear(dim, inner_dim * 2, bias = False),
        SwiGLU(),
        LayerNorm(inner_dim) if post_activation_norm else nn.Identity(),
        nn.Dropout(dropout),
        nn.Linear(inner_dim, dim, bias = False)
    )

### Attention

In [ ]:
class Attention(nn.Module):
    def __init__(
            self,
            dim,
            *,
            dim_head         = 64,
            heads            = 8,
            dropout          = 0.,
            causal           = False,
            rotary_emb       = None,
            cosine_sim       = True,
            cosine_sim_scale = 16
        ):
        super().__init__()
        self.scale      = cosine_sim_scale if cosine_sim else (dim_head ** -0.5)
        self.cosine_sim = cosine_sim
        self.heads      = heads
        inner_dim       = dim_head * heads
        # scale=16 dim_head=64 heads=12 inner_dim=768 cosine_sim=True cosine_sim_scale=16
        
        self.causal  = causal
        self.norm    = LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

        self.null_kv = nn.Parameter(torch.randn(2, dim_head))
        self.to_q    = nn.Linear(dim, inner_dim, bias = False)
        self.to_kv   = nn.Linear(dim, dim_head * 2, bias = False)

        self.rotary_emb = rotary_emb

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim, bias = False),
            LayerNorm(dim)
        )

    def forward(self, x, mask = None, attn_bias = None): # x.shape=[32, 81, 512], attn_bias.shape=[12, 81, 82]
        b, n, device = *x.shape[:2], x.device # b=batch size=32, n=sequence length=81
        x            = self.norm(x)  # apply normalization to input x ([32, 81, 512])
        q, k, v      = (self.to_q(x), *self.to_kv(x).chunk(2, dim = -1)) # project x to obtain the queries, keys, and values
        # q.shape=[32, 81, 768] k.shape=[32, 81, 64] v.shape=[32, 81, 64]

        q            = rearrange(q, 'b n (h d) -> b h n d', h = self.heads) # q.shape=[32, 12, 81, 64] 

        q            = q * self.scale  # q.shape=[32, 12, 81, 64]

        # apply rotary positional embeddings to queries and keys

        if exists(self.rotary_emb):
            q, k = map(self.rotary_emb.rotate_queries_or_keys, (q, k))
            # q.shape=[32, 12, 81, 64]  k.shape=[32, 81, 64]

        # add random key / value for classifier free guidance in prior network
        nk, nv = map(lambda t: repeat(t, 'd -> b 1 d', b = b), self.null_kv.unbind(dim = -2))
        k      = torch.cat((nk, k), dim = -2)
        v      = torch.cat((nv, v), dim = -2)
        # nk.shape=[32, 1, 64]  nv.shape=[32, 1, 64]
        # k.shape=[32, 82, 64]  v.shape=[32, 82, 64]

        # whether to use cosine similarity or not

        if self.cosine_sim:
            q, k = map(l2norm, (q, k)) # normalize q and k along the last dimension
            # q.shape=[32, 12, 81, 64]  k.shape=[32, 82, 64]

        q, k = map(lambda t: t * math.sqrt(self.scale), (q, k)) # scale q and k by sqrt(scale)

        # calculate query,key similarities

        sim = einsum('b h i d, b j d -> b h i j', q, k)
        # sim.shape=[32, 12, 81, 82]

        # add relative positional bias to similarity (T5 style)

        if exists(attn_bias):
            sim = sim + attn_bias

        # masking

        max_neg_value = -torch.finfo(sim.dtype).max # maximum negative value of floating type ('sim')

        if exists(mask): 

            mask = F.pad(mask, (1, 0), value = True)     # pad 'mask' at the beginning of last dimension with True
            mask = rearrange(mask, 'b j -> b 1 1 j')     # reshape 'mask' for broadcasting
            sim  = sim.masked_fill(~mask, max_neg_value) # fill positions of 'sim' where 'mask' is False with max negative value

        if self.causal:
            i, j        = sim.shape[-2:] # i=81,j=82, are the last two dimensions of 'sim'
            # create a boolean mask with shape (i, j) and filled with True on and above the (j-i+1=2) 2nd diagonal
            causal_mask = torch.ones((i, j), dtype = torch.bool, device = device).triu(j - i + 1) # causal_mask.shape=[81, 82]
            sim         = sim.masked_fill(causal_mask, max_neg_value) # fill positions of 'sim' where 'causal_mask' is True with max negative value
            # sim.shape=[32, 12, 81, 82]
 
        # calculate attention scores

        # attention scores = result of softmax applied to 'sim' along the last dimension
        attn = sim.softmax(dim = -1, dtype = torch.float32)
        attn = attn.type(sim.dtype) # attn.shape=[32, 12, 81, 82]

        # Apply dropout with the selected probability to the attention scores
        attn = self.dropout(attn)

        # multiply the attention scores tensor by the values tensor

        # attn.shape=[32, 12, 81, 82], v.shape=[32, 82, 64] ->  out.shape=[32, 12, 81, 64]
        out = einsum('b h i j, b j d -> b h i d', attn, v)
        
        out = rearrange(out, 'b h n d -> b n (h d)') # out.shape=[32, 81, 768]

        # Pass the output through the final Linear and normalization layers
        out = self.to_out(out)  # out.shape=[32, 81, 512]

        return out


### Causal Transformer

### Diffusion Prior Model

In [ ]:
class CausalTransformer(nn.Module):
    def __init__(
            self,
            *,
            dim,
            depth,
            dim_head     = 64,
            heads        = 8,
            ff_mult      = 4,
            norm_in      = False,
            norm_out     = True,
            attn_dropout = 0.,
            ff_dropout   = 0.,
            final_proj   = True,
            normformer   = False,
            rotary_emb   = True
        ):
        super().__init__()
        self.init_norm    = LayerNorm(dim) if norm_in else nn.Identity() # from latest BLOOM model and Yandex's YaLM

        self.rel_pos_bias = RelPosBias(heads = heads)

        rotary_emb = RotaryEmbedding(dim = min(32, dim_head)) if rotary_emb else None

        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(
                    dim = dim, 
                    causal = True, 
                    dim_head = dim_head,
                    heads = heads,
                    dropout = attn_dropout,
                    rotary_emb = rotary_emb
                    ),
                FeedForwardPrior(
                    dim = dim, 
                    mult = ff_mult,
                    dropout = ff_dropout,
                    post_activation_norm = normformer
                    )
            ]))

        # unclear in paper whether they projected after the classic layer norm for the 
        # final denoised image embedding, or just had the transformer output it directly: 
        # plan on offering both options
        self.norm        = LayerNorm(dim, stable = True) if norm_out else nn.Identity()  
        self.project_out = nn.Linear(dim, dim, bias = False) if final_proj else nn.Identity()

    def forward(self, x):
        n, device = x.shape[1], x.device

        x = self.init_norm(x)

        attn_bias = self.rel_pos_bias(n, n + 1, device = device)
        
        for attn, ff in self.layers:
            x = attn(x, attn_bias = attn_bias) + x
            x = ff(x) + x

        out = self.norm(x)
        return self.project_out(out)


In [ ]:
class DiffusionPriorNetwork(nn.Module):
    def __init__(
            self,
            dim,
            num_timesteps    = None,
            num_time_embeds  = 1,
            num_image_embeds = 1,
            num_text_embeds  = 1,
            max_text_len     = 256,
            self_cond        = False,
            **kwargs
        ):
        super().__init__()
        self.dim = dim

        self.num_time_embeds  = num_time_embeds
        self.num_image_embeds = num_image_embeds
        self.num_text_embeds  = num_text_embeds

        self.to_text_embeds   = nn.Sequential(
            nn.Linear(dim, dim * num_text_embeds) if num_text_embeds > 1 else nn.Identity(),
            Rearrange('b (n d) -> b n d', n = num_text_embeds)
        )

        self.continuous_embedded_time = not exists(num_timesteps)

        self.to_time_embeds = nn.Sequential(
            nn.Embedding(num_timesteps, dim * num_time_embeds) if exists(num_timesteps) else nn.Sequential(SinusoidalPosEmb(dim), MLP(dim, dim * num_time_embeds)), # also offer a continuous version of timestep embeddings, with a 2 layer MLP
            Rearrange('b (n d) -> b n d', n = num_time_embeds)
        )

        self.to_image_embeds = nn.Sequential(
            nn.Linear(dim, dim * num_image_embeds) if num_image_embeds > 1 else nn.Identity(),
            Rearrange('b (n d) -> b n d', n = num_image_embeds)
        )

        self.learned_query      = nn.Parameter(torch.randn(dim))
        self.causal_transformer = CausalTransformer(dim = dim, **kwargs)

        # dalle1 learned padding strategy

        self.max_text_len = max_text_len

        self.null_text_encodings = nn.Parameter(torch.randn(1, max_text_len, dim))
        self.null_text_embeds    = nn.Parameter(torch.randn(1, num_text_embeds, dim))
        self.null_image_embed    = nn.Parameter(torch.randn(1, dim))

        # whether to use self conditioning, Hinton's group's new ddpm technique

        self.self_cond = self_cond

    def forward_with_cond_scale(
        self,
        *args,
        cond_scale = 1.,
        **kwargs
    ):
        logits = self.forward(*args, **kwargs)

        if cond_scale == 1:
            return logits

        null_logits = self.forward(*args, text_cond_drop_prob = 1., image_cond_drop_prob = 1, **kwargs)
        return null_logits + (logits - null_logits) * cond_scale

    def forward(
            self,
            image_embed,
            diffusion_timesteps,
            *,
            text_embed,
            text_encodings       = None,
            self_cond            = None,
            text_cond_drop_prob  = 0.,
            image_cond_drop_prob = 0.
        ):
        batch, dim, device, dtype = *image_embed.shape, image_embed.device, image_embed.dtype

        num_time_embeds, num_image_embeds, num_text_embeds = self.num_time_embeds, self.num_image_embeds, self.num_text_embeds

        # setup self conditioning

        if self.self_cond:
            self_cond = default(self_cond, lambda: torch.zeros(batch, self.dim, device = device, dtype = dtype))
            self_cond = rearrange(self_cond, 'b d -> b 1 d')

        # In section 2.2, last paragraph:
        # "... consisting of encoded text, CLIP text embedding, diffusion timestep 
        #  embedding, noised CLIP image embedding, final embedding for prediction".

        text_embed  = self.to_text_embeds(text_embed)
        image_embed = self.to_image_embeds(image_embed)

        # classifier free guidance masks

        text_keep_mask = prob_mask_like((batch,), 1 - text_cond_drop_prob, device = device)
        text_keep_mask = rearrange(text_keep_mask, 'b -> b 1 1')

        image_keep_mask = prob_mask_like((batch,), 1 - image_cond_drop_prob, device = device)
        image_keep_mask = rearrange(image_keep_mask, 'b -> b 1 1')

        # make text encodings optional
        # although the paper seems to suggest it is present

        if not exists(text_encodings):
            text_encodings = torch.empty((batch, 0, dim), device = device, dtype = dtype)

        mask = torch.any(text_encodings != 0., dim = -1)

        # replace any padding in the text encodings with learned padding tokens unique across position

        text_encodings = text_encodings[:, :self.max_text_len]
        mask           = mask[:, :self.max_text_len]

        text_len  = text_encodings.shape[-2]
        remainder = self.max_text_len - text_len

        if remainder > 0:
            text_encodings = F.pad(text_encodings, (0, 0, 0, remainder), value = 0.)
            mask           = F.pad(mask, (0, remainder), value = False)

        # mask out text encodings with random encodings

        null_text_encodings = self.null_text_encodings.to(text_encodings.dtype)

        text_encodings = torch.where(
            rearrange(mask, 'b n -> b n 1').clone() & text_keep_mask,
            text_encodings,
            null_text_encodings
        )

        # mask out text embeddings with random text embeddings

        null_text_embeds = self.null_text_embeds.to(text_embed.dtype)

        text_embed = torch.where(
            text_keep_mask,
            text_embed,
            null_text_embeds
        )

        # mask out image embeddings with random image embeddings

        null_image_embed = self.null_image_embed.to(image_embed.dtype)

        image_embed = torch.where(
            image_keep_mask,
            image_embed,
            null_image_embed
        )

        # Whether text embedding is used for conditioning depends on whether 
        # text encodings are available for attention (for classifier free guidance, 
        # even though it seems from the paper it was not used in the prior DDPM, 
        # as the objective is different). But let us just do it right.

        if self.continuous_embedded_time:
            diffusion_timesteps = diffusion_timesteps.type(dtype)

        time_embed = self.to_time_embeds(diffusion_timesteps)

        learned_queries = repeat(self.learned_query, 'd -> b 1 d', b = batch)

        if self.self_cond:
            learned_queries = torch.cat((self_cond, learned_queries), dim = -2)

        tokens = torch.cat((
            text_encodings,
            text_embed,
            time_embed,
            image_embed,
            learned_queries
        ), dim = -2)

        # attend

        tokens = self.causal_transformer(tokens)

        # get learned query, which should predict the image embedding (per DDPM timestep)

        pred_image_embed = tokens[..., -1, :]

        return pred_image_embed

In [ ]:
class DiffusionPrior(nn.Module):
    def __init__(
            self,
            net,
            *,
            clip                        = None,
            image_embed_dim             = None,
            image_size                  = None,
            image_channels              = 3,
            timesteps                   = 1000,
            sample_timesteps            = None,
            cond_drop_prob              = 0.,
            text_cond_drop_prob         = None,
            image_cond_drop_prob        = None,
            loss_type                   = "l2",
            predict_x_start             = True,
            predict_v                   = False,
            beta_schedule               = "cosine",
            condition_on_text_encodings = True,  # the paper suggests this is needed, but you can turn it off for your CLIP preprocessed text embed -> image embed training
            sampling_clamp_l2norm       = False, # whether to l2norm clamp the image embed at each denoising iteration (analogous to -1 to 1 clipping for usual DDPMs)
            sampling_final_clamp_l2norm = False, # whether to l2norm the final image embedding output (this is also done for images in ddpm)
            training_clamp_l2norm       = False,
            init_image_embed_l2norm     = False,
            image_embed_scale           = None,   # this is for scaling the l2-normed image embedding, so it is more suitable for gaussian diffusion, as outlined by Katherine (@crowsonkb) https://github.com/lucidrains/DALLE2-pytorch/issues/60#issue-1226116132
            clip_adapter_overrides      = dict()
        ):
        super().__init__()

        self.sample_timesteps = sample_timesteps

        self.noise_scheduler = NoiseScheduler(
            beta_schedule = beta_schedule,
            timesteps     = timesteps,
            loss_type     = loss_type
        )

        if exists(clip):
            assert image_channels == clip.image_channels, f'channels of image ({image_channels}) should be equal to the channels that CLIP accepts ({clip.image_channels})'

            if isinstance(clip, CLIP):
                clip = XClipAdapter(clip, **clip_adapter_overrides)
            elif isinstance(clip, CoCa):
                clip = CoCaAdapter(clip, **clip_adapter_overrides)

            assert isinstance(clip, BaseClipAdapter)
            freeze_model_and_make_eval_(clip)
            self.clip = clip
        else:
            assert exists(image_embed_dim), 'latent dimension must be given, if training prior network without CLIP given'
            self.clip = None

        self.net             = net
        self.image_embed_dim = default(image_embed_dim, lambda: clip.dim_latent)

        assert net.dim == self.image_embed_dim, f'your diffusion prior network has a dimension of {net.dim}, but you set your image embedding dimension (keyword image_embed_dim) on DiffusionPrior to {self.image_embed_dim}'
        assert not exists(clip) or clip.dim_latent == self.image_embed_dim, f'you passed in a CLIP to the diffusion prior with latent dimensions of {clip.dim_latent}, but your image embedding dimension (keyword image_embed_dim) for the DiffusionPrior was set to {self.image_embed_dim}'

        self.channels = default(image_channels, lambda: clip.image_channels)

        self.text_cond_drop_prob  = default(text_cond_drop_prob, cond_drop_prob)
        self.image_cond_drop_prob = default(image_cond_drop_prob, cond_drop_prob)

        self.can_classifier_guidance     = self.text_cond_drop_prob > 0. and self.image_cond_drop_prob > 0.
        self.condition_on_text_encodings = condition_on_text_encodings

        # in paper, they do not predict the noise, but predict x0 directly for image embedding, claiming empirically better results. I'll just offer both.

        self.predict_x_start = predict_x_start
        self.predict_v       = predict_v # takes precedence over predict_x_start

        # suggestion from: https://github.com/lucidrains/DALLE2-pytorch/issues/60#issue-1226116132

        self.image_embed_scale = default(image_embed_scale, self.image_embed_dim ** 0.5)

        # whether to force an l2norm, similar to clipping denoised, when sampling

        self.sampling_clamp_l2norm       = sampling_clamp_l2norm
        self.sampling_final_clamp_l2norm = sampling_final_clamp_l2norm

        self.training_clamp_l2norm   = training_clamp_l2norm
        self.init_image_embed_l2norm = init_image_embed_l2norm

        # device tracker

        self.register_buffer('_dummy', torch.tensor([True]), persistent = False)

    @property
    def device(self):
        return self._dummy.device

    def l2norm_clamp_embed(self, image_embed):
        return l2norm(image_embed) * self.image_embed_scale

    def p_mean_variance(self, x, t, text_cond, self_cond = None, clip_denoised = False, cond_scale = 1.):
        assert not (cond_scale != 1. and not self.can_classifier_guidance), 'the model was not trained with conditional dropout, and thus one cannot use classifier free guidance (cond_scale anything other than 1)'

        pred = self.net.forward_with_cond_scale(x, t, cond_scale = cond_scale, self_cond = self_cond, **text_cond)

        if self.predict_v:
            x_start = self.noise_scheduler.predict_start_from_v(x, t = t, v = pred)
        elif self.predict_x_start:
            x_start = pred
        else:
            x_start = self.noise_scheduler.predict_start_from_noise(x, t = t, noise = pred)

        if clip_denoised and not self.predict_x_start:
            x_start.clamp_(-1., 1.)

        if self.predict_x_start and self.sampling_clamp_l2norm:
            x_start = l2norm(x_start) * self.image_embed_scale

        model_mean, posterior_variance, posterior_log_variance = self.noise_scheduler.q_posterior(x_start=x_start, x_t=x, t=t)
        return model_mean, posterior_variance, posterior_log_variance, x_start

    @torch.no_grad()
    def p_sample(self, x, t, text_cond = None, self_cond = None, clip_denoised = True, cond_scale = 1.):
        b, *_, device = *x.shape, x.device
        model_mean, _, model_log_variance, x_start = self.p_mean_variance(x = x, t = t, text_cond = text_cond, self_cond = self_cond, clip_denoised = clip_denoised, cond_scale = cond_scale)
        noise = torch.randn_like(x)
        # no noise when t == 0
        nonzero_mask = (1 - (t == 0).float()).reshape(b, *((1,) * (len(x.shape) - 1)))
        pred = model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise
        return pred, x_start

    @torch.no_grad()
    def p_sample_loop_ddpm(self, shape, text_cond, cond_scale = 1.):
        batch, device = shape[0], self.device

        image_embed = torch.randn(shape, device = device)
        x_start = None # for self-conditioning

        if self.init_image_embed_l2norm:
            image_embed = l2norm(image_embed) * self.image_embed_scale

        for i in tqdm(reversed(range(0, self.noise_scheduler.num_timesteps)), desc='sampling loop time step', total=self.noise_scheduler.num_timesteps):
            times = torch.full((batch,), i, device = device, dtype = torch.long)

            self_cond = x_start if self.net.self_cond else None
            image_embed, x_start = self.p_sample(image_embed, times, text_cond = text_cond, self_cond = self_cond, cond_scale = cond_scale)

        if self.sampling_final_clamp_l2norm and self.predict_x_start:
            image_embed = self.l2norm_clamp_embed(image_embed)

        return image_embed

    @torch.no_grad()
    def p_sample_loop_ddim(self, shape, text_cond, *, timesteps, eta = 1., cond_scale = 1.):
        batch, device, alphas, total_timesteps = shape[0], self.device, self.noise_scheduler.alphas_cumprod_prev, self.noise_scheduler.num_timesteps

        times      = torch.linspace(-1., total_timesteps, steps = timesteps + 1)[:-1]

        times      = list(reversed(times.int().tolist()))
        time_pairs = list(zip(times[:-1], times[1:]))

        image_embed = torch.randn(shape, device = device)

        x_start     = None # for self-conditioning

        if self.init_image_embed_l2norm:
            image_embed = l2norm(image_embed) * self.image_embed_scale

        for time, time_next in tqdm(time_pairs, desc = 'sampling loop time step'):
            alpha = alphas[time]
            alpha_next = alphas[time_next]

            time_cond = torch.full((batch,), time, device = device, dtype = torch.long)

            self_cond = x_start if self.net.self_cond else None

            pred = self.net.forward_with_cond_scale(image_embed, time_cond, self_cond = self_cond, cond_scale = cond_scale, **text_cond)

            # derive x0

            if self.predict_v:
                x_start = self.noise_scheduler.predict_start_from_v(image_embed, t = time_cond, v = pred)
            elif self.predict_x_start:
                x_start = pred
            else:
                x_start = self.noise_scheduler.predict_start_from_noise(image_embed, t = time_cond, noise = pred)

            # clip x0 before maybe predicting noise

            if not self.predict_x_start:
                x_start.clamp_(-1., 1.)

            if self.predict_x_start and self.sampling_clamp_l2norm:
                x_start = self.l2norm_clamp_embed(x_start)

            # predict noise

            pred_noise = self.noise_scheduler.predict_noise_from_start(image_embed, t = time_cond, x0 = x_start)

            if time_next < 0:
                image_embed = x_start
                continue

            c1 = eta * ((1 - alpha / alpha_next) * (1 - alpha_next) / (1 - alpha)).sqrt()
            c2 = ((1 - alpha_next) - torch.square(c1)).sqrt()
            noise = torch.randn_like(image_embed) if time_next > 0 else 0.

            image_embed = x_start * alpha_next.sqrt() + \
                          c1 * noise + \
                          c2 * pred_noise

        if self.predict_x_start and self.sampling_final_clamp_l2norm:
            image_embed = self.l2norm_clamp_embed(image_embed)

        return image_embed

    @torch.no_grad()
    def p_sample_loop(self, *args, timesteps = None, **kwargs):
        timesteps = default(timesteps, self.noise_scheduler.num_timesteps)
        assert timesteps <= self.noise_scheduler.num_timesteps
        is_ddim = timesteps < self.noise_scheduler.num_timesteps

        if not is_ddim:
            normalized_image_embed = self.p_sample_loop_ddpm(*args, **kwargs)
        else:
            normalized_image_embed = self.p_sample_loop_ddim(*args, **kwargs, timesteps = timesteps)

        image_embed = normalized_image_embed / self.image_embed_scale
        return image_embed

    def p_losses(self, image_embed, times, text_cond, noise = None):
        noise = default(noise, lambda: torch.randn_like(image_embed))

        image_embed_noisy = self.noise_scheduler.q_sample(x_start = image_embed, t = times, noise = noise)

        #print(f'[DEBUG] diffusion prior p_losses image_embed_noisy: {image_embed_noisy}')

        self_cond = None
        if self.net.self_cond and random.random() < 0.5:
            with torch.no_grad():
                self_cond = self.net(image_embed_noisy, times, **text_cond).detach()

        pred = self.net(
            image_embed_noisy,
            times,
            self_cond = self_cond,
            text_cond_drop_prob = self.text_cond_drop_prob,
            image_cond_drop_prob = self.image_cond_drop_prob,
            **text_cond
        )

        #print(f'[DEBUG] diffusion prior p_losses pred: {pred}')

        if self.predict_x_start and self.training_clamp_l2norm:
            pred = self.l2norm_clamp_embed(pred)

        if self.predict_v:
            target = self.noise_scheduler.calculate_v(image_embed, times, noise)
        elif self.predict_x_start:
            target = image_embed
        else:
            target = noise

        #print(f'[DEBUG] diffusion prior p_losses target: {target}')

        loss = self.noise_scheduler.loss_fn(pred, target)
        return loss

    @torch.no_grad()
    @eval_decorator
    def sample_batch_size(self, batch_size, text_cond, cond_scale = 1.):
        device = self.betas.device
        shape  = (batch_size, self.image_embed_dim)

        img    = torch.randn(shape, device = device)

        for i in tqdm(reversed(range(0, self.noise_scheduler.num_timesteps)), desc = 'sampling loop time step', total = self.noise_scheduler.num_timesteps):
            img = self.p_sample(img, torch.full((batch_size,), i, device = device, dtype = torch.long), text_cond = text_cond, cond_scale = cond_scale)
        return img

    @torch.no_grad()
    @eval_decorator
    def sample(
            self,
            text,
            num_samples_per_batch = 2,
            cond_scale            = 1.,
            timesteps             = None
        ):
        timesteps = default(timesteps, self.sample_timesteps)

        # in the paper, what they did was
        # sample 2 image embeddings, choose the top 1 similarity, as judged by CLIP
        text = repeat(text, 'b ... -> (b r) ...', r = num_samples_per_batch)

        batch_size      = text.shape[0]
        image_embed_dim = self.image_embed_dim

        text_embed, text_encodings = self.clip.embed_text(text)

        text_cond = dict(text_embed = text_embed)

        if self.condition_on_text_encodings:
            text_cond = {**text_cond, 'text_encodings': text_encodings}

        image_embeds = self.p_sample_loop((batch_size, image_embed_dim), text_cond = text_cond, cond_scale = cond_scale, timesteps = timesteps)

        # retrieve original unscaled image embed

        text_embeds  = text_cond['text_embed']

        text_embeds  = rearrange(text_embeds, '(b r) d -> b r d', r = num_samples_per_batch)
        image_embeds = rearrange(image_embeds, '(b r) d -> b r d', r = num_samples_per_batch)

        text_image_sims = einsum('b r d, b r d -> b r', l2norm(text_embeds), l2norm(image_embeds))
        top_sim_indices = text_image_sims.topk(k = 1).indices

        top_sim_indices = repeat(top_sim_indices, 'b 1 -> b 1 d', d = image_embed_dim)

        top_image_embeds = image_embeds.gather(1, top_sim_indices)
        return rearrange(top_image_embeds, 'b 1 d -> b d')

    def forward(
        self,
        text           = None,
        image          = None,
        text_embed     = None,  # allow for training on preprocessed CLIP text and image embeddings
        image_embed    = None,
        text_encodings = None,  # as well as CLIP text encodings
        *args,
        **kwargs
    ):
        assert exists(text) ^ exists(text_embed), 'either text or text embedding must be supplied'
        assert exists(image) ^ exists(image_embed), 'either image or image embedding must be supplied'
        assert not (self.condition_on_text_encodings and (not exists(text_encodings) and not exists(text))), 'text encodings must be present if you specified you wish to condition on it on initialization'

        if exists(image):
            image_embed, _ = self.clip.embed_image(image)

        # calculate text conditioning, based on what is passed in

        if exists(text):
            text_embed, text_encodings = self.clip.embed_text(text)

        text_cond = dict(text_embed = text_embed)

        if self.condition_on_text_encodings:
            assert exists(text_encodings), 'text encodings must be present for diffusion prior if specified'
            text_cond = {**text_cond, 'text_encodings': text_encodings}

        # timestep conditioning from DDPM

        batch, device = image_embed.shape[0], image_embed.device
        times = self.noise_scheduler.sample_random_times(batch)

        # scale image embed

        image_embed *= self.image_embed_scale

        # calculate forward loss

        loss = self.p_losses(image_embed, times, text_cond = text_cond, *args, **kwargs)

        return loss


## Decoder

In [ ]:
def NearestUpsample(dim, dim_out = None):
    dim_out = default(dim_out, dim)

    return nn.Sequential(
        nn.Upsample(scale_factor = 2, mode = 'nearest'),
        nn.Conv2d(dim, dim_out, 3, padding = 1)
    )

def Downsample(dim, dim_out = None):
    # https://arxiv.org/abs/2208.03641 shows this is the most optimal way to downsample
    # named SP-conv in the paper, but basically a pixel unshuffle
    dim_out = default(dim_out, dim)
    return nn.Sequential(
        Rearrange('b c (h s1) (w s2) -> b (c s1 s2) h w', s1 = 2, s2 = 2),
        nn.Conv2d(dim * 4, dim_out, 1)
    )


In [ ]:
class PixelShuffleUpsample(nn.Module):
    """
    code shared by @MalumaDev at DALLE2-pytorch for addressing checkboard artifacts
    https://arxiv.org/ftp/arxiv/papers/1707/1707.02937.pdf
    """
    def __init__(self, dim, dim_out = None):
        super().__init__()
        dim_out = default(dim_out, dim)
        conv    = nn.Conv2d(dim, dim_out * 4, 1)

        self.net = nn.Sequential(
            conv,
            nn.SiLU(),
            nn.PixelShuffle(2)
        )

        self.init_conv_(conv)

    def init_conv_(self, conv):
        o, i, h, w  = conv.weight.shape
        conv_weight = torch.empty(o // 4, i, h, w)
        nn.init.kaiming_uniform_(conv_weight)
        conv_weight = repeat(conv_weight, 'o ... -> (o 4) ...')

        conv.weight.data.copy_(conv_weight)
        nn.init.zeros_(conv.bias.data)

    def forward(self, x):
        return self.net(x)

In [ ]:
class WeightStandardizedConv2d(nn.Conv2d):
    """
    https://arxiv.org/abs/1903.10520
    weight standardization purportedly works synergistically with group normalization
    """
    def forward(self, x):
        eps = 1e-5 if x.dtype == torch.float32 else 1e-3

        weight            = self.weight
        flattened_weights = rearrange(weight, 'o ... -> o (...)')

        mean = reduce(weight, 'o ... -> o 1 1 1', 'mean')

        var  = torch.var(flattened_weights, dim = -1, unbiased = False)
        var  = rearrange(var, 'o -> o 1 1 1')

        weight = (weight - mean) * (var + eps).rsqrt()

        return F.conv2d(x, weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        dtype, device = x.dtype, x.device
        assert is_float_dtype(dtype), 'input to sinusoidal pos emb must be a float type'

        half_dim = self.dim // 2
        emb      = math.log(10000) / (half_dim - 1)
        emb      = torch.exp(torch.arange(half_dim, device = device, dtype = dtype) * -emb)
        emb      = rearrange(x, 'i -> i 1') * rearrange(emb, 'j -> 1 j')
        return torch.cat((emb.sin(), emb.cos()), dim = -1).type(dtype)


In [ ]:
class Block(nn.Module):
    def __init__(
        self,
        dim,
        dim_out,
        groups = 8,
        weight_standardization = False
    ):
        super().__init__()
        conv_klass = nn.Conv2d if not weight_standardization else WeightStandardizedConv2d

        self.project = conv_klass(dim, dim_out, 3, padding = 1)
        self.norm    = nn.GroupNorm(groups, dim_out)
        self.act     = nn.SiLU()

    def forward(self, x, scale_shift = None):
        x = self.project(x)
        x = self.norm(x)

        if exists(scale_shift):
            scale, shift = scale_shift
            x = x * (scale + 1) + shift

        x = self.act(x)
        return x


In [ ]:
class ResnetBlock(nn.Module):
    def __init__(
        self,
        dim,
        dim_out,
        *,
        cond_dim               = None,
        time_cond_dim          = None,
        groups                 = 8,
        weight_standardization = False,
        cosine_sim_cross_attn  = False
    ):
        super().__init__()

        self.time_mlp = None

        if exists(time_cond_dim):
            self.time_mlp = nn.Sequential(
                nn.SiLU(),
                nn.Linear(time_cond_dim, dim_out * 2)
            )

        self.cross_attn = None

        if exists(cond_dim):
            self.cross_attn = CrossAttention(
                dim         = dim_out,
                context_dim = cond_dim,
                cosine_sim  = cosine_sim_cross_attn
            )

        self.block1   = Block(dim, dim_out, groups = groups, weight_standardization = weight_standardization)
        self.block2   = Block(dim_out, dim_out, groups = groups, weight_standardization = weight_standardization)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb = None, cond = None):

        scale_shift = None
        if exists(self.time_mlp) and exists(time_emb):
            time_emb    = self.time_mlp(time_emb)
            time_emb    = rearrange(time_emb, 'b c -> b c 1 1')
            scale_shift = time_emb.chunk(2, dim = 1)

        h = self.block1(x, scale_shift = scale_shift)

        if exists(self.cross_attn):
            assert exists(cond)

            h     = rearrange(h, 'b c ... -> b ... c')
            h, ps = pack([h], 'b * c')

            h = self.cross_attn(h, context = cond) + h

            h, = unpack(h, ps, 'b * c')
            h  = rearrange(h, 'b ... c -> b c ...')

        h = self.block2(h)
        return h + self.res_conv(x)


In [ ]:
class CrossAttention(nn.Module):
    def __init__(
            self,
            dim,
            *,
            context_dim      = None,
            dim_head         = 64,
            heads            = 8,
            dropout          = 0.,
            norm_context     = False,
            cosine_sim       = False,
            cosine_sim_scale = 16
        ):
        super().__init__()
        self.cosine_sim = cosine_sim
        self.scale = cosine_sim_scale if cosine_sim else (dim_head ** -0.5)
        self.heads = heads
        inner_dim  = dim_head * heads

        context_dim = default(context_dim, dim)

        self.norm         = LayerNorm(dim)
        self.norm_context = LayerNorm(context_dim) if norm_context else nn.Identity()
        self.dropout      = nn.Dropout(dropout)

        self.null_kv = nn.Parameter(torch.randn(2, dim_head))
        self.to_q    = nn.Linear(dim, inner_dim, bias = False)
        self.to_kv   = nn.Linear(context_dim, inner_dim * 2, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim, bias = False),
            LayerNorm(dim)
        )

    def forward(self, x, context, mask = None):
        b, n, device = *x.shape[:2], x.device

        x       = self.norm(x)
        context = self.norm_context(context)

        q, k, v = (self.to_q(x), *self.to_kv(context).chunk(2, dim = -1))

        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), (q, k, v))

        # add null key / value for classifier free guidance in prior net

        nk, nv = map(lambda t: repeat(t, 'd -> b h 1 d', h = self.heads,  b = b), self.null_kv.unbind(dim = -2))

        k = torch.cat((nk, k), dim = -2)
        v = torch.cat((nv, v), dim = -2)

        if self.cosine_sim:
            q, k = map(l2norm, (q, k))

        q, k = map(lambda t: t * math.sqrt(self.scale), (q, k))

        sim  = einsum('b h i d, b h j d -> b h i j', q, k)
        max_neg_value = -torch.finfo(sim.dtype).max

        if exists(mask):
            mask = F.pad(mask, (1, 0), value = True)
            mask = rearrange(mask, 'b j -> b 1 1 j')
            sim  = sim.masked_fill(~mask, max_neg_value)

        attn = sim.softmax(dim = -1, dtype = torch.float32)
        attn = attn.type(sim.dtype)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

In [ ]:
class LinearAttention(nn.Module):
    def __init__(
            self,
            dim,
            dim_head = 32,
            heads    = 8,
            **kwargs
        ):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        inner_dim  = dim_head * heads
        self.norm  = ChanLayerNorm(dim)

        self.nonlin = nn.GELU()
        self.to_qkv = nn.Conv2d(dim, inner_dim * 3, 1, bias = False)

        self.to_out = nn.Sequential(
            nn.Conv2d(inner_dim, dim, 1, bias = False),
            ChanLayerNorm(dim)
        )

    def forward(self, fmap):
        h, x, y = self.heads, *fmap.shape[-2:]
        seq_len = x * y

        fmap    = self.norm(fmap)
        q, k, v = self.to_qkv(fmap).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> (b h) (x y) c', h = h), (q, k, v))

        q       = q.softmax(dim = -1)
        k       = k.softmax(dim = -2)

        q       = q * self.scale
        v       = l2norm(v)

        k, v    = map(lambda t: t / math.sqrt(seq_len), (k, v))

        context = einsum('b n d, b n e -> b d e', k, v)
        out     = einsum('b n d, b d e -> b n e', q, context)
        out     = rearrange(out, '(b h) (x y) d -> b (h d) x y', h = h, x = x, y = y)

        out     = self.nonlin(out)
        return self.to_out(out)


In [ ]:
class CrossEmbedLayer(nn.Module):
    def __init__(
            self,
            dim_in,
            kernel_sizes,
            dim_out = None,
            stride  = 2
        ):
        super().__init__()
        assert all([*map(lambda t: (t % 2) == (stride % 2), kernel_sizes)])
        dim_out = default(dim_out, dim_in)

        kernel_sizes = sorted(kernel_sizes)
        num_scales   = len(kernel_sizes)

        # calculate the dimension at each scale
        dim_scales = [int(dim_out / (2 ** i)) for i in range(1, num_scales)]
        dim_scales = [*dim_scales, dim_out - sum(dim_scales)]

        self.convs = nn.ModuleList([])
        for kernel, dim_scale in zip(kernel_sizes, dim_scales):
            self.convs.append(nn.Conv2d(dim_in, dim_scale, kernel, stride = stride, padding = (kernel - stride) // 2))

    def forward(self, x):
        fmaps = tuple(map(lambda conv: conv(x), self.convs))
        return torch.cat(fmaps, dim = 1)


In [ ]:
class UpsampleCombiner(nn.Module):
    def __init__(
            self,
            dim,
            *,
            enabled  = False,
            dim_ins  = tuple(),
            dim_outs = tuple()
        ):
        super().__init__()
        assert len(dim_ins) == len(dim_outs)
        self.enabled = enabled

        if not self.enabled:
            self.dim_out = dim
            return

        self.fmap_convs = nn.ModuleList([Block(dim_in, dim_out) for dim_in, dim_out in zip(dim_ins, dim_outs)])
        self.dim_out    = dim + (sum(dim_outs) if len(dim_outs) > 0 else 0)

    def forward(self, x, fmaps = None):
        target_size = x.shape[-1]

        fmaps = default(fmaps, tuple())

        if not self.enabled or len(fmaps) == 0 or len(self.fmap_convs) == 0:
            return x

        fmaps = [resize_image_to(fmap, target_size) for fmap in fmaps]
        outs  = [conv(fmap) for fmap, conv in zip(fmaps, self.fmap_convs)]
        return torch.cat((x, *outs), dim = 1)

### U-Net

In [ ]:
class Unet(nn.Module):
    def __init__(
            self,
            dim,
            *,
            image_embed_dim                     = None,
            text_embed_dim                      = None,
            cond_dim                            = None,
            num_image_tokens                    = 4,
            num_time_tokens                     = 2,
            out_dim                             = None,
            dim_mults                           = (1, 2, 4, 8),
            channels                            = 3,
            channels_out                        = None,
            self_attn                           = False,
            attn_dim_head                       = 32,
            attn_heads                          = 16,
            lowres_cond                         = False,  # for cascading diffusion - https://cascaded-diffusion.github.io/
            lowres_noise_cond                   = False,  # for conditioning on low resolution noising, based on Imagen
            self_cond                           = False,  # set this to True to use the self-conditioning technique from - https://arxiv.org/abs/2208.04202
            sparse_attn                         = False,
            cosine_sim_cross_attn               = False,
            cosine_sim_self_attn                = False,
            attend_at_middle                    = True,    # whether to have a layer of attention at the bottleneck (can turn off for higher resolution in cascading DDPM, before bringing in efficient attention)
            cond_on_text_encodings              = False,
            max_text_len                        = 256,
            cond_on_image_embeds                = False,
            add_image_embeds_to_time            = True,    # alerted by @mhh0318 to a phrase in the paper - "Specifically, we modify the architecture described in Nichol et al. (2021) by projecting and adding CLIP embeddings to the existing timestep embedding"
            init_dim                            = None,
            init_conv_kernel_size               = 7,
            resnet_groups                       = 8,
            resnet_weight_standardization       = False,
            num_resnet_blocks                   = 2,
            init_cross_embed                    = True,
            init_cross_embed_kernel_sizes       = (3, 7, 15),
            cross_embed_downsample              = False,
            cross_embed_downsample_kernel_sizes = (2, 4),
            memory_efficient                    = False,
            scale_skip_connection               = False,
            pixel_shuffle_upsample              = True,
            final_conv_kernel_size              = 1,
            combine_upsample_fmaps              = False, # whether to combine the outputs of all upsample blocks, as in unet squared paper
            checkpoint_during_training          = False,
            **kwargs
        ):
        super().__init__()
        # save locals to take care of some hyperparameters for cascading DDPM

        self._locals = locals()
        del self._locals['self']
        del self._locals['__class__']

        # for eventual cascading diffusion

        self.lowres_cond = lowres_cond

        # whether to do self conditioning

        self.self_cond = self_cond

        # determine dimensions

        self.channels     = channels
        self.channels_out = default(channels_out, channels)

         # initial number of channels depends on
         # (1) low resolution conditioning from cascading ddpm paper, conditioned on previous unet output in the cascade
         # (2) self conditioning (bit diffusion paper)

        init_channels  = channels * (1 + int(lowres_cond) + int(self_cond))

        init_dim       = default(init_dim, dim)

        self.init_conv = CrossEmbedLayer(init_channels, dim_out = init_dim, kernel_sizes = init_cross_embed_kernel_sizes, stride = 1) if init_cross_embed else nn.Conv2d(init_channels, init_dim, init_conv_kernel_size, padding = init_conv_kernel_size // 2)

        dims           = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out         = list(zip(dims[:-1], dims[1:]))

        num_stages     = len(in_out)

        # time, image embeddings, and optional text encoding

        cond_dim       = default(cond_dim, dim)
        time_cond_dim  = dim * 4

        self.to_time_hiddens = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, time_cond_dim),
            nn.GELU()
        )

        self.to_time_tokens = nn.Sequential(
            nn.Linear(time_cond_dim, cond_dim * num_time_tokens),
            Rearrange('b (r d) -> b r d', r = num_time_tokens)
        )

        self.to_time_cond = nn.Sequential(
            nn.Linear(time_cond_dim, time_cond_dim)
        )

        self.image_to_tokens = nn.Sequential(
            nn.Linear(image_embed_dim, cond_dim * num_image_tokens),
            Rearrange('b (n d) -> b n d', n = num_image_tokens)
        ) if cond_on_image_embeds and image_embed_dim != cond_dim else nn.Identity()

        #print(f'[DEBUG] UNet __init__(): image_embed_dim is {image_embed_dim}')

        self.to_image_hiddens = nn.Sequential(
            nn.Linear(image_embed_dim, time_cond_dim),
            nn.GELU()
        ) if cond_on_image_embeds and add_image_embeds_to_time else None

        self.norm_cond     = nn.LayerNorm(cond_dim)
        self.norm_mid_cond = nn.LayerNorm(cond_dim)

        # text encoding conditioning (optional)

        self.text_to_cond   = None
        self.text_embed_dim = None

        if cond_on_text_encodings:
            assert exists(text_embed_dim), 'text_embed_dim must be given to the unet if cond_on_text_encodings is True'
            self.text_to_cond   = nn.Linear(text_embed_dim, cond_dim)
            self.text_embed_dim = text_embed_dim

        # low resolution noise conditiong, based on Imagen's upsampler training technique

        self.lowres_noise_cond    = lowres_noise_cond

        self.to_lowres_noise_cond = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, time_cond_dim),
            nn.GELU(),
            nn.Linear(time_cond_dim, time_cond_dim)
        ) if lowres_noise_cond else None

        # finer control over whether to condition on image embeddings and text encodings
        # so one can have the latter unets in the cascading DDPMs only focus on super-resoluting

        self.cond_on_text_encodings = cond_on_text_encodings
        self.cond_on_image_embeds   = cond_on_image_embeds

        # for classifier free guidance

        self.null_image_embed   = nn.Parameter(torch.randn(1, num_image_tokens, cond_dim))
        self.null_image_hiddens = nn.Parameter(torch.randn(1, time_cond_dim))

        self.max_text_len    = max_text_len
        self.null_text_embed = nn.Parameter(torch.randn(1, max_text_len, cond_dim))

        # whether to scale skip connection, adopted in Imagen

        self.skip_connect_scale = 1. if not scale_skip_connection else (2 ** -0.5)

        # attention related params

        attn_kwargs = dict(heads = attn_heads, dim_head = attn_dim_head, cosine_sim = cosine_sim_self_attn) ############### ERROR ARGUMENT 'cosine_sim' OF Attention

        self_attn   = cast_tuple(self_attn, num_stages)

        create_self_attn = lambda dim: RearrangeToSequence(Residual(Attention(dim, **attn_kwargs)))         ############### ERROR ARGUMENT 'cosine_sim'

        # resnet block class

        resnet_groups          = cast_tuple(resnet_groups, num_stages)
        top_level_resnet_group = first(resnet_groups)

        num_resnet_blocks      = cast_tuple(num_resnet_blocks, num_stages)

        # downsample class

        downsample_klass = Downsample
        if cross_embed_downsample:
            downsample_klass = partial(CrossEmbedLayer, kernel_sizes = cross_embed_downsample_kernel_sizes)

        # upsample class

        upsample_klass = NearestUpsample if not pixel_shuffle_upsample else PixelShuffleUpsample

        # prepare resnet class

        resnet_block = partial(ResnetBlock, cosine_sim_cross_attn = cosine_sim_cross_attn, weight_standardization = resnet_weight_standardization)

        # give memory efficient unet an initial resnet block

        self.init_resnet_block = resnet_block(init_dim, init_dim, time_cond_dim = time_cond_dim, groups = top_level_resnet_group) if memory_efficient else None

        # layers

        self.downs      = nn.ModuleList([])
        self.ups        = nn.ModuleList([])
        num_resolutions = len(in_out)

        skip_connect_dims      = []     # keeping track of skip connection dimensions
        upsample_combiner_dims = []     # keeping track of dimensions for final upsample feature map combiner

        for ind, ((dim_in, dim_out), groups, layer_num_resnet_blocks, layer_self_attn) in enumerate(zip(in_out, resnet_groups, num_resnet_blocks, self_attn)):
            is_first       = ind == 0
            is_last        = ind >= (num_resolutions - 1)
            layer_cond_dim = cond_dim if not is_first else None

            dim_layer = dim_out if memory_efficient else dim_in
            skip_connect_dims.append(dim_layer)

            attention = nn.Identity()
            if layer_self_attn:
                attention = create_self_attn(dim_layer)
            elif sparse_attn:
                attention = Residual(LinearAttention(dim_layer, **attn_kwargs))

            self.downs.append(nn.ModuleList([
                downsample_klass(dim_in, dim_out = dim_out) if memory_efficient else None,
                resnet_block(dim_layer, dim_layer, time_cond_dim = time_cond_dim, groups = groups),
                nn.ModuleList([resnet_block(dim_layer, dim_layer, cond_dim = layer_cond_dim, time_cond_dim = time_cond_dim, groups = groups) for _ in range(layer_num_resnet_blocks)]),
                attention,
                downsample_klass(dim_layer, dim_out = dim_out) if not is_last and not memory_efficient else nn.Conv2d(dim_layer, dim_out, 1)
            ]))

        mid_dim = dims[-1]

        self.mid_block1 = resnet_block(mid_dim, mid_dim, cond_dim = cond_dim, time_cond_dim = time_cond_dim, groups = resnet_groups[-1])
        self.mid_attn   = create_self_attn(mid_dim)
        self.mid_block2 = resnet_block(mid_dim, mid_dim, cond_dim = cond_dim, time_cond_dim = time_cond_dim, groups = resnet_groups[-1])

        for ind, ((dim_in, dim_out), groups, layer_num_resnet_blocks, layer_self_attn) in enumerate(zip(reversed(in_out), reversed(resnet_groups), reversed(num_resnet_blocks), reversed(self_attn))):
            is_last = ind >= (len(in_out) - 1)
            layer_cond_dim = cond_dim if not is_last else None

            skip_connect_dim = skip_connect_dims.pop()

            attention = nn.Identity()
            if layer_self_attn:
                attention = create_self_attn(dim_out)
            elif sparse_attn:
                attention = Residual(LinearAttention(dim_out, **attn_kwargs))

            upsample_combiner_dims.append(dim_out)

            self.ups.append(nn.ModuleList([
                resnet_block(dim_out + skip_connect_dim, dim_out, cond_dim = layer_cond_dim, time_cond_dim = time_cond_dim, groups = groups),
                nn.ModuleList([resnet_block(dim_out + skip_connect_dim, dim_out, cond_dim = layer_cond_dim, time_cond_dim = time_cond_dim, groups = groups)  for _ in range(layer_num_resnet_blocks)]),
                attention,
                upsample_klass(dim_out, dim_in) if not is_last or memory_efficient else nn.Identity()
            ]))

        # whether to combine outputs from all upsample blocks for final resnet block

        self.upsample_combiner = UpsampleCombiner(
            dim = dim,
            enabled = combine_upsample_fmaps,
            dim_ins = upsample_combiner_dims,
            dim_outs = (dim,) * len(upsample_combiner_dims)
        )

        # a final resnet block

        self.final_resnet_block = resnet_block(self.upsample_combiner.dim_out + dim, dim, time_cond_dim = time_cond_dim, groups = top_level_resnet_group)

        out_dim_in = dim + (channels if lowres_cond else 0)

        self.to_out = nn.Conv2d(out_dim_in, self.channels_out, kernel_size = final_conv_kernel_size, padding = final_conv_kernel_size // 2)

        zero_init_(self.to_out) # since both OpenAI and @crowsonkb are doing it

        # whether to checkpoint during training

        self.checkpoint_during_training = checkpoint_during_training

    # if the current settings for the unet are not correct for cascading DDPM, 
    # then reinitialize the unet with the right settings
    def cast_model_parameters(
        self,
        *,
        lowres_cond,
        lowres_noise_cond,
        channels,
        channels_out,
        cond_on_image_embeds,
        cond_on_text_encodings,
    ):
        if lowres_cond == self.lowres_cond and \
            channels == self.channels and \
            cond_on_image_embeds == self.cond_on_image_embeds and \
            cond_on_text_encodings == self.cond_on_text_encodings and \
            lowres_noise_cond == self.lowres_noise_cond and \
            channels_out == self.channels_out:
            return self

        updated_kwargs = dict(
            lowres_cond = lowres_cond,
            channels = channels,
            channels_out = channels_out,
            cond_on_image_embeds = cond_on_image_embeds,
            cond_on_text_encodings = cond_on_text_encodings,
            lowres_noise_cond = lowres_noise_cond
        )

        return self.__class__(**{**self._locals, **updated_kwargs})

    def forward_with_cond_scale(
            self,
            *args,
            cond_scale = 1.,
            **kwargs
        ):
        logits = self.forward(*args, **kwargs)

        if cond_scale == 1:
            return logits

        null_logits = self.forward(*args, text_cond_drop_prob = 1., image_cond_drop_prob = 1., **kwargs)
        return null_logits + (logits - null_logits) * cond_scale

    def forward(
            self,
            x,
            time,
            *,
            image_embed,
            lowres_cond_img      = None,
            lowres_noise_level   = None,
            text_encodings       = None,
            image_cond_drop_prob = 0.,
            text_cond_drop_prob  = 0.,
            blur_sigma           = None,
            blur_kernel_size     = None,
            disable_checkpoint   = False,
            self_cond            = None
        ):

        batch_size, device = x.shape[0], x.device

        # add low resolution conditioning, if present

        assert not (self.lowres_cond and not exists(lowres_cond_img)), 'low resolution conditioning image must be present'

        # concat self conditioning, if needed

        if self.self_cond:
            self_cond = default(self_cond, lambda: torch.zeros_like(x))
            x = torch.cat((x, self_cond), dim = 1)

        # concat low resolution conditioning

        if exists(lowres_cond_img):
            x = torch.cat((x, lowres_cond_img), dim = 1)

        # initial convolution

        x = self.init_conv(x)
        r = x.clone() # final residual

        # time conditioning

        time         = time.type_as(x)
        time_hiddens = self.to_time_hiddens(time)

        time_tokens  = self.to_time_tokens(time_hiddens)
        t            = self.to_time_cond(time_hiddens)

        # low res noise conditioning (similar to time above)

        if exists(lowres_noise_level):
            assert exists(self.to_lowres_noise_cond), 'lowres_noise_cond must be set to True on instantiation of the unet in order to conditiong on lowres noise'
            lowres_noise_level = lowres_noise_level.type_as(x)
            t                  = t + self.to_lowres_noise_cond(lowres_noise_level)

        # conditional dropout

        image_keep_mask = prob_mask_like((batch_size,), 1 - image_cond_drop_prob, device = device)
        text_keep_mask  = prob_mask_like((batch_size,), 1 - text_cond_drop_prob, device = device)

        text_keep_mask  = rearrange(text_keep_mask, 'b -> b 1 1')

        # image embedding to be summed to time embedding
        # discovered by @mhh0318 in the paper

        if exists(image_embed) and exists(self.to_image_hiddens):

            image_hiddens          = self.to_image_hiddens(image_embed)
            image_keep_mask_hidden = rearrange(image_keep_mask, 'b -> b 1')
            null_image_hiddens     = self.null_image_hiddens.to(image_hiddens.dtype)

            image_hiddens = torch.where(
                image_keep_mask_hidden,
                image_hiddens,
                null_image_hiddens
            )

            t = t + image_hiddens

        # mask out image embedding depending on condition dropout
        # for classifier free guidance

        image_tokens = None

        if self.cond_on_image_embeds:
            image_keep_mask_embed = rearrange(image_keep_mask, 'b -> b 1 1')
            image_tokens          = self.image_to_tokens(image_embed)
            null_image_embed      = self.null_image_embed.to(image_tokens.dtype) # for some reason pytorch AMP not working

            image_tokens = torch.where(
                image_keep_mask_embed,
                image_tokens,
                null_image_embed
            )

        # take care of text encodings (optional)

        text_tokens = None

        if exists(text_encodings) and self.cond_on_text_encodings:
            assert text_encodings.shape[0] == batch_size, f'the text encodings being passed into the unet does not have the proper batch size - text encoding shape {text_encodings.shape} - required batch size is {batch_size}'
            assert self.text_embed_dim == text_encodings.shape[-1], f'the text encodings you are passing in have a dimension of {text_encodings.shape[-1]}, but the unet was created with text_embed_dim of {self.text_embed_dim}.'

            text_mask   = torch.any(text_encodings != 0., dim = -1)

            text_tokens = self.text_to_cond(text_encodings)

            text_tokens = text_tokens[:, :self.max_text_len]
            text_mask   = text_mask[:, :self.max_text_len]

            text_tokens_len = text_tokens.shape[1]
            remainder       = self.max_text_len - text_tokens_len

            if remainder > 0:
                text_tokens = F.pad(text_tokens, (0, 0, 0, remainder))
                text_mask   = F.pad(text_mask, (0, remainder), value = False)

            text_mask = rearrange(text_mask, 'b n -> b n 1')

            assert text_mask.shape[0] == text_keep_mask.shape[0], f'text_mask has shape of {text_mask.shape} while text_keep_mask has shape {text_keep_mask.shape}. text encoding is of shape {text_encodings.shape}'
            text_keep_mask = text_mask & text_keep_mask

            null_text_embed = self.null_text_embed.to(text_tokens.dtype) # for some reason pytorch AMP not working

            text_tokens = torch.where(
                text_keep_mask,
                text_tokens,
                null_text_embed
            )

        # main conditioning tokens (c)

        c = time_tokens

        if exists(image_tokens):
            c = torch.cat((c, image_tokens), dim = -2)

        # text and image conditioning tokens (mid_c)
        # to save on compute, only do cross attention based conditioning on the inner most layers of the Unet

        mid_c = c if not exists(text_tokens) else torch.cat((c, text_tokens), dim = -2)

        # normalize conditioning tokens

        c = self.norm_cond(c)
        mid_c = self.norm_mid_cond(mid_c)

        # gradient checkpointing

        can_checkpoint      = self.training and self.checkpoint_during_training and not disable_checkpoint
        apply_checkpoint_fn = make_checkpointable if can_checkpoint else identity

        # make checkpointable modules

        init_resnet_block, mid_block1, mid_attn, mid_block2, final_resnet_block = [maybe(apply_checkpoint_fn)(module) for module in (self.init_resnet_block, self.mid_block1, self.mid_attn, self.mid_block2, self.final_resnet_block)]

        can_checkpoint_cond = lambda m: isinstance(m, ResnetBlock)
        downs, ups = [maybe(apply_checkpoint_fn)(m, condition = can_checkpoint_cond) for m in (self.downs, self.ups)]

        # initial resnet block

        if exists(init_resnet_block):
            x = init_resnet_block(x, t)

        # go through the layers of the unet, down and up

        down_hiddens = []
        up_hiddens   = []

        for pre_downsample, init_block, resnet_blocks, attn, post_downsample in downs:
            if exists(pre_downsample):
                x = pre_downsample(x)

            x = init_block(x, t, c)

            for resnet_block in resnet_blocks:
                x = resnet_block(x, t, c)
                down_hiddens.append(x.contiguous())

            x = attn(x)
            down_hiddens.append(x.contiguous())

            if exists(post_downsample):
                x = post_downsample(x)

        x = mid_block1(x, t, mid_c)

        if exists(mid_attn):
            x = mid_attn(x)

        x = mid_block2(x, t, mid_c)

        connect_skip = lambda fmap: torch.cat((fmap, down_hiddens.pop() * self.skip_connect_scale), dim = 1)

        for init_block, resnet_blocks, attn, upsample in ups:
            x = connect_skip(x)
            x = init_block(x, t, c)

            for resnet_block in resnet_blocks:
                x = connect_skip(x)
                x = resnet_block(x, t, c)

            x = attn(x)

            up_hiddens.append(x.contiguous())
            x = upsample(x)

        x = self.upsample_combiner(x, up_hiddens)

        x = torch.cat((x, r), dim = 1)

        x = final_resnet_block(x, t)

        if exists(lowres_cond_img):
            x = torch.cat((x, lowres_cond_img), dim = 1)

        return self.to_out(x)


In [ ]:
class LowresConditioner(nn.Module):
    def __init__(
        self,
        downsample_first   = True,
        use_blur           = True,
        blur_prob          = 0.5,
        blur_sigma         = 0.6,
        blur_kernel_size   = 3,
        use_noise          = False,
        input_image_range  = None,
        normalize_img_fn   = identity,
        unnormalize_img_fn = identity
    ):
        super().__init__()
        self.downsample_first  = downsample_first
        self.input_image_range = input_image_range

        self.use_blur         = use_blur
        self.blur_prob        = blur_prob
        self.blur_sigma       = blur_sigma
        self.blur_kernel_size = blur_kernel_size

        self.use_noise        = use_noise
        self.normalize_img    = normalize_img_fn
        self.unnormalize_img  = unnormalize_img_fn
        self.noise_scheduler  = NoiseScheduler(beta_schedule = 'linear', timesteps = 1000, loss_type = 'l2') if use_noise else None

    def noise_image(self, cond_fmap, noise_levels = None):
        assert exists(self.noise_scheduler)

        batch     = cond_fmap.shape[0]
        cond_fmap = self.normalize_img(cond_fmap)

        random_noise_levels = default(noise_levels, lambda: self.noise_scheduler.sample_random_times(batch))
        cond_fmap = self.noise_scheduler.q_sample(cond_fmap, t = random_noise_levels, noise = torch.randn_like(cond_fmap))

        cond_fmap = self.unnormalize_img(cond_fmap)
        return cond_fmap, random_noise_levels

    def forward(
        self,
        cond_fmap,
        *,
        target_image_size,
        downsample_image_size = None,
        should_blur           = True,
        blur_sigma            = None,
        blur_kernel_size      = None
    ):
        if self.downsample_first and exists(downsample_image_size):
            cond_fmap = resize_image_to(cond_fmap, downsample_image_size, clamp_range = self.input_image_range, nearest = True)

        # Blur is only applied 50% of the time,
        # section 3.1 in https://arxiv.org/abs/2106.15282

        if self.use_blur and should_blur and random.random() < self.blur_prob:

            # when training, blur the low resolution conditional image

            blur_sigma = default(blur_sigma, self.blur_sigma)
            blur_kernel_size = default(blur_kernel_size, self.blur_kernel_size)

            # allow for drawing a random sigma between lo and hi float values

            if isinstance(blur_sigma, tuple):
                blur_sigma = tuple(map(float, blur_sigma))
                blur_sigma = random.uniform(*blur_sigma)

            # allow for drawing a random kernel size between lo and hi int values

            if isinstance(blur_kernel_size, tuple):
                blur_kernel_size = tuple(map(int, blur_kernel_size))
                kernel_size_lo, kernel_size_hi = blur_kernel_size
                blur_kernel_size = random.randrange(kernel_size_lo, kernel_size_hi + 1)

            cond_fmap = gaussian_blur2d(cond_fmap, cast_tuple(blur_kernel_size, 2), cast_tuple(blur_sigma, 2))

        # resize to target image size

        cond_fmap = resize_image_to(cond_fmap, target_image_size, clamp_range = self.input_image_range, nearest = True)

        # noise conditioning, as done in Imagen
        # as a replacement for the BSR noising, and potentially replace blurring for first stage too

        random_noise_levels = None

        if self.use_noise:
            cond_fmap, random_noise_levels = self.noise_image(cond_fmap)

        # return conditioning feature map, as well as the augmentation noise levels

        return cond_fmap, random_noise_levels

## VQGan-VAE

### Decorators

In [ ]:
def eval_decorator(fn):
    def inner(model, *args, **kwargs):
        was_training = model.training
        model.eval()
        out = fn(model, *args, **kwargs)
        model.train(was_training)
        return out
    return inner

def remove_vgg(fn):
    @wraps(fn)
    def inner(self, *args, **kwargs):
        has_vgg = hasattr(self, 'vgg')
        if has_vgg:
            vgg = self.vgg
            delattr(self, 'vgg')

        out = fn(self, *args, **kwargs)

        if has_vgg:
            self.vgg = vgg

        return out
    return inner

### Keyword argument helpers

In [ ]:
def pick_and_pop(keys, d):
    values = list(map(lambda key: d.pop(key), keys))
    return dict(zip(keys, values))

def group_dict_by_key(cond, d):
    return_val = [dict(),dict()]
    for key in d.keys():
        match = bool(cond(key))
        ind = int(not match)
        return_val[ind][key] = d[key]
    return (*return_val,)

def string_begins_with(prefix, string_input):
    return string_input.startswith(prefix)

def group_by_key_prefix(prefix, d):
    return group_dict_by_key(partial(string_begins_with, prefix), d)

def groupby_prefix_and_trim(prefix, d):
    kwargs_with_prefix, kwargs = group_dict_by_key(partial(string_begins_with, prefix), d)
    kwargs_without_prefix = dict(map(lambda x: (x[0][len(prefix):], x[1]), tuple(kwargs_with_prefix.items())))
    return kwargs_without_prefix, kwargs

### Tensor helper functions

In [ ]:
def log(t, eps = 1e-10):
    return torch.log(t + eps)

def gradient_penalty(images, output, weight = 10):
    batch_size = images.shape[0]
    gradients = torch_grad(outputs = output, inputs = images,
                           grad_outputs = torch.ones(output.size(), device = images.device),
                           create_graph = True, retain_graph = True, only_inputs = True)[0]

    gradients = rearrange(gradients, 'b ... -> b (...)')
    return weight * ((gradients.norm(2, dim = 1) - 1) ** 2).mean()

def l2norm(t):
    return F.normalize(t, dim = -1)

def leaky_relu(p = 0.1):
    return nn.LeakyReLU(0.1)

def stable_softmax(t, dim = -1, alpha = 32 ** 2):
    t = t / alpha
    t = t - torch.amax(t, dim = dim, keepdim = True).detach()
    return (t * alpha).softmax(dim = dim)

def safe_div(numer, denom, eps = 1e-8):
    return numer / (denom + eps)

### GAN losses

In [ ]:
def hinge_discr_loss(fake, real):
    return (F.relu(1 + fake) + F.relu(1 - real)).mean()

def hinge_gen_loss(fake):
    return -fake.mean()

def bce_discr_loss(fake, real):
    return (-log(1 - torch.sigmoid(fake)) - log(torch.sigmoid(real))).mean()

def bce_gen_loss(fake):
    return -log(torch.sigmoid(fake)).mean()

def grad_layer_wrt_loss(loss, layer):
    return torch_grad(
        outputs = loss,
        inputs = layer,
        grad_outputs = torch.ones_like(loss),
        retain_graph = True
    )[0].detach()

### LayerNorm channel-wise

In [ ]:
class LayerNormChan(nn.Module):
    def __init__(
        self,
        dim,
        eps = 1e-5
    ):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        var = torch.var(x, dim = 1, unbiased = False, keepdim = True)
        mean = torch.mean(x, dim = 1, keepdim = True)
        return (x - mean) / (var + self.eps).sqrt() * self.gamma

### VQGan-VAE discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(
        self,
        dims,
        channels = 3,
        groups = 16,
        init_kernel_size = 5
    ):
        super().__init__()
        dim_pairs = zip(dims[:-1], dims[1:])

        self.layers = MList([nn.Sequential(nn.Conv2d(channels, dims[0], init_kernel_size, padding = init_kernel_size // 2), leaky_relu())])

        for dim_in, dim_out in dim_pairs:
            self.layers.append(nn.Sequential(
                nn.Conv2d(dim_in, dim_out, 4, stride = 2, padding = 1),
                nn.GroupNorm(groups, dim_out),
                leaky_relu()
            ))

        dim = dims[-1]
        self.to_logits = nn.Sequential( # return 5 x 5, for PatchGAN-esque training
            nn.Conv2d(dim, dim, 1),
            leaky_relu(),
            nn.Conv2d(dim, 1, 4)
        )

    def forward(self, x):
        for net in self.layers:
            x = net(x)

        return self.to_logits(x)

### Positional encoding

In [ ]:
class ContinuousPositionBias(nn.Module):
    """ from https://arxiv.org/abs/2111.09883 """

    def __init__(self, *, dim, heads, layers = 2):
        super().__init__()
        self.net = MList([])
        self.net.append(nn.Sequential(nn.Linear(2, dim), leaky_relu()))

        for _ in range(layers - 1):
            self.net.append(nn.Sequential(nn.Linear(dim, dim), leaky_relu()))

        self.net.append(nn.Linear(dim, heads))
        self.register_buffer('rel_pos', None, persistent = False)

    def forward(self, x):
        n, device = x.shape[-1], x.device
        fmap_size = int(sqrt(n))

        if not exists(self.rel_pos):
            pos = torch.arange(fmap_size, device = device)
            grid = torch.stack(torch.meshgrid(pos, pos, indexing = 'ij'))
            grid = rearrange(grid, 'c i j -> (i j) c')
            rel_pos = rearrange(grid, 'i c -> i 1 c') - rearrange(grid, 'j c -> 1 j c')
            rel_pos = torch.sign(rel_pos) * torch.log(rel_pos.abs() + 1)
            self.register_buffer('rel_pos', rel_pos, persistent = False)

        rel_pos = self.rel_pos.float()

        for layer in self.net:
            rel_pos = layer(rel_pos)

        bias = rearrange(rel_pos, 'i j h -> h i j')
        return x + bias

### ResNet encoder / decoder

In [ ]:
class ResnetEncDec(nn.Module):
    def __init__(
        self,
        dim,
        *,
        channels               = 3,
        layers                 = 4,
        layer_mults            = None,
        num_resnet_blocks      = 1,
        resnet_groups          = 16,
        first_conv_kernel_size = 5,
        use_attn               = True,
        attn_dim_head          = 64,
        attn_heads             = 8,
        attn_dropout           = 0.,
    ):
        super().__init__()
        assert dim % resnet_groups == 0, f'dimension {dim} must be divisible by {resnet_groups} (groups for the groupnorm)'

        self.layers = layers

        self.encoders = MList([])
        self.decoders = MList([])

        layer_mults = default2(layer_mults, list(map(lambda t: 2 ** t, range(layers))))
        assert len(layer_mults) == layers, 'layer multipliers must be equal to designated number of layers'

        layer_dims = [dim * mult for mult in layer_mults]
        dims = (dim, *layer_dims)

        self.encoded_dim = dims[-1]

        dim_pairs = zip(dims[:-1], dims[1:])

        append = lambda arr, t: arr.append(t)
        prepend = lambda arr, t: arr.insert(0, t)

        if not isinstance(num_resnet_blocks, tuple):
            num_resnet_blocks = (*((0,) * (layers - 1)), num_resnet_blocks)

        if not isinstance(use_attn, tuple):
            use_attn = (*((False,) * (layers - 1)), use_attn)

        assert len(num_resnet_blocks) == layers, 'number of resnet blocks config must be equal to number of layers'
        assert len(use_attn) == layers

        for layer_index, (dim_in, dim_out), layer_num_resnet_blocks, layer_use_attn in zip(range(layers), dim_pairs, num_resnet_blocks, use_attn):
            append(self.encoders, nn.Sequential(nn.Conv2d(dim_in, dim_out, 4, stride = 2, padding = 1), leaky_relu()))
            prepend(self.decoders, nn.Sequential(nn.ConvTranspose2d(dim_out, dim_in, 4, 2, 1), leaky_relu()))

            if layer_use_attn:
                prepend(self.decoders, VQGanAttention(dim = dim_out, heads = attn_heads, dim_head = attn_dim_head, dropout = attn_dropout))

            for _ in range(layer_num_resnet_blocks):
                append(self.encoders, ResBlock(dim_out, groups = resnet_groups))
                prepend(self.decoders, GLUResBlock(dim_out, groups = resnet_groups))

            if layer_use_attn:
                append(self.encoders, VQGanAttention(dim = dim_out, heads = attn_heads, dim_head = attn_dim_head, dropout = attn_dropout))

        prepend(self.encoders, nn.Conv2d(channels, dim, first_conv_kernel_size, padding = first_conv_kernel_size // 2))
        append(self.decoders, nn.Conv2d(dim, channels, 1))

    def get_encoded_fmap_size(self, image_size):
        return image_size // (2 ** self.layers)

    @property
    def last_dec_layer(self):
        return self.decoders[-1].weight

    def encode(self, x):
        for enc in self.encoders:
            x = enc(x)
        return x

    def decode(self, x):
        for dec in self.decoders:
            x = dec(x)
        return x

class GLUResBlock(nn.Module):
    def __init__(self, chan, groups = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(chan, chan * 2, 3, padding = 1),
            nn.GLU(dim = 1),
            nn.GroupNorm(groups, chan),
            nn.Conv2d(chan, chan * 2, 3, padding = 1),
            nn.GLU(dim = 1),
            nn.GroupNorm(groups, chan),
            nn.Conv2d(chan, chan, 1)
        )

    def forward(self, x):
        return self.net(x) + x

class ResBlock(nn.Module):
    def __init__(self, chan, groups = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(chan, chan, 3, padding = 1),
            nn.GroupNorm(groups, chan),
            leaky_relu(),
            nn.Conv2d(chan, chan, 3, padding = 1),
            nn.GroupNorm(groups, chan),
            leaky_relu(),
            nn.Conv2d(chan, chan, 1)
        )

    def forward(self, x):
        return self.net(x) + x

### VQGan attention layer

In [ ]:
class VQGanAttention(nn.Module):
    def __init__(
        self,
        *,
        dim,
        dim_head = 64,
        heads    = 8,
        dropout  = 0.
    ):
        super().__init__()
        self.heads = heads
        self.scale = dim_head ** -0.5
        inner_dim  = heads * dim_head

        self.dropout = nn.Dropout(dropout)
        self.pre_norm = LayerNormChan(dim)

        self.cpb    = ContinuousPositionBias(dim = dim // 4, heads = heads)
        self.to_qkv = nn.Conv2d(dim, inner_dim * 3, 1, bias = False)
        self.to_out = nn.Conv2d(inner_dim, dim, 1, bias = False)

    def forward(self, x):
        h = self.heads
        height, width, residual = *x.shape[-2:], x.clone()

        x       = self.pre_norm(x)

        q, k, v = self.to_qkv(x).chunk(3, dim = 1)

        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h = h), (q, k, v))

        sim  = einsum('b h c i, b h c j -> b h i j', q, k) * self.scale

        sim  = self.cpb(sim)

        attn = stable_softmax(sim, dim = -1)
        attn = self.dropout(attn)

        out  = einsum('b h i j, b h c j -> b h c i', attn, v)
        out  = rearrange(out, 'b h c (x y) -> b (h c) x y', x = height, y = width)
        out  = self.to_out(out)

        return out + residual

### ViT encoder / decoder

In [ ]:
class RearrangeImage(nn.Module):
    def forward(self, x):
        n = x.shape[1]
        w = h = int(sqrt(n))
        return rearrange(x, 'b (h w) ... -> b h w ...', h = h, w = w)

### ViT attention class

In [ ]:
class AttentionViT(nn.Module):
    def __init__(
        self,
        dim,
        *,
        heads    = 8,
        dim_head = 32
    ):
        super().__init__()
        self.norm  = nn.LayerNorm(dim)
        self.heads = heads
        self.scale = dim_head ** -0.5
        inner_dim  = dim_head * heads

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        h = self.heads

        x = self.norm(x)

        q, k, v = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = h), (q, k, v))

        q = q * self.scale
        sim = einsum('b h i d, b h j d -> b h i j', q, k)

        sim = sim - sim.amax(dim = -1, keepdim = True).detach()
        attn = sim.softmax(dim = -1)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)

        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

### Transformer class

In [ ]:
def FeedForwardViT(dim, mult = 4):
    return nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, dim * mult, bias = False),
        nn.GELU(),
        nn.Linear(dim * mult, dim, bias = False)
    )

class Transformer(nn.Module):
    def __init__(
        self,
        dim,
        *,
        layers,
        dim_head = 32,
        heads    = 8,
        ff_mult  = 4
    ):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(layers):
            self.layers.append(nn.ModuleList([
                AttentionViT(dim = dim, dim_head = dim_head, heads = heads),
                FeedForwardViT(dim = dim, mult = ff_mult)
            ]))

        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x

        return self.norm(x)



### ViT encoder / decoder class

In [ ]:
class ViTEncDec(nn.Module):
    def __init__(
        self,
        dim,
        channels   = 3,
        layers     = 4,
        patch_size = 8,
        dim_head   = 32,
        heads      = 8,
        ff_mult    = 4
    ):
        super().__init__()
        self.encoded_dim = dim
        self.patch_size  = patch_size

        input_dim = channels * (patch_size ** 2)

        self.encoder = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_size, p2 = patch_size),
            nn.Linear(input_dim, dim),
            Transformer(
                dim      = dim,
                dim_head = dim_head,
                heads    = heads,
                ff_mult  = ff_mult,
                layers   = layers
            ),
            RearrangeImage(),
            Rearrange('b h w c -> b c h w')
        )

        self.decoder = nn.Sequential(
            Rearrange('b c h w -> b (h w) c'),
            Transformer(
                dim      = dim,
                dim_head = dim_head,
                heads    = heads,
                ff_mult  = ff_mult,
                layers   = layers
            ),
            nn.Sequential(
                nn.Linear(dim, dim * 4, bias = False),
                nn.Tanh(),
                nn.Linear(dim * 4, input_dim, bias = False),
            ),
            RearrangeImage(),
            Rearrange('b h w (p1 p2 c) -> b c (h p1) (w p2)', p1 = patch_size, p2 = patch_size)
        )

    def get_encoded_fmap_size(self, image_size):
        return image_size // self.patch_size

    @property
    def last_dec_layer(self):
        return self.decoder[-3][-1].weight

    def encode(self, x):
        return self.encoder(x)

    def decode(self, x):
        return self.decoder(x)

### Null VQGan-VAE class

In [ ]:
class NullVQGanVAE(nn.Module):
    def __init__(
        self,
        *,
        channels
    ):
        super().__init__()
        self.encoded_dim = channels
        self.layers      = 0

    def get_encoded_fmap_size(self, size):
        return size

    def copy_for_eval(self):
        return self

    def encode(self, x):
        return x

    def decode(self, x):
        return x


### VQGan-VAE class

In [ ]:
class VQGanVAE(nn.Module):
    def __init__(
        self,
        *,
        dim,
        image_size,
        channels             = 3,
        layers               = 4,
        l2_recon_loss        = False,
        use_hinge_loss       = True,
        vgg                  = None,
        vq_codebook_dim      = 256,
        vq_codebook_size     = 512,
        vq_decay             = 0.8,
        vq_commitment_weight = 1.,
        vq_kmeans_init       = True,
        vq_use_cosine_sim    = True,
        use_vgg_and_gan      = True,
        vae_type             = 'resnet',
        discr_layers         = 4,
        **kwargs
    ):
        super().__init__()
        vq_kwargs, kwargs     = groupby_prefix_and_trim('vq_', kwargs)
        encdec_kwargs, kwargs = groupby_prefix_and_trim('encdec_', kwargs)

        self.image_size    = image_size
        self.channels      = channels
        self.codebook_size = vq_codebook_size

        if vae_type == 'resnet':
            enc_dec_klass = ResnetEncDec
        elif vae_type == 'vit':
            enc_dec_klass = ViTEncDec
        else:
            raise ValueError(f'{vae_type} not valid')

        self.enc_dec = enc_dec_klass(
            dim      = dim,
            channels = channels,
            layers   = layers,
            **encdec_kwargs
        )

        self.vq = VQ(
            dim               = self.enc_dec.encoded_dim,
            codebook_dim      = vq_codebook_dim,
            codebook_size     = vq_codebook_size,
            decay             = vq_decay,
            commitment_weight = vq_commitment_weight,
            accept_image_fmap = True,
            kmeans_init       = vq_kmeans_init,
            use_cosine_sim    = vq_use_cosine_sim,
            **vq_kwargs
        )

        # reconstruction loss

        self.recon_loss_fn = F.mse_loss if l2_recon_loss else F.l1_loss

        # turn off GAN and perceptual loss if grayscale

        self.vgg             = None
        self.discr           = None
        self.use_vgg_and_gan = use_vgg_and_gan

        if not use_vgg_and_gan:
            return

        # perceptual loss

        if exists(vgg):
            self.vgg = vgg
        else:
            self.vgg            = torchvision.models.vgg16(pretrained = True)
            self.vgg.classifier = nn.Sequential(*self.vgg.classifier[:-2])

        # gan related losses

        layer_mults = list(map(lambda t: 2 ** t, range(discr_layers)))
        layer_dims  = [dim * mult for mult in layer_mults]
        dims        = (dim, *layer_dims)

        self.discr  = Discriminator(dims = dims, channels = channels)

        self.discr_loss = hinge_discr_loss if use_hinge_loss else bce_discr_loss
        self.gen_loss   = hinge_gen_loss if use_hinge_loss else bce_gen_loss

    @property
    def encoded_dim(self):
        return self.enc_dec.encoded_dim

    def get_encoded_fmap_size(self, image_size):
        return self.enc_dec.get_encoded_fmap_size(image_size)

    def copy_for_eval(self):
        device   = next(self.parameters()).device
        vae_copy = copy.deepcopy(self.cpu())

        if vae_copy.use_vgg_and_gan:
            del vae_copy.discr
            del vae_copy.vgg

        vae_copy.eval()
        return vae_copy.to(device)

    @remove_vgg
    def state_dict(self, *args, **kwargs):
        return super().state_dict(*args, **kwargs)

    @remove_vgg
    def load_state_dict(self, *args, **kwargs):
        return super().load_state_dict(*args, **kwargs)

    @property
    def codebook(self):
        return self.vq.codebook

    def encode(self, fmap):
        fmap = self.enc_dec.encode(fmap)
        return fmap

    def decode(self, fmap, return_indices_and_loss = False):
        fmap, indices, commit_loss = self.vq(fmap)

        fmap = self.enc_dec.decode(fmap)

        if not return_indices_and_loss:
            return fmap

        return fmap, indices, commit_loss

    def forward(
        self,
        img,
        return_loss          = False,
        return_discr_loss    = False,
        return_recons        = False,
        add_gradient_penalty = True
    ):
        batch, channels, height, width, device = *img.shape, img.device
        assert height == self.image_size and width == self.image_size, 'height and width of input image must be equal to {self.image_size}'
        assert channels == self.channels, 'number of channels on image or sketch is not equal to the channels set on this VQGanVAE'

        fmap = self.encode(img)

        fmap, indices, commit_loss = self.decode(fmap, return_indices_and_loss = True)

        if not return_loss and not return_discr_loss:
            return fmap

        assert return_loss ^ return_discr_loss, 'you should either return autoencoder loss or discriminator loss, but not both'

        # whether to return discriminator loss

        if return_discr_loss:
            assert exists(self.discr), 'discriminator must exist to train it'

            fmap.detach_()
            img.requires_grad_()

            fmap_discr_logits, img_discr_logits = map(self.discr, (fmap, img))

            discr_loss = self.discr_loss(fmap_discr_logits, img_discr_logits)

            if add_gradient_penalty:
                gp   = gradient_penalty(img, img_discr_logits)
                loss = discr_loss + gp

            if return_recons:
                return loss, fmap

            return loss

        # reconstruction loss

        recon_loss = self.recon_loss_fn(fmap, img)

        # early return if training on grayscale

        if not self.use_vgg_and_gan:
            if return_recons:
                return recon_loss, fmap

            return recon_loss

        # perceptual loss

        img_vgg_input  = img
        fmap_vgg_input = fmap

        if img.shape[1] == 1:
            # handle grayscale for vgg
            img_vgg_input, fmap_vgg_input = map(lambda t: repeat(t, 'b 1 ... -> b c ...', c = 3), (img_vgg_input, fmap_vgg_input))

        img_vgg_feats   = self.vgg(img_vgg_input)
        recon_vgg_feats = self.vgg(fmap_vgg_input)
        perceptual_loss = F.mse_loss(img_vgg_feats, recon_vgg_feats)

        # generator loss

        gen_loss = self.gen_loss(self.discr(fmap))

        # calculate adaptive weight

        last_dec_layer = self.enc_dec.last_dec_layer

        norm_grad_wrt_gen_loss        = grad_layer_wrt_loss(gen_loss, last_dec_layer).norm(p = 2)
        norm_grad_wrt_perceptual_loss = grad_layer_wrt_loss(perceptual_loss, last_dec_layer).norm(p = 2)

        adaptive_weight = safe_div(norm_grad_wrt_perceptual_loss, norm_grad_wrt_gen_loss)
        adaptive_weight.clamp_(max = 1e4)

        # combine losses

        loss = recon_loss + perceptual_loss + commit_loss + adaptive_weight * gen_loss

        if return_recons:
            return loss, fmap

        return loss

### Decoder class

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        unet,
        *,
        clip                            = None,
        image_size                      = None,
        channels                        = 3,
        vae                             = tuple(),
        timesteps                       = 1000,
        sample_timesteps                = None,
        image_cond_drop_prob            = 0.1,
        text_cond_drop_prob             = 0.5,
        loss_type                       = 'l2',
        beta_schedule                   = None,
        predict_x_start                 = False,
        predict_v                       = False,
        predict_x_start_for_latent_diffusion = False,
        image_sizes                     = None,     # for cascading ddpm, image size at each stage
        random_crop_sizes               = None,     # whether to random crop the image at that stage in the cascade (super resoluting convolutions at the end may be able to generalize on smaller crops)
        use_noise_for_lowres_cond       = False,    # whether to use Imagen-like noising for low resolution conditioning  
        use_blur_for_lowres_cond        = True,     # whether to use the blur conditioning used in the original cascading ddpm paper, as well as DALL-E2
        lowres_downsample_first         = True,     # cascading ddpm - resizes to lower resolution, then to next conditional resolution + blur
        blur_prob                       = 0.5,      # cascading ddpm - when training, the gaussian blur is only applied 50% of the time
        blur_sigma                      = 0.6,      # cascading ddpm - blur sigma
        blur_kernel_size                = 3,        # cascading ddpm - blur kernel size
        lowres_noise_sample_level       = 0.2,      # in imagen paper, they use a 0.2 noise level at sample time for low resolution conditioning
        clip_denoised                   = True,
        clip_x_start                    = True,
        clip_adapter_overrides          = dict(),
        learned_variance                = True,
        learned_variance_constrain_frac = False,
        vb_loss_weight                  = 0.001,
        unconditional                   = False,    # set to True for generating images without conditioning
        auto_normalize_img              = True,     # whether to take care of normalizing the image from [0, 1] to [-1, 1] and back automatically - you can turn this off if you want to pass in the [-1, 1] ranged image yourself from the dataloader
        use_dynamic_thres               = False,    # from the Imagen paper
        dynamic_thres_percentile        = 0.95,
        p2_loss_weight_gamma            = 0.,       # p2 loss weight, from https://arxiv.org/abs/2204.00227 - 0 is equivalent to weight of 1 across time - 1. is recommended
        p2_loss_weight_k                = 1,
        ddim_sampling_eta               = 0.        # can be set to 0. for deterministic sampling afaict
        ):
        super().__init__()

        # clip

        self.clip = None
        if exists(clip):
            assert not unconditional, 'clip must not be given if doing unconditional image training'
            assert channels == clip.image_channels, f'channels of image ({channels}) should be equal to the channels that CLIP accepts ({clip.image_channels})'

            if isinstance(clip, CLIP):
                clip = XClipAdapter(clip, **clip_adapter_overrides)
            elif isinstance(clip, CoCa):
                clip = CoCaAdapter(clip, **clip_adapter_overrides)

            freeze_model_and_make_eval_(clip)
            assert isinstance(clip, BaseClipAdapter)

            self.clip = clip

        # determine image size, with image_size and image_sizes taking precedence

        if exists(image_size) or exists(image_sizes):
            assert exists(image_size) ^ exists(image_sizes), 'only one of image_size or image_sizes must be given'
            image_size = default(image_size, lambda: image_sizes[-1])
        elif exists(clip):
            image_size = clip.image_size
        else:
            raise Error('either image_size, image_sizes, or clip must be given to decoder')

        # channels

        self.channels = channels

        # normalize and unnormalize image functions

        self.normalize_img   = normalize_neg_one_to_one if auto_normalize_img else identity
        self.unnormalize_img = unnormalize_zero_to_one  if auto_normalize_img else identity

        # verify conditioning method

        unets              = cast_tuple(unet)
        num_unets          = len(unets)
        self.num_unets     = num_unets

        self.unconditional = unconditional

        # automatically take care of ensuring that first unet is unconditional
        # while the rest of the unets are conditioned on the low resolution image produced by previous unet

        vaes = pad_tuple_to_length(
            cast_tuple(vae),
            len(unets),
            fillvalue = NullVQGanVAE(channels = self.channels)
        )

        # whether to use learned variance, defaults to True for the first unet in the cascade, as in paper

        learned_variance      = pad_tuple_to_length(
            cast_tuple(learned_variance), 
            len(unets), 
            fillvalue = False
        )
        self.learned_variance = learned_variance
        self.learned_variance_constrain_frac = learned_variance_constrain_frac # whether to constrain the output of the network (the interpolation fraction) from 0 to 1
        self.vb_loss_weight   = vb_loss_weight

        # default and validate conditioning parameters

        use_noise_for_lowres_cond = cast_tuple(use_noise_for_lowres_cond, num_unets - 1, validate = False)
        use_blur_for_lowres_cond  = cast_tuple(use_blur_for_lowres_cond, num_unets - 1, validate = False)

        if len(use_noise_for_lowres_cond) < num_unets:
            use_noise_for_lowres_cond = (False, *use_noise_for_lowres_cond)

        if len(use_blur_for_lowres_cond) < num_unets:
            use_blur_for_lowres_cond = (False, *use_blur_for_lowres_cond)

        assert not use_noise_for_lowres_cond[0], 'first unet will never need low res noise conditioning'
        assert not use_blur_for_lowres_cond[0], 'first unet will never need low res blur conditioning'

        assert num_unets == 1 or all((use_noise or use_blur) for use_noise, use_blur in zip(use_noise_for_lowres_cond[1:], use_blur_for_lowres_cond[1:]))

        # Instantiate U-Nets and VAEs

        self.unets = nn.ModuleList([])
        self.vaes  = nn.ModuleList([])

        for ind, (one_unet, one_vae, one_unet_learned_var, lowres_noise_cond) in \
                enumerate(zip(unets, vaes, learned_variance, use_noise_for_lowres_cond)):
            assert isinstance(one_unet, Unet)
            assert isinstance(one_vae, (VQGanVAE, NullVQGanVAE))

            is_first = ind == 0
            latent_dim = one_vae.encoded_dim if exists(one_vae) else None

            unet_channels     = default(latent_dim, self.channels)
            unet_channels_out = unet_channels * (1 if not one_unet_learned_var else 2)

            one_unet = one_unet.cast_model_parameters(
                lowres_cond            = not is_first,
                lowres_noise_cond      = lowres_noise_cond,
                cond_on_image_embeds   = not unconditional and is_first,
                cond_on_text_encodings = not unconditional and one_unet.cond_on_text_encodings,
                channels               = unet_channels,
                channels_out           = unet_channels_out
            )

            self.unets.append(one_unet)
            self.vaes.append(one_vae.copy_for_eval())

        # sampling timesteps, defaults to non-ddim with full timesteps sampling

        self.sample_timesteps = cast_tuple(sample_timesteps, num_unets)
        self.ddim_sampling_eta = ddim_sampling_eta

        # create noise schedulers per unet

        if not exists(beta_schedule):
            beta_schedule    = ('cosine', *(('cosine',) * max(num_unets - 2, 0)), *(('linear',) * int(num_unets > 1)))

        beta_schedule        = cast_tuple(beta_schedule, num_unets)
        p2_loss_weight_gamma = cast_tuple(p2_loss_weight_gamma, num_unets)

        self.noise_schedulers = nn.ModuleList([])

        for ind, (unet_beta_schedule, unet_p2_loss_weight_gamma, sample_timesteps) in enumerate(zip(beta_schedule, p2_loss_weight_gamma, self.sample_timesteps)):
            assert not exists(sample_timesteps) or sample_timesteps <= timesteps, f'sampling timesteps {sample_timesteps} must be less than or equal to the number of training timesteps {timesteps} for unet {ind + 1}'

            noise_scheduler = NoiseScheduler(
                beta_schedule        = unet_beta_schedule,
                timesteps            = timesteps,
                loss_type            = loss_type,
                p2_loss_weight_gamma = unet_p2_loss_weight_gamma,
                p2_loss_weight_k     = p2_loss_weight_k
            )

            self.noise_schedulers.append(noise_scheduler)

        # unet image sizes

        image_sizes = default(image_sizes, (image_size,))
        image_sizes = tuple(sorted(set(image_sizes)))

        assert self.num_unets == len(image_sizes), f'you did not supply the correct number of u-nets ({self.num_unets}) for resolutions {image_sizes}'
        self.image_sizes     = image_sizes
        self.sample_channels = cast_tuple(self.channels, len(image_sizes))

        # random crop sizes (for super-resoluting unets at the end of cascade?)

        self.random_crop_sizes = cast_tuple(random_crop_sizes, len(image_sizes))
        assert not exists(self.random_crop_sizes[0]), 'you would not need to randomly crop the image for the base unet'

        # predict x0 config

        self.predict_x_start = cast_tuple(predict_x_start, len(unets)) if not predict_x_start_for_latent_diffusion else tuple(map(lambda t: isinstance(t, VQGanVAE), self.vaes))

        # predict v

        self.predict_v = cast_tuple(predict_v, len(unets))

        # input image range

        self.input_image_range = (-1. if not auto_normalize_img else 0., 1.)

        # cascading ddpm related stuff

        lowres_conditions = tuple(map(lambda t: t.lowres_cond, self.unets))
        assert lowres_conditions == (False, *((True,) * (num_unets - 1))), 'the first unet must be unconditioned (by low resolution image), and the rest of the unets must have `lowres_cond` set to True'

        self.lowres_conds = nn.ModuleList([])

        for unet_index, use_noise, use_blur in zip(range(num_unets), use_noise_for_lowres_cond, use_blur_for_lowres_cond):
            if unet_index == 0:
                self.lowres_conds.append(None)
                continue

            lowres_cond = LowresConditioner(
                downsample_first   = lowres_downsample_first,
                use_blur           = use_blur,
                use_noise          = use_noise,
                blur_prob          = blur_prob,
                blur_sigma         = blur_sigma,
                blur_kernel_size   = blur_kernel_size,
                input_image_range  = self.input_image_range,
                normalize_img_fn   = self.normalize_img,
                unnormalize_img_fn = self.unnormalize_img
            )

            self.lowres_conds.append(lowres_cond)

        self.lowres_noise_sample_level = lowres_noise_sample_level

        # classifier free guidance

        self.image_cond_drop_prob      = image_cond_drop_prob
        self.text_cond_drop_prob       = text_cond_drop_prob
        self.can_classifier_guidance   = image_cond_drop_prob > 0. or text_cond_drop_prob > 0.

        # whether to clip when sampling

        self.clip_denoised             = clip_denoised
        self.clip_x_start              = clip_x_start

        # dynamic thresholding settings, if clipping denoised during sampling

        self.use_dynamic_thres         = use_dynamic_thres
        self.dynamic_thres_percentile  = dynamic_thres_percentile

        # device tracker

        self.register_buffer('_dummy', torch.Tensor([True]), persistent = False)

    @property
    def device(self):
        return self._dummy.device

    @property
    def condition_on_text_encodings(self):
        return any([unet.cond_on_text_encodings for unet in self.unets if isinstance(unet, Unet)])

    def get_unet(self, unet_number):
        assert 0 < unet_number <= self.num_unets
        index = unet_number - 1
        return self.unets[index]

    def parse_unet_output(self, learned_variance, output):
        var_interp_frac_unnormalized = None

        if learned_variance:
            output, var_interp_frac_unnormalized = output.chunk(2, dim = 1)

        return UnetOutput(output, var_interp_frac_unnormalized)

    @contextmanager
    def one_unet_in_gpu(self, unet_number = None, unet = None):
        assert exists(unet_number) ^ exists(unet)

        if exists(unet_number):
            unet = self.get_unet(unet_number)

        # devices

        cuda, cpu = torch.device('cuda:0'), torch.device('cpu')

        self.cuda()

        devices = [module_device(unet) for unet in self.unets]

        self.unets.to(cpu)
        unet.to(cuda)

        yield

        for unet, device in zip(self.unets, devices):
            unet.to(device)

    def dynamic_threshold(self, x):
        """
        Proposed in https://arxiv.org/abs/2205.11487 as an improved
        clamping in the setting of classifier free guidance.
        """
        # s is the threshold amount
        # static thresholding would just be s = 1
        s = 1.
        if self.use_dynamic_thres:
            s = torch.quantile(
                rearrange(x, 'b ... -> b (...)').abs(),
                self.dynamic_thres_percentile,
                dim = -1
            )

            s.clamp_(min = 1.)
            s = s.view(-1, *((1,) * (x.ndim - 1)))

        # clip by threshold, depending on whether static or dynamic
        x = x.clamp(-s, s) / s
        return x

    def p_mean_variance(self, unet, x, t, image_embed, noise_scheduler, text_encodings = None, lowres_cond_img = None, self_cond = None, clip_denoised = True, predict_x_start = False, predict_v = False, learned_variance = False, cond_scale = 1., model_output = None, lowres_noise_level = None):
        assert not (cond_scale != 1. and not self.can_classifier_guidance), 'the decoder was not trained with conditional dropout, and thus one cannot use classifier free guidance (cond_scale anything other than 1)'

        model_output = default(model_output, lambda: unet.forward_with_cond_scale(x, t, image_embed = image_embed, text_encodings = text_encodings, cond_scale = cond_scale, lowres_cond_img = lowres_cond_img, self_cond = self_cond, lowres_noise_level = lowres_noise_level))

        pred, var_interp_frac_unnormalized = self.parse_unet_output(learned_variance, model_output)

        if predict_v:
            x_start = noise_scheduler.predict_start_from_v(x, t = t, v = pred)
        elif predict_x_start:
            x_start = pred
        else:
            x_start = noise_scheduler.predict_start_from_noise(x, t = t, noise = pred)

        if clip_denoised:
            x_start = self.dynamic_threshold(x_start)

        model_mean, posterior_variance, posterior_log_variance = noise_scheduler.q_posterior(x_start=x_start, x_t=x, t=t)

        if learned_variance:
            # if learned variance, posterio variance and posterior log variance are predicted by the network
            # by an interpolation of the max and min log beta values
            # eq 15 - https://arxiv.org/abs/2102.09672
            min_log = extract(noise_scheduler.posterior_log_variance_clipped, t, x.shape)
            max_log = extract(torch.log(noise_scheduler.betas), t, x.shape)
            var_interp_frac = unnormalize_zero_to_one(var_interp_frac_unnormalized)

            if self.learned_variance_constrain_frac:
                var_interp_frac = var_interp_frac.sigmoid()

            posterior_log_variance = var_interp_frac * max_log + (1 - var_interp_frac) * min_log
            posterior_variance = posterior_log_variance.exp()

        return model_mean, posterior_variance, posterior_log_variance, x_start

    @torch.no_grad()
    def p_sample(self, unet, x, t, image_embed, noise_scheduler, text_encodings = None, cond_scale = 1., lowres_cond_img = None, self_cond = None, predict_x_start = False, predict_v = False, learned_variance = False, clip_denoised = True, lowres_noise_level = None):
        b, *_, device = *x.shape, x.device

        #print(f'[DEBUG] p_sample():  x device is {device}')

        model_mean, _, model_log_variance, x_start = self.p_mean_variance(
            unet, 
            x = x,
            t = t,
            image_embed = image_embed,
            text_encodings = text_encodings,
            cond_scale = cond_scale,
            lowres_cond_img = lowres_cond_img,
            self_cond = self_cond,
            clip_denoised = clip_denoised,
            predict_x_start = predict_x_start, 
            predict_v = predict_v, 
            noise_scheduler = noise_scheduler, 
            learned_variance = learned_variance, 
            lowres_noise_level = lowres_noise_level
        )
        noise = torch.randn_like(x)
        # no noise when t == 0
        nonzero_mask = (1 - (t == 0).float()).reshape(b, *((1,) * (len(x.shape) - 1)))
        pred = model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise
        return pred, x_start

    @torch.no_grad()
    def p_sample_loop_ddpm(
            self,
            unet,
            shape,
            image_embed,
            noise_scheduler,
            predict_x_start        = False,
            predict_v              = False,
            learned_variance       = False,
            clip_denoised          = True,
            lowres_cond_img        = None,
            text_encodings         = None,
            cond_scale             = 1,
            is_latent_diffusion    = False,
            lowres_noise_level     = None,
            inpaint_image          = None,
            inpaint_mask           = None,
            inpaint_resample_times = 5
        ):

        device         = self.device
        b              = shape[0]
        img            = torch.randn(shape, device = device)

        x_start        = None # for self-conditioning

        is_inpaint     = exists(inpaint_image)
        resample_times = inpaint_resample_times if is_inpaint else 1

        if is_inpaint:
            inpaint_image = self.normalize_img(inpaint_image)
            inpaint_image = resize_image_to(inpaint_image, shape[-1], nearest = True)
            inpaint_mask  = rearrange(inpaint_mask, 'b h w -> b 1 h w').float()
            inpaint_mask  = resize_image_to(inpaint_mask, shape[-1], nearest = True)
            inpaint_mask  = inpaint_mask.bool()

        if not is_latent_diffusion:
            lowres_cond_img = maybe(self.normalize_img)(lowres_cond_img)

        for time in tqdm(reversed(range(0, noise_scheduler.num_timesteps)), desc = 'sampling loop time step', total = noise_scheduler.num_timesteps):
            is_last_timestep = time == 0

            for r in reversed(range(0, resample_times)):
                is_last_resample_step = r == 0

                times = torch.full((b,), time, device = device, dtype = torch.long)

                if is_inpaint:
                    # following the repaint paper
                    # https://arxiv.org/abs/2201.09865
                    noised_inpaint_image = noise_scheduler.q_sample(inpaint_image, t = times)
                    img = (img * ~inpaint_mask) + (noised_inpaint_image * inpaint_mask)

                self_cond = x_start if unet.self_cond else None

                img, x_start = self.p_sample(
                    unet,
                    img,
                    times,
                    image_embed = image_embed,
                    text_encodings = text_encodings,
                    cond_scale = cond_scale,
                    self_cond = self_cond,
                    lowres_cond_img = lowres_cond_img,
                    lowres_noise_level = lowres_noise_level,
                    predict_x_start = predict_x_start,
                    predict_v = predict_v,
                    noise_scheduler = noise_scheduler,
                    learned_variance = learned_variance,
                    clip_denoised = clip_denoised
                )

                if is_inpaint and not (is_last_timestep or is_last_resample_step):
                    # in repaint, you renoise and resample up to 10 times every step
                    img = noise_scheduler.q_sample_from_to(img, times - 1, times)

        if is_inpaint:
            img = (img * ~inpaint_mask) + (inpaint_image * inpaint_mask)

        unnormalize_img = self.unnormalize_img(img)
        return unnormalize_img

    @torch.no_grad()
    def p_sample_loop_ddim(
            self,
            unet,
            shape,
            image_embed,
            noise_scheduler,
            timesteps,
            eta                    = 1.,
            predict_x_start        = False,
            predict_v              = False,
            learned_variance       = False,
            clip_denoised          = True,
            lowres_cond_img        = None,
            text_encodings         = None,
            cond_scale             = 1,
            is_latent_diffusion    = False,
            lowres_noise_level     = None,
            inpaint_image          = None,
            inpaint_mask           = None,
            inpaint_resample_times = 5
        ):
        batch, device, total_timesteps, alphas, eta = shape[0], self.device, noise_scheduler.num_timesteps, noise_scheduler.alphas_cumprod, self.ddim_sampling_eta

        times      = torch.linspace(0., total_timesteps, steps = timesteps + 2)[:-1]

        times      = list(reversed(times.int().tolist()))
        time_pairs = list(zip(times[:-1], times[1:]))
        time_pairs = list(filter(lambda t: t[0] > t[1], time_pairs))

        is_inpaint     = exists(inpaint_image)
        resample_times = inpaint_resample_times if is_inpaint else 1

        if is_inpaint:
            inpaint_image = self.normalize_img(inpaint_image)
            inpaint_image = resize_image_to(inpaint_image, shape[-1], nearest = True)
            inpaint_mask  = rearrange(inpaint_mask, 'b h w -> b 1 h w').float()
            inpaint_mask  = resize_image_to(inpaint_mask, shape[-1], nearest = True)
            inpaint_mask  = inpaint_mask.bool()

        img = torch.randn(shape, device = device)

        x_start = None # for self-conditioning

        if not is_latent_diffusion:
            lowres_cond_img = maybe(self.normalize_img)(lowres_cond_img)

        for time, time_next in tqdm(time_pairs, desc = 'sampling loop time step'):
            is_last_timestep = time_next == 0

            for r in reversed(range(0, resample_times)):
                is_last_resample_step = r == 0

                alpha = alphas[time]
                alpha_next = alphas[time_next]

                time_cond = torch.full((batch,), time, device = device, dtype = torch.long)

                if is_inpaint:
                    # following the repaint paper
                    # https://arxiv.org/abs/2201.09865
                    noised_inpaint_image = noise_scheduler.q_sample(inpaint_image, t = time_cond)
                    img = (img * ~inpaint_mask) + (noised_inpaint_image * inpaint_mask)

                self_cond = x_start if unet.self_cond else None

                unet_output = unet.forward_with_cond_scale(img, time_cond, image_embed = image_embed, text_encodings = text_encodings, cond_scale = cond_scale, self_cond = self_cond, lowres_cond_img = lowres_cond_img, lowres_noise_level = lowres_noise_level)

                pred, _ = self.parse_unet_output(learned_variance, unet_output)

                # predict x0

                if predict_v:
                    x_start = noise_scheduler.predict_start_from_v(img, t = time_cond, v = pred)
                elif predict_x_start:
                    x_start = pred
                else:
                    x_start = noise_scheduler.predict_start_from_noise(img, t = time_cond, noise = pred)

                # maybe clip x0

                if clip_denoised:
                    x_start = self.dynamic_threshold(x_start)

                # predict noise

                pred_noise = noise_scheduler.predict_noise_from_start(img, t = time_cond, x0 = x_start)

                c1 = eta * ((1 - alpha / alpha_next) * (1 - alpha_next) / (1 - alpha)).sqrt()
                c2 = ((1 - alpha_next) - torch.square(c1)).sqrt()
                noise = torch.randn_like(img) if not is_last_timestep else 0.

                img = x_start * alpha_next.sqrt() + \
                      c1 * noise + \
                      c2 * pred_noise

                if is_inpaint and not (is_last_timestep or is_last_resample_step):
                    # in repaint, you renoise and resample up to 10 times every step
                    time_next_cond = torch.full((batch,), time_next, device = device, dtype = torch.long)
                    img = noise_scheduler.q_sample_from_to(img, time_next_cond, time_cond)

        if exists(inpaint_image):
            img = (img * ~inpaint_mask) + (inpaint_image * inpaint_mask)

        img = self.unnormalize_img(img)
        return img

    @torch.no_grad()
    def p_sample_loop(self, *args, noise_scheduler, timesteps = None, **kwargs):
        num_timesteps = noise_scheduler.num_timesteps

        timesteps = default(timesteps, num_timesteps)
        assert timesteps <= num_timesteps
        is_ddim = timesteps < num_timesteps

        if not is_ddim:
            return self.p_sample_loop_ddpm(*args, noise_scheduler = noise_scheduler, **kwargs)

        return self.p_sample_loop_ddim(*args, noise_scheduler = noise_scheduler, timesteps = timesteps, **kwargs)

    def p_losses(self, unet, x_start, times, *, image_embed, noise_scheduler, lowres_cond_img = None, text_encodings = None, predict_x_start = False, predict_v = False, noise = None, learned_variance = False, clip_denoised = False, is_latent_diffusion = False, lowres_noise_level = None):
        noise = default(noise, lambda: torch.randn_like(x_start))

        # normalize to [-1, 1]

        if not is_latent_diffusion:
            x_start = self.normalize_img(x_start)
            lowres_cond_img = maybe(self.normalize_img)(lowres_cond_img)

        # get x_t

        x_noisy = noise_scheduler.q_sample(x_start = x_start, t = times, noise = noise)

        # unet kwargs

        unet_kwargs = dict(
            image_embed        = image_embed,
            text_encodings     = text_encodings,
            lowres_cond_img    = lowres_cond_img,
            lowres_noise_level = lowres_noise_level,
        )

        # self conditioning

        self_cond = None

        if unet.self_cond and random.random() < 0.5:
            with torch.no_grad():
                unet_output  = unet(x_noisy, times, **unet_kwargs)
                self_cond, _ = self.parse_unet_output(learned_variance, unet_output)
                self_cond    = self_cond.detach()

        # forward to get model prediction

        unet_output = unet(
            x_noisy,
            times,
            **unet_kwargs,
            self_cond            = self_cond,
            image_cond_drop_prob = self.image_cond_drop_prob,
            text_cond_drop_prob  = self.text_cond_drop_prob,
        )

        pred, _ = self.parse_unet_output(learned_variance, unet_output)

        #print(f'[DEBUG] Decode p_losses(): pred shape is {pred.shape}')

        if predict_v:
            target = noise_scheduler.calculate_v(x_start, times, noise)
        elif predict_x_start:
            target = x_start
        else:
            target = noise

        loss = noise_scheduler.loss_fn(pred, target, reduction = 'none')
        loss = reduce(loss, 'b ... -> b (...)', 'mean')

        loss = noise_scheduler.p2_reweigh_loss(loss, times)

        loss = loss.mean()

        if not learned_variance:
            # return simple loss if not using learned variance
            return loss

        # most of the code below is transcribed from
        # https://github.com/hojonathanho/diffusion/blob/master/diffusion_tf/diffusion_utils_2.py
        # the Improved DDPM paper then further modified it so that the mean is detached (shown a couple lines before), and weighted to be smaller than the l1 or l2 "simple" loss
        # it is questionable whether this is really needed, looking at some of the figures in the paper, but may as well stay faithful to their implementation

        # if learning the variance, also include the extra weight kl loss

        true_mean, _, true_log_variance_clipped = noise_scheduler.q_posterior(x_start = x_start, x_t = x_noisy, t = times)
        model_mean, _, model_log_variance, _ = self.p_mean_variance(unet, x = x_noisy, t = times, image_embed = image_embed, noise_scheduler = noise_scheduler, clip_denoised = clip_denoised, learned_variance = True, model_output = unet_output)

        # kl loss with detached model predicted mean, for stability reasons as in paper

        detached_model_mean = model_mean.detach()

        kl = normal_kl(true_mean, true_log_variance_clipped, detached_model_mean, model_log_variance)
        kl = meanflat(kl) * NAT

        decoder_nll = -discretized_gaussian_log_likelihood(x_start, means = detached_model_mean, log_scales = 0.5 * model_log_variance)
        decoder_nll = meanflat(decoder_nll) * NAT

        # at the first timestep return the decoder NLL, otherwise return KL(q(x_{t-1}|x_t,x_0) || p(x_{t-1}|x_t))

        vb_losses = torch.where(times == 0, decoder_nll, kl)

        # weight the vb loss smaller, for stability, as in the paper (recommended 0.001)

        vb_loss = vb_losses.mean() * self.vb_loss_weight

        return loss + vb_loss

    @torch.no_grad()
    @eval_decorator
    def sample(
            self,
            image                   = None,
            image_embed             = None,
            text                    = None,
            text_encodings          = None,
            batch_size              = 1,
            cond_scale              = 1.,
            start_at_unet_number    = 1,
            stop_at_unet_number     = None,
            distributed             = False,
            inpaint_image           = None,
            inpaint_mask            = None,
            inpaint_resample_times  = 5,
            one_unet_in_gpu_at_time = True
        ):
        assert self.unconditional or exists(image_embed), 'image embed must be present on sampling from decoder unless if trained unconditionally'

        if not self.unconditional:
            batch_size = image_embed.shape[0]

        if exists(text) and not exists(text_encodings) and not self.unconditional:
            assert exists(self.clip)
            _, text_encodings = self.clip.embed_text(text)

        assert not (self.condition_on_text_encodings and not exists(text_encodings)), 'text or text encodings must be passed into decoder if specified'
        assert not (not self.condition_on_text_encodings and exists(text_encodings)), 'decoder specified not to be conditioned on text, yet it is presented'

        assert not (exists(inpaint_image) ^ exists(inpaint_mask)), 'inpaint_image and inpaint_mask (boolean mask of [batch, height, width]) must be both given for inpainting'

        img = None
        if start_at_unet_number > 1:
            # Then we are not generating the first image and one must have been passed in
            assert exists(image), 'image must be passed in if starting at unet number > 1'
            assert image.shape[0] == batch_size, 'image must have batch size of {} if starting at unet number > 1'.format(batch_size)
            prev_unet_output_size = self.image_sizes[start_at_unet_number - 2]
            img = resize_image_to(image, prev_unet_output_size, nearest = True)

        is_cuda    = next(self.parameters()).is_cuda

        #print(f'[DEBUG] Decoder device is cuda? {is_cuda}')

        num_unets  = self.num_unets
        cond_scale = cast_tuple(cond_scale, num_unets)

        for unet_number, unet, vae, channel, image_size, predict_x_start, predict_v, learned_variance, noise_scheduler, lowres_cond, sample_timesteps, unet_cond_scale in tqdm(zip(range(1, num_unets + 1), self.unets, self.vaes, self.sample_channels, self.image_sizes, self.predict_x_start, self.predict_v, self.learned_variance, self.noise_schedulers, self.lowres_conds, self.sample_timesteps, cond_scale)):
            if unet_number < start_at_unet_number:
                continue  # It's the easiest way to do it

            context = self.one_unet_in_gpu(unet = unet) if is_cuda and one_unet_in_gpu_at_time else null_context()

            with context:
                # prepare low resolution conditioning for upsamplers

                lowres_cond_img = lowres_noise_level = None
                shape = (batch_size, channel, image_size, image_size)

                if unet.lowres_cond:
                    lowres_cond_img = resize_image_to(img, target_image_size = image_size, clamp_range = self.input_image_range, nearest = True)

                    if lowres_cond.use_noise:
                        lowres_noise_level = torch.full((batch_size,), int(self.lowres_noise_sample_level * 1000), dtype = torch.long, device = self.device)
                        lowres_cond_img, _ = lowres_cond.noise_image(lowres_cond_img, lowres_noise_level)

                # latent diffusion

                is_latent_diffusion = isinstance(vae, VQGanVAE)
                image_size          = vae.get_encoded_fmap_size(image_size)
                shape               = (batch_size, vae.encoded_dim, image_size, image_size)

                lowres_cond_img     = maybe(vae.encode)(lowres_cond_img)

                # denoising loop for image

                img = self.p_sample_loop(
                    unet,
                    shape,
                    image_embed            = image_embed,
                    text_encodings         = text_encodings,
                    cond_scale             = unet_cond_scale,
                    predict_x_start        = predict_x_start,
                    predict_v              = predict_v,
                    learned_variance       = learned_variance,
                    clip_denoised          = not is_latent_diffusion,
                    lowres_cond_img        = lowres_cond_img,
                    lowres_noise_level     = lowres_noise_level,
                    is_latent_diffusion    = is_latent_diffusion,
                    noise_scheduler        = noise_scheduler,
                    timesteps              = sample_timesteps,
                    inpaint_image          = inpaint_image,
                    inpaint_mask           = inpaint_mask,
                    inpaint_resample_times = inpaint_resample_times
                )

                img = vae.decode(img)

            if exists(stop_at_unet_number) and stop_at_unet_number == unet_number:
                break

        return img

    def forward(
            self,
            image,
            text           = None,
            image_embed    = None,
            text_encodings = None,
            unet_number    = None,
            return_lowres_cond_image = False # whether to return the low resolution conditioning images, for debugging upsampler purposes
        ):
        assert not (self.num_unets > 1 and not exists(unet_number)), f'you must specify which unet you want trained, from a range of 1 to {self.num_unets}, if you are training cascading DDPM (multiple unets)'
        unet_number = default(unet_number, 1)
        unet_index  = unet_number - 1

        unet        = self.get_unet(unet_number)

        vae                 = self.vaes[unet_index]
        noise_scheduler     = self.noise_schedulers[unet_index]
        lowres_conditioner  = self.lowres_conds[unet_index]
        target_image_size   = self.image_sizes[unet_index]
        predict_x_start     = self.predict_x_start[unet_index]
        predict_v           = self.predict_v[unet_index]
        random_crop_size    = self.random_crop_sizes[unet_index]
        learned_variance    = self.learned_variance[unet_index]
        b, c, h, w, device, = *image.shape, image.device

        #print(f'[DEBUG] Decode forward(): #1 image_embed shape is {image_embed.shape}')

        assert image.shape[1] == self.channels
        assert h >= target_image_size and w >= target_image_size

        times = torch.randint(0, noise_scheduler.num_timesteps, (b,), device = device, dtype = torch.long)

        if not exists(image_embed) and not self.unconditional:
            assert exists(self.clip), 'if you want to derive CLIP image embeddings automatically, you must supply `clip` to the decoder on init'
            image_embed, _ = self.clip.embed_image(image)

        if exists(text) and not exists(text_encodings) and not self.unconditional:
            assert exists(self.clip), 'if you are passing in raw text, you need to supply `clip` to the decoder'
            _, text_encodings = self.clip.embed_text(text)

        assert not (self.condition_on_text_encodings and not exists(text_encodings)), 'text or text encodings must be passed into decoder if specified'
        assert not (not self.condition_on_text_encodings and exists(text_encodings)), 'decoder specified not to be conditioned on text, yet it is presented'

        lowres_cond_img, lowres_noise_level = lowres_conditioner(image, target_image_size = target_image_size, downsample_image_size = self.image_sizes[unet_index - 1]) if exists(lowres_conditioner) else (None, None)
        image = resize_image_to(image, target_image_size, nearest = True)

        if exists(random_crop_size):
            aug = K.RandomCrop((random_crop_size, random_crop_size), p = 1.)

            # make sure low res conditioner and image both get augmented the same way
            # detailed https://kornia.readthedocs.io/en/latest/augmentation.module.html?highlight=randomcrop#kornia.augmentation.RandomCrop
            image = aug(image)
            lowres_cond_img = aug(lowres_cond_img, params = aug._params)

        is_latent_diffusion = not isinstance(vae, NullVQGanVAE)

        vae.eval()
        with torch.no_grad():
            image = vae.encode(image)
            lowres_cond_img = maybe(vae.encode)(lowres_cond_img)

        losses = self.p_losses(
            unet,
            image,
            times,
            image_embed         = image_embed,
            text_encodings      = text_encodings,
            lowres_cond_img     = lowres_cond_img,
            predict_x_start     = predict_x_start,
            predict_v           = predict_v,
            learned_variance    = learned_variance,
            is_latent_diffusion = is_latent_diffusion,
            noise_scheduler     = noise_scheduler,
            lowres_noise_level  = lowres_noise_level
        )

        if not return_lowres_cond_image:
            return losses

        return losses, lowres_cond_img

## DALL-E 2 main class

In [ ]:
class DALLE2(nn.Module):
    def __init__(
        self,
        *,
        prior,
        decoder,
        prior_num_samples = 2
    ):
        super().__init__()
        assert isinstance(prior, DiffusionPrior)
        assert isinstance(decoder, Decoder)
        self.prior   = prior
        self.decoder = decoder

        self.prior_num_samples      = prior_num_samples
        self.decoder_need_text_cond = self.decoder.condition_on_text_encodings

        self.to_pil = T.ToPILImage()

    @torch.no_grad()
    @eval_decorator
    def forward(
        self,
        text,
        cond_scale        = 1.,
        prior_cond_scale  = 1.,
        return_pil_images = False
    ):
        device   = module_device(self)
        one_text = isinstance(text, str) or (not is_list_str(text) and text.shape[0] == 1)

        if isinstance(text, str) or is_list_str(text):
            text = [text] if not isinstance(text, (list, tuple)) else text
            text = tokenizer.tokenize(text).to(device)

        image_embed = self.prior.sample(text, num_samples_per_batch = self.prior_num_samples, cond_scale = prior_cond_scale)

        text_cond = text if self.decoder_need_text_cond else None
        images    = self.decoder.sample(image_embed = image_embed, text = text_cond, cond_scale = cond_scale)

        if return_pil_images:
            images = list(map(self.to_pil, images.unbind(dim = 0)))

        if one_text:
            return first(images)

        return images


## Optimizer

In [ ]:
def separate_weight_decayable_params(params):
    wd_params, no_wd_params = [], []
    for param in params:
        param_list = no_wd_params if param.ndim < 2 else wd_params
        param_list.append(param)
    return wd_params, no_wd_params


def get_optimizer(
    params,
    lr    = 1e-4,
    wd    = 1e-2,
    betas = (0.9, 0.99),
    eps   = 1e-8,
    filter_by_requires_grad = False,
    group_wd_params         = True,
    **kwargs
):
    if filter_by_requires_grad:
        params = list(filter(lambda t: t.requires_grad, params))

    if wd == 0:
        return Adam(params, lr = lr, betas = betas, eps = eps)

    if group_wd_params:
        wd_params, no_wd_params = separate_weight_decayable_params(params)

        params = [
            {'params': wd_params},
            {'params': no_wd_params, 'weight_decay': 0},
        ]

    return AdamW(params, lr = lr, weight_decay = wd, betas = betas, eps = eps)


## Tokenizer

Taken from `https://github.com/openai/CLIP/blob/main/clip/simple_tokenizer.py` to give users a quick easy start to training DALL-E without doing BPE.

### OpenAI simple tokenizer

In [ ]:
#@lru_cache()
#def default_bpe():
#    return os.path.join(os.path.dirname(os.path.abspath(__file__)), "data/bpe_simple_vocab_16e6.txt")

@lru_cache()
def default_bpe():
    bpe_dir = os.path.abspath('')
    bpe_dir = os.path.join(bpe_dir, "../dalle2_pytorch/dalle2_pytorch/data/bpe_simple_vocab_16e6.txt")
    print(f'[INFO] Simple tokenizer vocabulary path: {bpe_dir}')
    return bpe_dir

@lru_cache()
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(2 ** 8):
        if b not in bs:
            bs.append(b)
            cs.append(2 ** 8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

def get_pairs(word):
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

def basic_clean(text):
    text = ftfy.fix_text(text)
    text = html.unescape(html.unescape(text))
    return text.strip()

def whitespace_clean(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


class SimpleTokenizer(object):
    def __init__(self, bpe_path = default_bpe()):
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        merges = Path(bpe_path).read_text(encoding='utf8').split('\n')
        merges = merges[1:49152 - 256 - 2 + 1]
        merges = [tuple(merge.split()) for merge in merges]
        vocab  = list(bytes_to_unicode().values())
        vocab  = vocab + [v + '</w>' for v in vocab]
        for merge in merges:
            vocab.append(''.join(merge))
        vocab.extend(['<|startoftext|>', '<|endoftext|>'])

        self.vocab_size = 49408

        self.encoder   = dict(zip(vocab, range(len(vocab))))
        self.decoder   = {v: k for k, v in self.encoder.items()}
        self.bpe_ranks = dict(zip(merges, range(len(merges))))
        self.cache     = {'<|startoftext|>': '<|startoftext|>', '<|endoftext|>': '<|endoftext|>'}
        self.pat       = re.compile(
            r"""<\|startoftext\|>|<\|endoftext\|>|'s|'t|'re|'ve|'m|'ll|'d|[\p{L}]+|[\p{N}]|[^\s\p{L}\p{N}]+""",
            re.IGNORECASE)

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word  = tuple(token[:-1]) + (token[-1] + '</w>',)
        pairs = get_pairs(word)

        if not pairs:
            return token + '</w>'

        while True:
            bigram = min(pairs, key=lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                    new_word.append(first + second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = ' '.join(word)
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        text = whitespace_clean(basic_clean(text)).lower()
        for token in re.findall(self.pat, text):
            token = ''.join(self.byte_encoder[b] for b in token.encode('utf-8'))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(' '))
        return bpe_tokens

    def decode(self, tokens, remove_start_end = True, pad_tokens = set()):
        if torch.is_tensor(tokens):
            tokens = tokens.tolist()

        if remove_start_end:
            tokens = [token for token in tokens if token not in (49406, 40407, 0)]
        text = ''.join([self.decoder[token] for token in tokens if token not in pad_tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode('utf-8', errors="replace").replace('</w>', ' ')
        return text

    def tokenize(self, texts, context_length = 256, truncate_text = False):
        if isinstance(texts, str):
            texts = [texts]

        all_tokens = [self.encode(text) for text in texts]
        result = torch.zeros(len(all_tokens), context_length, dtype=torch.long)

        for i, tokens in enumerate(all_tokens):
            if len(tokens) > context_length:
                if truncate_text:
                    tokens = tokens[:context_length]
                else:
                    raise RuntimeError(f"Input {texts[i]} is too long for context length {context_length}")
            result[i, :len(tokens)] = torch.tensor(tokens)

        return result

### YTTM tokenizer

In [ ]:
class YttmTokenizer:
    def __init__(self, bpe_path = None):
        bpe_path = Path(bpe_path)
        assert bpe_path.exists(), f'BPE json path {str(bpe_path)} does not exist'

        self.yttm = import_or_print_error('youtokentome', 'you need to install youtokentome by `pip install youtokentome`')

        tokenizer       = self.yttm.BPE(model = str(bpe_path))
        self.tokenizer  = tokenizer
        self.vocab_size = tokenizer.vocab_size()

    def decode(self, tokens, pad_tokens = set()):
        if torch.is_tensor(tokens):
            tokens = tokens.tolist()

        return self.tokenizer.decode(tokens, ignore_ids = pad_tokens.union({0}))

    def encode(self, texts):
        encoded = self.tokenizer.encode(texts, output_type = self.yttm.OutputType.ID)
        return list(map(torch.tensor, encoded))

    def tokenize(self, texts, context_length = 256, truncate_text = False):
        if isinstance(texts, str):
            texts = [texts]

        all_tokens = self.encode(texts)

        result = torch.zeros(len(all_tokens), context_length, dtype=torch.long)
        for i, tokens in enumerate(all_tokens):
            if len(tokens) > context_length:
                if truncate_text:
                    tokens = tokens[:context_length]
                else:
                    raise RuntimeError(f"Input {texts[i]} is too long for context length {context_length}")
            result[i, :len(tokens)] = torch.tensor(tokens)

        return result

### Decorator

In [ ]:
def cast_torch_tensor(fn):
    @wraps(fn)
    def inner(model, *args, **kwargs):
        device      = kwargs.pop('_device', next(model.parameters()).device)
        cast_device = kwargs.pop('_cast_device', True)
        cast_deepspeed_precision = kwargs.pop('_cast_deepspeed_precision', True)

        kwargs_keys = kwargs.keys()
        all_args    = (*args, *kwargs.values())
        split_kwargs_index = len(all_args) - len(kwargs_keys)
        all_args    = tuple(map(lambda t: torch.from_numpy(t) if exists(t) and isinstance(t, np.ndarray) else t, all_args))

        if cast_device:
            all_args = tuple(map(lambda t: t.to(device) if exists(t) and isinstance(t, torch.Tensor) else t, all_args))

        if cast_deepspeed_precision:
            try:
                accelerator = model.accelerator
                if accelerator is not None and accelerator.distributed_type == DistributedType.DEEPSPEED:
                    cast_type_map = {
                        "fp16": torch.half,
                        "bf16": torch.bfloat16,
                        "no":   torch.float
                    }
                    precision_type = cast_type_map[accelerator.mixed_precision]
                    all_args       = tuple(map(lambda t: t.to(precision_type) if exists(t) and isinstance(t, torch.Tensor) else t, all_args))
            except AttributeError:
                # Then this model doesn't have an accelerator
                pass

        args, kwargs_values = all_args[:split_kwargs_index], all_args[split_kwargs_index:]
        kwargs              = dict(tuple(zip(kwargs_keys, kwargs_values)))

        out = fn(model, *args, **kwargs)
        return out
    return inner

### Diffusion prior trainer class

In [ ]:
def prior_sample_in_chunks(fn):
    @wraps(fn)
    def inner(self, *args, max_batch_size = None, **kwargs):
        if not exists(max_batch_size):
            return fn(self, *args, **kwargs)

        outputs = [fn(self, *chunked_args, **chunked_kwargs) for _, (chunked_args, chunked_kwargs) in split_args_and_kwargs(*args, split_size = max_batch_size, **kwargs)]
        return torch.cat(outputs, dim = 0)
    return inner


class DiffusionPriorTrainer(nn.Module):
    def __init__(
        self,
        diffusion_prior,
        accelerator            = None,
        use_ema                = True,
        lr                     = 3e-4,
        wd                     = 1e-2,
        eps                    = 1e-6,
        max_grad_norm          = None,
        group_wd_params        = True,
        warmup_steps           = None,
        cosine_decay_max_steps = None,
        **kwargs
    ):
        super().__init__()
        assert isinstance(diffusion_prior, DiffusionPrior)

        ema_kwargs, kwargs         = groupby_prefix_and_trim('ema_', kwargs)
        accelerator_kwargs, kwargs = groupby_prefix_and_trim('accelerator_', kwargs)

        if not exists(accelerator):
            accelerator = Accelerator(**accelerator_kwargs)

        # assign some helpful member vars

        self.accelerator      = accelerator
        self.text_conditioned = diffusion_prior.condition_on_text_encodings

        # setting the device

        self.device = accelerator.device
        diffusion_prior.to(self.device)

        # save model

        self.diffusion_prior = diffusion_prior

        # mixed precision checks

        if (
            exists(self.accelerator)
            and self.accelerator.distributed_type == DistributedType.DEEPSPEED 
            and self.diffusion_prior.clip is not None# setup the learning rate scheduler
            ):
            # Then we need to make sure clip is using the correct precision or else deepspeed will error
            cast_type_map = {
                "fp16": torch.half,
                "bf16": torch.bfloat16,
                "no":   torch.float
            }
            precision_type = cast_type_map[accelerator.mixed_precision]
            assert precision_type == torch.float, "DeepSpeed currently only supports float32 precision when using on the fly embedding generation from clip"
            self.diffusion_prior.clip.to(precision_type)

        # optimizer stuff

        self.optim_kwargs = dict(lr=lr, wd=wd, eps=eps, group_wd_params=group_wd_params)

        self.optimizer = get_optimizer(
            self.diffusion_prior.parameters(),
            **self.optim_kwargs,
            **kwargs
        )

        if exists(cosine_decay_max_steps):
            self.scheduler = CosineAnnealingLR(self.optimizer, T_max = cosine_decay_max_steps)
        else:
            self.scheduler = LambdaLR(self.optimizer, lr_lambda = lambda _: 1.0)

        self.warmup_scheduler = warmup.LinearWarmup(self.optimizer, warmup_period = warmup_steps) if exists(warmup_steps) else None

        # distribute the model if using HFA

        self.diffusion_prior, self.optimizer, self.scheduler = self.accelerator.prepare(self.diffusion_prior, self.optimizer, self.scheduler)

        # exponential moving average stuff

        self.use_ema = use_ema

        if self.use_ema:
            self.ema_diffusion_prior = EMA(self.accelerator.unwrap_model(self.diffusion_prior), **ema_kwargs)

        # gradient clipping if needed

        self.max_grad_norm = max_grad_norm

        # track steps internally

        self.register_buffer('step', torch.tensor([0], device = self.device))


    '''
    The save() method saves the diffusion prior state, the optimizer state, 
    the scheduler state, the learning rate warmup scheduler state, and the training step.
    '''
    def save(self, path, overwrite = True, **kwargs):

        # only save on the main process
        if self.accelerator.is_main_process:
            print(f"Saving checkpoint at step: {self.step.item()}")
            path = Path(path)
            assert not (path.exists() and not overwrite)
            path.parent.mkdir(parents = True, exist_ok = True)

            # FIXME: LambdaLR can't be saved due to pickling issues
            save_obj = dict(
                optimizer        = self.optimizer.state_dict(),
                scheduler        = self.scheduler.state_dict(),
                warmup_scheduler = self.warmup_scheduler,
                model            = self.accelerator.unwrap_model(self.diffusion_prior).state_dict(),
                version          = version.parse(__version__),
                step             = self.step,
                **kwargs
            )

            if self.use_ema:
                save_obj = {
                    **save_obj,
                    'ema':       self.ema_diffusion_prior.state_dict(),
                    'ema_model': self.ema_diffusion_prior.ema_model.state_dict() # save the ema model specifically for easy ema-only reload
                }

            torch.save(save_obj, str(path))

    '''
    The load() method loads a checkpoint of a diffusion prior training state, 
    including the diffusion prior state, the optimizer state, the scheduler state, 
    the learning rate warmup scheduler state, the training step, and the EMA diffusion 
    prior state.
    '''
    def load(self, path_or_state, overwrite_lr = True, strict = True):
        """
        Load a checkpoint of a diffusion prior trainer.

        Will load the entire trainer, including the optimizer and EMA.

        Params:
            - path_or_state (str | torch): a path to the DiffusionPriorTrainer checkpoint file
            - overwrite_lr (bool): wether or not to overwrite the stored LR with the LR specified in the new trainer
            - strict (bool): kwarg for `torch.nn.Module.load_state_dict`, will force an exact checkpoint match

        Returns:
            loaded_obj (dict): The loaded checkpoint dictionary
        """

        # all processes need to load checkpoint. no restriction here
        if isinstance(path_or_state, str):
            path = Path(path_or_state)
            assert path.exists()
            loaded_obj = torch.load(str(path), map_location=self.device)

        elif isinstance(path_or_state, dict):
            loaded_obj = path_or_state

        if version.parse(__version__) != loaded_obj['version']:
            print(f'loading saved diffusion prior at version {loaded_obj["version"]} but current package version is at {__version__}')

        # unwrap the model when loading from checkpoint
        self.accelerator.unwrap_model(self.diffusion_prior).load_state_dict(loaded_obj['model'], strict = strict)
        self.step.copy_(torch.ones_like(self.step, device=self.device) * loaded_obj['step'].to(self.device))

        self.optimizer.load_state_dict(loaded_obj['optimizer'])
        self.scheduler.load_state_dict(loaded_obj['scheduler'])

        # set warmupstep
        if exists(self.warmup_scheduler):
            self.warmup_scheduler.last_step = self.step.item()

        # ensure new lr is used if different from old one
        if overwrite_lr:
            new_lr = self.optim_kwargs["lr"]

            for group in self.optimizer.param_groups:
                group["lr"] = new_lr if group["lr"] > 0.0 else 0.0

        if self.use_ema:
            assert 'ema' in loaded_obj
            self.ema_diffusion_prior.load_state_dict(loaded_obj['ema'], strict = strict)
            # below might not be necessary, but I had a suspicion that this wasn't being loaded correctly
            self.ema_diffusion_prior.ema_model.load_state_dict(loaded_obj["ema_model"])

        return loaded_obj


    '''
    The update() method performs an optimizer step over the prior model, the an optimizer 
    step over the EMA model, and resets the accumulated gradients.
    '''
    def update(self):

        if exists(self.max_grad_norm):
            self.accelerator.clip_grad_norm_(self.diffusion_prior.parameters(), self.max_grad_norm)

        self.optimizer.step()
        self.optimizer.zero_grad()

        # accelerator will occasionally skip optimizer steps in a "dynamic loss scaling strategy"
        if not self.accelerator.optimizer_step_was_skipped:
            sched_context = self.warmup_scheduler.dampening if exists(self.warmup_scheduler) else nullcontext
            with sched_context():
                self.scheduler.step()

        if self.use_ema:
            self.ema_diffusion_prior.update()

        self.step += 1

    '''
    The p_sample_loop() method draws image embeddings samples from noise, by running a 
    reverse denoising loop through the DDPM or DDIM samplers and using the main prior model 
    or the EMA prior model to predict the  image embedding of the next timestep image given 
    the current image embedding. This method delegates the work on the prior model's 
    p_sample_loop() method.
    '''
    @torch.no_grad()
    @cast_torch_tensor
    @prior_sample_in_chunks
    def p_sample_loop(self, *args, **kwargs):
        model = self.ema_diffusion_prior.ema_model if self.use_ema else self.diffusion_prior
        return model.p_sample_loop(*args, **kwargs)

    '''
    sample() also delegates the work on the prior model's sample() method to generate 
    multiple image embedding samples for each given text prompt, calculate the similarity
    score between each prompt and the multiple image embeddings generated from that prompt, 
    returning the embedding with the best score for each given prompt.
    '''
    @torch.no_grad()
    @cast_torch_tensor
    @prior_sample_in_chunks
    def sample(self, *args, **kwargs):
        model = self.ema_diffusion_prior.ema_model if self.use_ema else self.diffusion_prior
        return model.sample(*args, **kwargs)

    '''
    sample_batch_size() is also a wrapper for the method with the same name of the 
    DiffusionPrior class. So, it is used to draw a batch of fully denoised image embeddings 
    with the help of the selected prior.
    '''
    @torch.no_grad()
    def sample_batch_size(self, *args, **kwargs):
        model = self.ema_diffusion_prior.ema_model if self.use_ema else self.diffusion_prior
        return model.sample_batch_size(*args, **kwargs)

    '''
    Given a text prompt, embed_text() resorts to the configured CLIP model to obtain 
    a text embedding for that prompt.
    '''
    @torch.no_grad()
    @cast_torch_tensor
    @prior_sample_in_chunks
    def embed_text(self, *args, **kwargs):
        return self.accelerator.unwrap_model(self.diffusion_prior).clip.embed_text(*args, **kwargs)

    '''
    The forward computation consist in iterating over all training batches, and for each batch 
    delegated the work on the forward() method of the DiffusionPrior class to compute image 
    embeddings, calculate text embeddings, sample a batch of DDPM timesteps to condition the 
    diffusion process, scale the image embeddings, and finally compute the loss. It also 
    accumulates the loss and calculates the gradients of the loss relative to the prior network 
    weights and biases.
    '''
    @cast_torch_tensor
    def forward(
        self,
        *args,
        max_batch_size = None,
        **kwargs
    ):
        total_loss = 0.

        for chunk_size_frac, (chunked_args, chunked_kwargs) in split_args_and_kwargs(*args, split_size = max_batch_size, **kwargs):
            with self.accelerator.autocast():
                loss = self.diffusion_prior(*chunked_args, **chunked_kwargs)
                loss = loss * chunk_size_frac

            total_loss += loss.item()

            if self.training:
                self.accelerator.backward(loss)

        return total_loss

### Decoder trainer class

In [ ]:
def decoder_sample_in_chunks(fn):
    @wraps(fn)
    def inner(self, *args, max_batch_size = None, **kwargs):
        if not exists(max_batch_size):
            return fn(self, *args, **kwargs)

        if self.decoder.unconditional:
            batch_size = kwargs.get('batch_size')
            batch_sizes = num_to_groups(batch_size, max_batch_size)
            outputs = [fn(self, *args, **{**kwargs, 'batch_size': sub_batch_size}) for sub_batch_size in batch_sizes]
        else:
            outputs = [fn(self, *chunked_args, **chunked_kwargs) for _, (chunked_args, chunked_kwargs) in split_args_and_kwargs(*args, split_size = max_batch_size, **kwargs)]

        return torch.cat(outputs, dim = 0)
    return inner

class DecoderTrainer(nn.Module):
    def __init__(
            self,
            decoder,
            accelerator            = None,
            dataloaders            = None,
            use_ema                = True,
            lr                     = 1e-4,
            wd                     = 1e-2,
            eps                    = 1e-8,
            warmup_steps           = None,
            cosine_decay_max_steps = None,
            max_grad_norm          = 0.5,
            amp                    = False,
            group_wd_params        = True,
            **kwargs
        ):
        super().__init__()
        assert isinstance(decoder, Decoder)
        ema_kwargs, kwargs = groupby_prefix_and_trim('ema_', kwargs)

        self.accelerator   = default(accelerator, Accelerator)

        self.num_unets     = len(decoder.unets)

        self.use_ema       = use_ema
        self.ema_unets     = nn.ModuleList([])

        self.amp           = amp

        # be able to finely customize learning rate, weight decay
        # per unet

        lr, wd, eps, warmup_steps, cosine_decay_max_steps = map(partial(cast_tuple3, length = self.num_unets), (lr, wd, eps, warmup_steps, cosine_decay_max_steps))

        assert all([unet_lr <= 1e-2 for unet_lr in lr]), 'your learning rate is too high, recommend sticking with 1e-4, at most 5e-4'

        optimizers        = []
        schedulers        = []
        warmup_schedulers = []

        for unet, unet_lr, unet_wd, unet_eps, unet_warmup_steps, unet_cosine_decay_max_steps in zip(decoder.unets, lr, wd, eps, warmup_steps, cosine_decay_max_steps):
            if isinstance(unet, nn.Identity):
                optimizers.append(None)
                schedulers.append(None)
                warmup_schedulers.append(None)
            else:
                optimizer = get_optimizer(
                    unet.parameters(),
                    lr              = unet_lr,
                    wd              = unet_wd,
                    eps             = unet_eps,
                    group_wd_params = group_wd_params,
                    **kwargs
                )

                optimizers.append(optimizer)

                if exists(unet_cosine_decay_max_steps):
                    scheduler = CosineAnnealingLR(optimizer, T_max = unet_cosine_decay_max_steps)
                else:
                    scheduler = LambdaLR(optimizer, lr_lambda = lambda step: 1.0)

                warmup_scheduler = warmup.LinearWarmup(optimizer, warmup_period = unet_warmup_steps) if exists(unet_warmup_steps) else None
                warmup_schedulers.append(warmup_scheduler)

                schedulers.append(scheduler)

            if self.use_ema:
                self.ema_unets.append(EMA(unet, **ema_kwargs))

        # gradient clipping if needed

        self.max_grad_norm = max_grad_norm

        self.register_buffer('steps', torch.tensor([0] * self.num_unets))

        if self.accelerator.distributed_type == DistributedType.DEEPSPEED and decoder.clip is not None:
            # Then we need to make sure clip is using the correct precision or else deepspeed will error
            cast_type_map = {
                "fp16": torch.half,
                "bf16": torch.bfloat16,
                "no": torch.float
            }
            precision_type = cast_type_map[accelerator.mixed_precision]
            assert precision_type == torch.float, "DeepSpeed currently only supports float32 precision when using on the fly embedding generation from clip"
            clip = decoder.clip
            clip.to(precision_type)

        decoder, *optimizers = list(self.accelerator.prepare(decoder, *optimizers))

        self.decoder = decoder

        # prepare dataloaders

        train_loader = val_loader = None
        if exists(dataloaders):
            train_loader, val_loader = self.accelerator.prepare(dataloaders["train"], dataloaders["val"])

        self.train_loader = train_loader
        self.val_loader   = val_loader

        # store optimizers

        for opt_ind, optimizer in zip(range(len(optimizers)), optimizers):
            setattr(self, f'optim{opt_ind}', optimizer)

        # store schedulers

        for sched_ind, scheduler in zip(range(len(schedulers)), schedulers):
            setattr(self, f'sched{sched_ind}', scheduler)

        # store warmup schedulers

        self.warmup_schedulers = warmup_schedulers

    def validate_and_return_unet_number(self, unet_number = None):
        if self.num_unets == 1:
            unet_number = default(unet_number, 1)

        assert exists(unet_number) and 1 <= unet_number <= self.num_unets
        return unet_number

    def num_steps_taken(self, unet_number = None):
        unet_number = self.validate_and_return_unet_number(unet_number)
        return self.steps[unet_number - 1].item()

    def save(self, path, overwrite = True, **kwargs):
        path = Path(path)
        assert not (path.exists() and not overwrite)
        path.parent.mkdir(parents = True, exist_ok = True)

        save_obj = dict(
            model   = self.accelerator.unwrap_model(self.decoder).state_dict(),
            version = __version__,
            steps   = self.steps.cpu(),
            **kwargs
        )

        for ind in range(0, self.num_unets):
            optimizer_key = f'optim{ind}'
            scheduler_key = f'sched{ind}'

            optimizer = getattr(self, optimizer_key)
            scheduler = getattr(self, scheduler_key)

            optimizer_state_dict = optimizer.state_dict() if exists(optimizer) else None
            scheduler_state_dict = scheduler.state_dict() if exists(scheduler) else None

            save_obj = {**save_obj, optimizer_key: optimizer_state_dict, scheduler_key: scheduler_state_dict}

        if self.use_ema:
            save_obj = {**save_obj, 'ema': self.ema_unets.state_dict()}

        self.accelerator.save(save_obj, str(path))

    def load_state_dict(self, loaded_obj, only_model = False, strict = True):
        if version.parse(__version__) != version.parse(loaded_obj['version']):
            self.accelerator.print(f'loading saved decoder at version {loaded_obj["version"]}, but current package version is {__version__}')

        self.accelerator.unwrap_model(self.decoder).load_state_dict(loaded_obj['model'], strict = strict)
        self.steps.copy_(loaded_obj['steps'])

        if only_model:
            return loaded_obj

        for ind, last_step in zip(range(0, self.num_unets), self.steps.tolist()):

            optimizer_key = f'optim{ind}'
            optimizer     = getattr(self, optimizer_key)

            scheduler_key = f'sched{ind}'
            scheduler     = getattr(self, scheduler_key)

            warmup_scheduler = self.warmup_schedulers[ind]

            if exists(optimizer):
                optimizer.load_state_dict(loaded_obj[optimizer_key])

            if exists(scheduler):
                scheduler.load_state_dict(loaded_obj[scheduler_key])

            if exists(warmup_scheduler):
                warmup_scheduler.last_step = last_step

        if self.use_ema:
            assert 'ema' in loaded_obj
            self.ema_unets.load_state_dict(loaded_obj['ema'], strict = strict)

    def load(self, path, only_model = False, strict = True):
        path = Path(path)
        assert path.exists()

        loaded_obj = torch.load(str(path), map_location = 'cpu')

        self.load_state_dict(loaded_obj, only_model = only_model, strict = strict)

        return loaded_obj

    @property
    def unets(self):
        return nn.ModuleList([ema.ema_model for ema in self.ema_unets])

    def increment_step(self, unet_number):
        assert 1 <= unet_number <= self.num_unets

        unet_index_tensor = torch.tensor(unet_number - 1, device = self.steps.device)
        self.steps += F.one_hot(unet_index_tensor, num_classes = len(self.steps))

    def update(self, unet_number = None):
        unet_number = self.validate_and_return_unet_number(unet_number)
        index = unet_number - 1

        optimizer = getattr(self, f'optim{index}')
        scheduler = getattr(self, f'sched{index}')

        if exists(self.max_grad_norm):
            self.accelerator.clip_grad_norm_(self.decoder.parameters(), self.max_grad_norm)  # Automatically unscales gradients

        optimizer.step()
        optimizer.zero_grad()

        warmup_scheduler = self.warmup_schedulers[index]
        scheduler_context = warmup_scheduler.dampening if exists(warmup_scheduler) else nullcontext

        with scheduler_context():
            scheduler.step()

        if self.use_ema:
            ema_unet = self.ema_unets[index]
            ema_unet.update()

        self.increment_step(unet_number)

    @torch.no_grad()
    @cast_torch_tensor
    @decoder_sample_in_chunks
    def sample(self, *args, **kwargs):
        distributed  = self.accelerator.num_processes > 1
        base_decoder = self.accelerator.unwrap_model(self.decoder)

        base_decoder = base_decoder.to(device)
        was_training = base_decoder.training
        base_decoder.eval()

        if kwargs.pop('use_non_ema', False) or not self.use_ema:
            out = base_decoder.sample(*args, **kwargs, distributed = distributed)
            base_decoder.train(was_training)
            return out

        trainable_unets    = self.accelerator.unwrap_model(self.decoder).unets
        base_decoder.unets = self.unets.to(device) # swap in exponential moving averaged unets for sampling

        output             = base_decoder.sample(*args, **kwargs, distributed = distributed)

        base_decoder.unets = trainable_unets             # restore original training unets

        # cast the ema_model unets back to original device
        for ema in self.ema_unets:
            ema.restore_ema_model_device()

        base_decoder.train(was_training)
        return output

    @torch.no_grad()
    @cast_torch_tensor
    @prior_sample_in_chunks
    def embed_text(self, *args, **kwargs):
        return self.accelerator.unwrap_model(self.decoder).clip.embed_text(*args, **kwargs)

    @torch.no_grad()
    @cast_torch_tensor
    @prior_sample_in_chunks
    def embed_image(self, *args, **kwargs):
        return self.accelerator.unwrap_model(self.decoder).clip.embed_image(*args, **kwargs)

    @cast_torch_tensor
    def forward(
            self,
            *args,
            unet_number              = None,
            max_batch_size           = None,
            return_lowres_cond_image = False,
            **kwargs
        ):
        unet_number = self.validate_and_return_unet_number(unet_number)

        total_loss  = 0.
        cond_images = []

        for chunk_size_frac, (chunked_args, chunked_kwargs) in \
                split_args_and_kwargs(*args, split_size = max_batch_size, **kwargs):
            with self.accelerator.autocast():
                loss_obj = self.decoder(
                    *chunked_args,
                    unet_number              = unet_number,
                    return_lowres_cond_image = return_lowres_cond_image,
                    **chunked_kwargs
                )
                # loss_obj may be a tuple with loss and cond_image
                if return_lowres_cond_image:
                    loss, cond_image = loss_obj
                else:
                    loss = loss_obj
                    cond_image = None
                loss = loss * chunk_size_frac
                if cond_image is not None:
                    cond_images.append(cond_image)

            total_loss += loss.item()

            if self.training:
                self.accelerator.backward(loss)

        if return_lowres_cond_image:
            return total_loss, torch.stack(cond_images)
        else:
            return total_loss

### VQGan-VAE trainer class

In [ ]:
class VQGanVAETrainer(nn.Module):
    def __init__(
        self,
        vae,
        *,
        num_train_steps,
        lr,
        batch_size,
        folder,
        grad_accum_every,
        wd                 = 0.,
        save_results_every = 100,
        save_model_every   = 1000,
        results_folder     = './results',
        valid_frac         = 0.05,
        random_split_seed  = 42,
        ema_beta           = 0.995,
        ema_update_after_step    = 500,
        ema_update_every         = 10,
        apply_grad_penalty_every = 4,
        amp                      = False
    ):
        super().__init__()
        assert isinstance(vae, VQGanVAE), 'vae must be instance of VQGanVAE'
        image_size = vae.image_size

        self.vae     = vae
        self.ema_vae = EMA(vae, update_after_step = ema_update_after_step, update_every = ema_update_every)

        self.register_buffer('steps', torch.Tensor([0]))

        self.num_train_steps  = num_train_steps
        self.batch_size       = batch_size
        self.grad_accum_every = grad_accum_every

        all_parameters   = set(vae.parameters())
        discr_parameters = set(vae.discr.parameters())
        vae_parameters   = all_parameters - discr_parameters

        self.optim       = get_optimizer(vae_parameters, lr = lr, wd = wd)
        self.discr_optim = get_optimizer(discr_parameters, lr = lr, wd = wd)

        self.amp          = amp
        self.scaler       = GradScaler(enabled = amp)
        self.discr_scaler = GradScaler(enabled = amp)

        # create dataset

        self.ds = ImageDataset(folder, image_size = image_size)

        # split for validation

        if valid_frac > 0:
            train_size = int((1 - valid_frac) * len(self.ds))
            valid_size = len(self.ds) - train_size
            self.ds, self.valid_ds = random_split(self.ds, [train_size, valid_size], generator = torch.Generator().manual_seed(random_split_seed))
            print(f'training with dataset of {len(self.ds)} samples and validating with randomly splitted {len(self.valid_ds)} samples')
        else:
            self.valid_ds = self.ds
            print(f'training with shared training and valid dataset of {len(self.ds)} samples')

        # dataloader

        self.dl = cycle(DataLoader(
            self.ds,
            batch_size = batch_size,
            shuffle = True
        ))

        self.valid_dl = cycle(DataLoader(
            self.valid_ds,
            batch_size = batch_size,
            shuffle    = True
        ))

        self.save_model_every   = save_model_every
        self.save_results_every = save_results_every

        self.apply_grad_penalty_every = apply_grad_penalty_every

        self.results_folder = Path(results_folder)

        if len([*self.results_folder.glob('**/*')]) > 0 and yes_or_no('do you want to clear previous experiment checkpoints and results?'):
            rmtree(str(self.results_folder))

        self.results_folder.mkdir(parents = True, exist_ok = True)

    def train_step(self):
        device = next(self.vae.parameters()).device
        steps  = int(self.steps.item())
        apply_grad_penalty = not (steps % self.apply_grad_penalty_every)

        self.vae.train()

        # logs

        logs = {}

        # update vae (generator)

        for _ in range(self.grad_accum_every):
            img = next(self.dl)
            img = img.to(device)

            with autocast(enabled = self.amp):
                loss = self.vae(
                    img,
                    return_loss = True,
                    apply_grad_penalty = apply_grad_penalty
                )


                self.scaler.scale(loss / self.grad_accum_every).backward()

            accum_log(logs, {'loss': loss.item() / self.grad_accum_every})

        self.scaler.step(self.optim)
        self.scaler.update()
        self.optim.zero_grad()

        # update discriminator

        if exists(self.vae.discr):
            discr_loss = 0
            for _ in range(self.grad_accum_every):
                img = next(self.dl)
                img = img.to(device)

                with autocast(enabled = self.amp):
                    loss = self.vae(img, return_discr_loss = True)

                    self.discr_scaler.scale(loss / self.grad_accum_every).backward()

                accum_log(logs, {'discr_loss': loss.item() / self.grad_accum_every})

            self.discr_scaler.step(self.discr_optim)
            self.discr_scaler.update()
            self.discr_optim.zero_grad()

            # log

            print(f"{steps}: vae loss: {logs['loss']} - discr loss: {logs['discr_loss']}")

        # update exponential moving averaged generator

        self.ema_vae.update()

        # sample results every so often

        if not (steps % self.save_results_every):
            for model, filename in ((self.ema_vae.ema_model, f'{steps}.ema'), (self.vae, str(steps))):
                model.eval()

                imgs = next(self.dl)
                imgs = imgs.to(device)

                recons = model(imgs)
                nrows  = int(sqrt(self.batch_size))

                imgs_and_recons = torch.stack((imgs, recons), dim = 0)
                imgs_and_recons = rearrange(imgs_and_recons, 'r b ... -> (b r) ...')

                imgs_and_recons = imgs_and_recons.detach().cpu().float().clamp(0., 1.)
                grid = make_grid(imgs_and_recons, nrow = 2, normalize = True, value_range = (0, 1))

                logs['reconstructions'] = grid

                save_image(grid, str(self.results_folder / f'{filename}.png'))

            print(f'{steps}: saving to {str(self.results_folder)}')

        # save model every so often

        if not (steps % self.save_model_every):
            state_dict = self.vae.state_dict()
            model_path = str(self.results_folder / f'vae.{steps}.pt')
            torch.save(state_dict, model_path)

            ema_state_dict = self.ema_vae.state_dict()
            model_path = str(self.results_folder / f'vae.{steps}.ema.pt')
            torch.save(ema_state_dict, model_path)

            print(f'{steps}: saving model to {str(self.results_folder)}')

        self.steps += 1
        return logs

    def train(self, log_fn = noop):
        device = next(self.vae.parameters()).device

        while self.steps < self.num_train_steps:
            logs = self.train_step()
            log_fn(logs)

        print('training complete')

## Trackers

### Helper functions

In [ ]:
# helper functions

def exists(val):
    return val is not None


# Class to measure a time interval

class Timer:
    def __init__(self):
        self.reset()

    def reset(self):
        self.last_time = time.time()

    def elapsed(self):
        return time.time() - self.last_time

# print helpers

def print_ribbon(s, symbol = '=', repeat = 40):
    flank = symbol * repeat
    return f'{flank} {s} {flank}'

# import helpers

def import_or_print_error(pkg_name, err_str = None):
    try:
        return importlib.import_module(pkg_name)
    except ModuleNotFoundError as e:
        if exists(err_str):
            print(err_str)
        exit()

### Loggers

In [ ]:
class BaseLogger:
    """
    An abstract class representing an object that can log data.
    Parameters:
        data_path (str): A file path for storing temporary data.
        verbose (bool): Whether of not to always print logs to the console.
    """
    def __init__(self, data_path: str, resume: bool = False, auto_resume: bool = False, verbose: bool = False, **kwargs):
        self.data_path   = Path(data_path)
        self.resume      = resume
        self.auto_resume = auto_resume
        self.verbose     = verbose

    def init(self, full_config: BaseModel, extra_config: dict, **kwargs) -> None:
        """
        Initializes the logger.
        Errors if the logger is invalid.
        full_config is the config file dict while extra_config is anything else from the script that is not defined the config file.
        """
        raise NotImplementedError

    def log(self, log, **kwargs) -> None:
        raise NotImplementedError

    def log_images(self, images, captions=[], image_section="images", **kwargs) -> None:
        raise NotImplementedError

    def log_file(self, file_path, **kwargs) -> None:
        raise NotImplementedError

    def log_error(self, error_string, **kwargs) -> None:
        raise NotImplementedError

    def get_resume_data(self, **kwargs) -> dict:
        """
        Sets tracker attributes that along with { "resume": True } will be used to resume training.
        It is assumed that after init is called this data will be complete.
        If the logger does not have any resume functionality, it should return an empty dict.
        """
        raise NotImplementedError

In [ ]:
class ConsoleLogger(BaseLogger):
    def init(self, full_config: BaseModel, extra_config: dict, **kwargs) -> None:
        print("Logging to console")

    def log(self, log, **kwargs) -> None:
        print(log)

    def log_images(self, images, captions=[], image_section="images", **kwargs) -> None:
        pass

    def log_file(self, file_path, **kwargs) -> None:
        pass

    def log_error(self, error_string, **kwargs) -> None:
        print(error_string)

    def get_resume_data(self, **kwargs) -> dict:
        return {}

In [ ]:
class WandbLogger(BaseLogger):
    """
    Logs to a wandb run.
    Parameters:
        data_path (str):      A file path for storing temporary data.
        wandb_entity (str):   The wandb entity to log to.
        wandb_project (str):  The wandb project to log to.
        wandb_run_id (str):   The wandb run id to resume.
        wandb_run_name (str): The wandb run name to use.
    """
    def __init__(self,
        data_path:      str,
        wandb_entity:   str,
        wandb_project:  str,
        wandb_run_id:   Optional[str] = None,
        wandb_run_name: Optional[str] = None,
        **kwargs
    ):
        super().__init__(data_path, **kwargs)
        self.entity   = wandb_entity
        self.project  = wandb_project
        self.run_id   = wandb_run_id
        self.run_name = wandb_run_name

    def init(self, full_config: BaseModel, extra_config: dict, **kwargs) -> None:
        assert self.entity is not None, "wandb_entity must be specified for wandb logger"
        assert self.project is not None, "wandb_project must be specified for wandb logger"
        self.wandb = import_or_print_error('wandb', '`pip install wandb` to use the wandb logger')
        os.environ["WANDB_SILENT"] = "true"
        # Initializes the wandb run
        init_object = {
            "entity": self.entity,
            "project": self.project,
            "config": {**full_config.dict(), **extra_config}
        }
        if self.run_name is not None:
            init_object['name'] = self.run_name
        if self.resume:
            assert self.run_id is not None, '`wandb_run_id` must be provided if `wandb_resume` is True'
            if self.run_name is not None:
                print("You are renaming a run. I hope that is what you intended.")
            init_object['resume'] = 'must'
            init_object['id'] = self.run_id

        self.wandb.init(**init_object)
        print(f"Logging to wandb run {self.wandb.run.path}-{self.wandb.run.name}")

    def log(self, log, **kwargs) -> None:
        if self.verbose:
            print(log)
        self.wandb.log(log, **kwargs)

    def log_images(self, images, captions=[], image_section="images", **kwargs) -> None:
        """
        Takes a tensor of images and a list of captions and logs them to wandb.
        """
        wandb_images = [self.wandb.Image(image, caption=caption) for image, caption in zip_longest(images, captions)]
        self.wandb.log({ image_section: wandb_images }, **kwargs)

    def log_file(self, file_path, base_path: Optional[str] = None, **kwargs) -> None:
        if base_path is None:
            # Then we take the basepath as the parent of the file_path
            base_path = Path(file_path).parent
        self.wandb.save(str(file_path), base_path = str(base_path))

    def log_error(self, error_string, step=None, **kwargs) -> None:
        if self.verbose:
            print(error_string)
        self.wandb.log({"error": error_string, **kwargs}, step=step)

    def get_resume_data(self, **kwargs) -> dict:
        # In order to resume, we need wandb_entity, wandb_project, and wandb_run_id
        return {
            "entity":  self.entity,
            "project": self.project,
            "run_id":  self.wandb.run.id
        }

In [ ]:
def create_logger(logger_type: str, data_path: str, **kwargs) -> BaseLogger:
    if logger_type == 'custom':
        raise NotImplementedError('Custom loggers are not supported yet. Please use a different logger type.')
    try:
        logger_class = logger_type_map[logger_type]
    except KeyError:
        raise ValueError(f'Unknown logger type: {logger_type}. Must be one of {list(logger_type_map.keys())}')
    return logger_class(data_path, **kwargs)

logger_type_map = {
    'console': ConsoleLogger,
    'wandb':   WandbLogger,
}

### Loaders

In [ ]:
class BaseLoader:
    """
    An abstract class representing an object that can load a model checkpoint.
    Parameters:
        data_path (str): A file path for storing temporary data.
    """
    def __init__(self, data_path: str, only_auto_resume: bool = False, **kwargs):
        self.data_path        = Path(data_path)
        self.only_auto_resume = only_auto_resume

    def init(self, logger: BaseLogger, **kwargs) -> None:
        raise NotImplementedError

    def recall() -> dict:
        raise NotImplementedError

class UrlLoader(BaseLoader):
    """
    A loader that downloads the file from a url and loads it.

    Parameters:
        data_path (str): A file path for storing temporary data.
        url (str): The url to download the file from.
    """
    def __init__(self, data_path: str, url: str, **kwargs):
        super().__init__(data_path, **kwargs)
        self.url = url

    def init(self, logger: BaseLogger, **kwargs) -> None:
        # Makes sure the file exists to be downloaded
        pass  # TODO: Actually implement that

    def recall(self) -> dict:
        # Download the file
        save_path = self.data_path / 'loaded_checkpoint.pth'
        urllib.request.urlretrieve(self.url, str(save_path))
        # Load the file
        return torch.load(str(save_path), map_location='cpu')


class LocalLoader(BaseLoader):
    """
    A loader that loads a file from a local path.

    Parameters:
        data_path (str): A file path for storing temporary data.
        file_path (str): The path to the file to load.
    """
    def __init__(self, data_path: str, file_path: str, **kwargs):
        super().__init__(data_path, **kwargs)
        self.file_path = Path(file_path)

    def init(self, logger: BaseLogger, **kwargs) -> None:
        # Makes sure the file exists to be loaded
        if not self.file_path.exists() and not self.only_auto_resume:
            raise FileNotFoundError(f'Model not found at {self.file_path}')

    def recall(self) -> dict:
        # Load the file
        return torch.load(str(self.file_path), map_location='cpu')


class WandbLoader(BaseLoader):
    """
    A loader that loads a model from an existing wandb run.
    """
    def __init__(self, data_path: str, wandb_file_path: str, wandb_run_path: Optional[str] = None, **kwargs):
        super().__init__(data_path, **kwargs)
        self.run_path  = wandb_run_path
        self.file_path = wandb_file_path

    def init(self, logger: BaseLogger, **kwargs) -> None:
        self.wandb = import_or_print_error('wandb', '`pip install wandb` to use the wandb recall function')
        # Make sure the file can be downloaded
        if self.wandb.run is not None and self.run_path is None:
            self.run_path = self.wandb.run.path
            assert self.run_path is not None, 'wandb run was not found to load from. If not using the wandb logger must specify the `wandb_run_path`.'
        assert self.run_path is not None, '`wandb_run_path` must be provided for the wandb loader'
        assert self.file_path is not None, '`wandb_file_path` must be provided for the wandb loader'

        os.environ["WANDB_SILENT"] = "true"
        pass  # TODO: Actually implement that

    def recall(self) -> dict:
        file_reference = self.wandb.restore(self.file_path, run_path=self.run_path)
        return torch.load(file_reference.name, map_location='cpu')


In [ ]:
loader_type_map = {
    'url':   UrlLoader,
    'local': LocalLoader,
    'wandb': WandbLoader,
}

### Savers

In [ ]:
class BaseSaver:
    def __init__(self,
            data_path:      str,
            save_latest_to: Optional[Union[str, bool]] = None,
            save_best_to:   Optional[Union[str, bool]] = None,
            save_meta_to:   Optional[str] = None,
            save_type:      str = 'checkpoint',
            **kwargs
        ):
        self.data_path      = Path(data_path)
        self.save_latest_to = save_latest_to
        self.saving_latest  = save_latest_to is not None and save_latest_to is not False
        self.save_best_to   = save_best_to
        self.saving_best    = save_best_to is not None and save_best_to is not False
        self.save_meta_to   = save_meta_to
        self.saving_meta    = save_meta_to is not None
        self.save_type      = save_type
        assert save_type in ['checkpoint', 'model'], '`save_type` must be one of `checkpoint` or `model`'
        assert self.saving_latest or self.saving_best or self.saving_meta, 'At least one saving option must be specified'

    def init(self, logger: BaseLogger, **kwargs) -> None:
        raise NotImplementedError

    def save_file(self, local_path: Path, save_path: str, is_best=False, is_latest=False, **kwargs) -> None:
        """
        Save a general file under save_meta_to
        """
        raise NotImplementedError

class LocalSaver(BaseSaver):
    def __init__(self,
        data_path: str,
        **kwargs
    ):
        super().__init__(data_path, **kwargs)

    def init(self, logger: BaseLogger, **kwargs) -> None:
        # Makes sure the directory exists to be saved to
        print(f"[INFO] Saving {self.save_type} locally")
        if not self.data_path.exists():
            self.data_path.mkdir(parents=True)

    def save_file(self, local_path: str, save_path: str, **kwargs) -> None:
        # Copy the file to save_path
        save_path_file_name = Path(save_path).name
        # Make sure parent directory exists
        save_path_parent = Path(save_path).parent
        if not save_path_parent.exists():
            save_path_parent.mkdir(parents=True)
        print(f"[INFO] Saving {save_path_file_name} {self.save_type} to local path {save_path}")
        shutil.copy(local_path, save_path)

class WandbSaver(BaseSaver):
    def __init__(self, data_path: str, wandb_run_path: Optional[str] = None, **kwargs):
        super().__init__(data_path, **kwargs)
        self.run_path = wandb_run_path

    def init(self, logger: BaseLogger, **kwargs) -> None:
        self.wandb = import_or_print_error('wandb', '`pip install wandb` to use the wandb logger')
        os.environ["WANDB_SILENT"] = "true"
        # Makes sure that the user can upload tot his run
        if self.run_path is not None:
            entity, project, run_id = self.run_path.split("/")
            self.run = self.wandb.init(entity=entity, project=project, id=run_id)
        else:
            assert self.wandb.run is not None, 'You must be using the wandb logger if you are saving to wandb and have not set `wandb_run_path`'
            self.run = self.wandb.run
        # TODO: Now actually check if upload is possible
        print(f"[INFO] Saving to wandb run {self.run.path}-{self.run.name}")

    def save_file(self, local_path: Path, save_path: str, **kwargs) -> None:
        # In order to log something in the correct place in wandb, we need to have the same file structure here
        save_path_file_name = Path(save_path).name
        print(f"[INFO] Saving {save_path_file_name} {self.save_type} to wandb run {self.run.path}-{self.run.name}")
        save_path = Path(self.data_path) / save_path
        save_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_path, save_path)
        self.run.save(str(save_path), base_path = str(self.data_path), policy='now')

class HuggingfaceSaver(BaseSaver):
    def __init__(self, data_path: str, huggingface_repo: str, token_path: Optional[str] = None, **kwargs):
        super().__init__(data_path, **kwargs)
        self.huggingface_repo = huggingface_repo
        self.token_path = token_path

    def init(self, logger: BaseLogger, **kwargs):
        # Makes sure this user can upload to the repo
        self.hub = import_or_print_error('huggingface_hub', '`pip install huggingface_hub` to use the huggingface saver')
        try:
            identity = self.hub.whoami()  # Errors if not logged in
            # Then we are logged in
        except:
            # We are not logged in. Use the token_path to set the token.
            if not os.path.exists(self.token_path):
                raise Exception("Not logged in to huggingface and no token_path specified. Please login with `huggingface-cli login` or if that does not work set the token_path.")
            with open(self.token_path, "r") as f:
                token = f.read().strip()
            self.hub.HfApi.set_access_token(token)
            identity = self.hub.whoami()
        print(f"[INFO] Saving to huggingface repo {self.huggingface_repo}")

    def save_file(self, local_path: Path, save_path: str, **kwargs) -> None:
        # Saving to huggingface is easy, we just need to upload the file with the correct name
        save_path_file_name = Path(save_path).name
        print(f"[INFO] Saving {save_path_file_name} {self.save_type} to huggingface repo {self.huggingface_repo}")
        self.hub.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=str(save_path),
            repo_id=self.huggingface_repo
        )

In [ ]:
saver_type_map = {
    'local':       LocalSaver,
    'wandb':       WandbSaver,
    'huggingface': HuggingfaceSaver
}

### Tracker

In [ ]:
class Tracker:
    def __init__(
            self,
            data_path:           Optional[str] = DEFAULT_DATA_PATH,
            overwrite_data_path: bool          = False,
            dummy_mode:          bool          = False
        ):
        self.data_path = Path(data_path)
        if not dummy_mode:
            if not overwrite_data_path:
                assert not self.data_path.exists(), f'Data path {self.data_path} already exists. Set overwrite_data_path to True to overwrite.'
                if not self.data_path.exists():
                    self.data_path.mkdir(parents=True)
        self.logger:    BaseLogger           = None
        self.loader:    Optional[BaseLoader] = None
        self.savers:    List[BaseSaver]      = []
        self.dummy_mode                      = dummy_mode

    def _load_auto_resume(self) -> bool:
        # If the file does not exist, we return False. If autoresume is enabled we print 
        # a warning so that the user can know that this is the first run.
        if not self.auto_resume_path.exists():
            if self.logger.auto_resume:
                print("[WARN] Auto resume is enabled but no auto_resume.json file exists. Assuming this is the first run.")
            else:
                print("[WARN] The auto_resume.json file does not exist.")
            return False

        # Now we know that the autoresume file exists, but if we are not auto resuming we should remove it so that we don't accidentally load it next time
        if not self.logger.auto_resume:
            print(f'[WARN] Removing auto_resume.json because auto resume is not enabled in the config')
            self.auto_resume_path.unlink()
            return False

        # Otherwise we read the json into a dictionary will will override parts of logger.__dict__
        with open(self.auto_resume_path, 'r') as f:
            auto_resume_dict = json.load(f)
        # Check if the logger is of the same type as the autoresume save
        if auto_resume_dict["logger_type"] != self.logger.__class__.__name__:
            raise Exception(f'The logger type in the auto_resume file is {auto_resume_dict["logger_type"]} but the current logger is {self.logger.__class__.__name__}. Either use the original logger type, set `auto_resume` to `False`, or delete your existing tracker-data folder.')
        # Then we are ready to override the logger with the autoresume save
        self.logger.__dict__["resume"] = True
        print(f"[INFO] Updating {self.logger.__dict__} with auto_resume.json content {auto_resume_dict}")
        self.logger.__dict__.update(auto_resume_dict)
        return True

    def _save_auto_resume(self):
        # Gets the autoresume dict from the logger and adds "logger_type" to it then saves it to the auto_resume file
        auto_resume_dict = self.logger.get_resume_data()
        auto_resume_dict['logger_type'] = self.logger.__class__.__name__
        with open(self.auto_resume_path, 'w') as f:
            json.dump(auto_resume_dict, f)

    def init(self, full_config: BaseModel, extra_config: dict):
        self.auto_resume_path = self.data_path / 'auto_resume.json'
        # Check for resuming the run
        self.did_auto_resume = self._load_auto_resume()
        if self.did_auto_resume:
            print(f'\n\n[WARN] RUN HAS BEEN AUTO-RESUMED WITH THE LOGGER TYPE {self.logger.__class__.__name__}.\nIf this was not your intention, stop this run and set `auto_resume` to `False` in the config.\n\n')
            print(f"New logger config: {self.logger.__dict__}")

        self.save_metadata = dict(
            version = version.parse(__version__)
        )  # Data that will be saved alongside the checkpoint or model
        self.blacklisted_checkpoint_metadata_keys = ['scaler', 'optimizer', 'model', 'version', 'step', 'steps']  # These keys would cause us to error if we try to save them as metadata

        assert self.logger is not None, '`logger` must be set before `init` is called'
        if self.dummy_mode:
            # The only thing we need is a loader
            if self.loader is not None:
                self.loader.init(self.logger)
            return
        assert len(self.savers) > 0, '`savers` must be set before `init` is called'

        self.logger.init(full_config, extra_config)
        if self.loader is not None:
            self.loader.init(self.logger)
        for saver in self.savers:
            saver.init(self.logger)

        if self.logger.auto_resume:
            # Then we need to save the autoresume file. It is assumed after logger.init is called that the logger is ready to be saved.
            self._save_auto_resume()

    def add_logger(self, logger: BaseLogger):
        self.logger = logger

    def add_loader(self, loader: BaseLoader):
        self.loader = loader

    def add_saver(self, saver: BaseSaver):
        self.savers.append(saver)

    def log(self, *args, **kwargs):
        if self.dummy_mode:
            return
        self.logger.log(*args, **kwargs)

    def log_images(self, *args, **kwargs):
        if self.dummy_mode:
            return
        self.logger.log_images(*args, **kwargs)

    def log_file(self, *args, **kwargs):
        if self.dummy_mode:
            return
        self.logger.log_file(*args, **kwargs)

    def save_config(self, current_config_path: str, config_name = 'config.json'):
        if self.dummy_mode:
            return
        # Save the config under config_name in the root folder of data_path
        shutil.copy(current_config_path, self.data_path / config_name)
        for saver in self.savers:
            if saver.saving_meta:
                remote_path = Path(saver.save_meta_to) / config_name
                saver.save_file(current_config_path, str(remote_path))

    def add_save_metadata(self, state_dict_key: str, metadata: Any):
        """
        Adds a new piece of metadata that will be saved along with the prior or decoder model.
        """
        self.save_metadata[state_dict_key] = metadata

    def _save_state_dict(self, trainer: Union[DiffusionPriorTrainer, DecoderTrainer], save_type: str, file_path: str, **kwargs) -> Path:
        """
        Gets the state dict to be saved and writes it to file_path.
        If save_type is 'checkpoint', we save the entire trainer state dict.
        If save_type is 'model', we save only the model state dict.
        """
        assert save_type in ['checkpoint', 'model']
        if save_type == 'checkpoint':
            # Create a metadata dict without the blacklisted keys so we do not error when we create the state dict
            metadata = {k: v for k, v in self.save_metadata.items() if k not in self.blacklisted_checkpoint_metadata_keys}
            trainer.save(file_path, overwrite=True, **kwargs, **metadata)
        elif save_type == 'model':
            if isinstance(trainer, DiffusionPriorTrainer):
                prior = trainer.ema_diffusion_prior.ema_model if trainer.use_ema else trainer.diffusion_prior
                prior: DiffusionPrior = trainer.accelerator.unwrap_model(prior)
                # Remove CLIP if it is part of the model
                original_clip = prior.clip
                prior.clip = None
                model_state_dict = prior.state_dict()
                prior.clip = original_clip
            elif isinstance(trainer, DecoderTrainer):
                decoder: Decoder = trainer.accelerator.unwrap_model(trainer.decoder)
                # Remove CLIP if it is part of the model
                original_clip = decoder.clip
                decoder.clip = None
                if trainer.use_ema:
                    trainable_unets = decoder.unets
                    decoder.unets = trainer.unets  # Swap EMA unets in
                    model_state_dict = decoder.state_dict()
                    decoder.unets = trainable_unets  # Swap back
                else:
                    model_state_dict = decoder.state_dict()
                decoder.clip = original_clip
            else:
                raise NotImplementedError('Saving this type of model with EMA mode enabled is not yet implemented. Actually, how did you get here?')
            state_dict = {
                **self.save_metadata,
                'model': model_state_dict
            }
            torch.save(state_dict, file_path)
        return Path(file_path)

    def save(self, trainer, is_best: bool, is_latest: bool, **kwargs):
        if self.dummy_mode:
            return
        if not is_best and not is_latest:
            # Nothing to do
            return
        # Save the checkpoint and model to data_path
        checkpoint_path = self.data_path / 'checkpoint.pth'
        self._save_state_dict(trainer, 'checkpoint', checkpoint_path, **kwargs)
        model_path = self.data_path / 'model.pth'
        self._save_state_dict(trainer, 'model', model_path, **kwargs)
        print("[INFO] Saved cached models")
        # Call the save methods on the savers
        for saver in self.savers:
            local_path = checkpoint_path if saver.save_type == 'checkpoint' else model_path
            if saver.saving_latest and is_latest:
                latest_checkpoint_path = saver.save_latest_to.format(**kwargs)
                try:
                    saver.save_file(local_path, latest_checkpoint_path, is_latest=True, **kwargs)
                except Exception as e:
                    self.logger.log_error(f'Error saving checkpoint {e}', **kwargs)
                    print(f'Error saving checkpoint {e}')
            if saver.saving_best and is_best:
                best_checkpoint_path = saver.save_best_to.format(**kwargs)
                try:
                    saver.save_file(local_path, best_checkpoint_path, is_best=True, **kwargs)
                except Exception as e:
                    self.logger.log_error(f'Error saving checkpoint {e}', **kwargs)
                    print(f'Error saving checkpoint {e}')

    @property
    def can_recall(self):
        # Defines whether a recall can be performed.
        return self.loader is not None and (not self.loader.only_auto_resume or self.did_auto_resume)

    def recall(self):
        if self.can_recall:
            return self.loader.recall()
        else:
            raise ValueError('Tried to recall, but no loader was set or auto-resume was not performed.')


## Train configurations

In [ ]:
class TrainSplitConfig(BaseModel):
    train: float = 0.75
    val:   float = 0.15
    test:  float = 0.1

    @model_validator(mode = 'after')
    def validate_all(self, m):
        actual_sum = sum([*dict(self).values()])
        if actual_sum != 1.:
            raise ValueError(f'{dict(self).keys()} must sum to 1.0. Found: {actual_sum}')
        return self

### Tracker configuration

In [ ]:
def create_saver(saver_type: str, data_path: str, **kwargs) -> BaseSaver:
    if saver_type == 'custom':
        raise NotImplementedError('Custom savers are not supported yet. Please use a different saver type.')
    try:
        saver_class = saver_type_map[saver_type]
    except KeyError:
        raise ValueError(f'Unknown saver type: {saver_type}. Must be one of {list(saver_type_map.keys())}')
    return saver_class(data_path, **kwargs)


def create_loader(loader_type: str, data_path: str, **kwargs) -> BaseLoader:
    if loader_type == 'custom':
        raise NotImplementedError('Custom loaders are not supported yet. Please use a different loader type.')
    try:
        loader_class = loader_type_map[loader_type]
    except KeyError:
        raise ValueError(f'Unknown loader type: {loader_type}. Must be one of {list(loader_type_map.keys())}')
    return loader_class(data_path, **kwargs)


class TrackerLogConfig(BaseModel):
    log_type:    str  = 'console'
    resume:      bool = False  # For logs that are saved to unique locations, resume a previous run
    auto_resume: bool = False  # If the process crashes and restarts, resume from the run that crashed
    verbose:     bool = False

    class Config:
        # Each individual log type has it's own arguments that will be passed through the config
        extra = "allow"

    def create(self, data_path: str):
        kwargs = self.dict()
        return create_logger(self.log_type, data_path, **kwargs)

class TrackerLoadConfig(BaseModel):
    load_from: Optional[str] = None
    only_auto_resume: bool = False  # Only attempt to load if the logger is auto-resuming

    class Config:
        extra = "allow"

    def create(self, data_path: str):
        kwargs = self.dict()
        if self.load_from is None:
            return None
        return create_loader(self.load_from, data_path, **kwargs)

class TrackerSaveConfig(BaseModel):
    save_to:     str  = 'local'
    save_all:    bool = False
    save_latest: bool = True
    save_best:   bool = True

    class Config:
        extra = "allow"

    def create(self, data_path: str):
        kwargs = self.dict()
        return create_saver(self.save_to, data_path, **kwargs)

class TrackerConfig(BaseModel):
    data_path:           str  = '.tracker_data'
    overwrite_data_path: bool = False
    log:                 TrackerLogConfig
    load:                Optional[TrackerLoadConfig] = None
    save:                Union[List[TrackerSaveConfig], TrackerSaveConfig]

    def create(self, full_config: BaseModel, extra_config: dict, dummy_mode: bool = False) -> Tracker:
        tracker = Tracker(self.data_path, dummy_mode=dummy_mode, overwrite_data_path=self.overwrite_data_path)
        # Add the logger
        tracker.add_logger(self.log.create(self.data_path))
        # Add the loader
        if self.load is not None:
            tracker.add_loader(self.load.create(self.data_path))
        # Add the saver or savers
        if isinstance(self.save, list):
            for save_config in self.save:
                tracker.add_saver(save_config.create(self.data_path))
        else:
            tracker.add_saver(self.save.create(self.data_path))
        # Initialize all the components and verify that all data is valid
        tracker.init(full_config, extra_config)
        return tracker

### Diffusion prior configuration

In [ ]:
class AdapterConfig(BaseModel):
    make:  str = "openai"
    model: str = "ViT-L/14"
    base_model_kwargs: Optional[Dict[str, Any]] = None

    def create(self):
        if self.make == "openai":
            return OpenAIClipAdapter(self.model)
        elif self.make == "open_clip":
            pretrained = dict(list_pretrained())
            checkpoint = pretrained[self.model]
            return OpenClipAdapter(name=self.model, pretrained=checkpoint)
        elif self.make == "x-clip":
            return XClipAdapter(XCLIP(**self.base_model_kwargs))
        elif self.make == "coca":
            return CoCaAdapter(CoCa(**self.base_model_kwargs))
        else:
            raise AttributeError("No adapter with that name is available.")

class DiffusionPriorNetworkConfig(BaseModel):
    dim:              int
    depth:            int
    max_text_len:     Optional[int] = None
    num_timesteps:    Optional[int] = None
    num_time_embeds:  int   = 1
    num_image_embeds: int   = 1
    num_text_embeds:  int   = 1
    dim_head:         int   = 64
    heads:            int   = 8
    ff_mult:          int   = 4
    norm_in:          bool  = False
    norm_out:         bool  = True
    attn_dropout:     float = 0.
    ff_dropout:       float = 0.
    final_proj:       bool  = True
    normformer:       bool  = False
    rotary_emb:       bool  = True

    class Config:
        extra = "allow"

    def create(self):
        kwargs = self.dict()
        return DiffusionPriorNetwork(**kwargs)

class DiffusionPriorConfig(BaseModel):
    clip:             Optional[AdapterConfig] = None
    net:              DiffusionPriorNetworkConfig
    image_embed_dim:  int
    image_size:       int
    image_channels:   int             = 3
    timesteps:        int             = 1000
    sample_timesteps: Optional[int]   = None
    cond_drop_prob:   float           = 0.
    loss_type:        str             = 'l2'
    predict_x_start:  bool            = True
    beta_schedule:    str             = 'cosine'
    condition_on_text_encodings: bool = True

    class Config:
        extra = "allow"

    def create(self):
        kwargs = self.dict()

        has_clip = exists(kwargs.pop('clip'))
        kwargs.pop('net')

        clip = None
        if has_clip:
            clip = self.clip.create()
            print(f'[DEBUG] created a CLIP model')

        diffusion_prior_network   = self.net.create()
        # ########################################################################### DEBUG CODE (begin)
        print(f'[DEBUG] PRIOR:\n{diffusion_prior_network}')
        print(f'[DEBUG] CLIP:\n{clip}')
        # ########################################################################### DEBUG CODE (end)
        return DiffusionPrior(net = diffusion_prior_network, clip = clip, **kwargs)

class DiffusionPriorTrainConfig(BaseModel):
    epochs:               int       = 1
    lr:                   float     = 1.1e-4
    wd:                   float     = 6.02e-2
    max_grad_norm:        float     = 0.5
    use_ema:              bool      = True
    ema_beta:             float     = 0.99
    amp:                  bool      = False
    warmup_steps:         Optional[int] = None # number of warmup steps
    log_every_steps:      int       = 50
    eval_every_steps:     int       = 10000 # how often to run validation (steps)
    save_every_steps:     int       = 10000 # how often to save (steps)
    eval_timesteps:       List[int] = [64]  # which sampling timesteps to evaluate with
    best_validation_loss: float     = 1e9   # the current best valudation loss observed
    current_epoch:        int       = 0     # the current epoch
    num_samples_seen:     int       = 0     # the current number of samples seen
    random_seed:          int       = 0     # manual seed for torch

class DiffusionPriorDataConfig(BaseModel):
    image_url:          str              # path to image embeddings folder
    txt_url:            str              # path to text embeddings folder
    meta_url:           str              # path to metadata (captions) for images
    splits:             TrainSplitConfig # define train, validation, test splits for your dataset
    batch_size:         int              # per-gpu batch size used to train the model
    num_data_points:    int = 1.5e6      # total number of datapoints to train on

class TrainDiffusionPriorConfig(BaseModel):
    prior:   DiffusionPriorConfig
    data:    DiffusionPriorDataConfig
    train:   DiffusionPriorTrainConfig
    tracker: TrackerConfig

    @classmethod
    def from_json_path(cls, json_path):
        with open(json_path) as f:
            config = json.load(f)
            print(f'[INFO] Training prior configuration:')
            printConfig(config, indent=0)
        return cls(**config)

### Decoder configuration

In [ ]:
def printConfig(config, indent=0):
    for key, value in config.items():
        base_str = '\t' * indent + str(key)
        if isinstance(value, dict):
            print(f'{base_str}:')
            printConfig(value, indent+1)
        elif isinstance(value, list):
            for n, val in enumerate(value):
                if isinstance(val, dict):
                    if n==0:
                        print(f'{base_str}:')
                    printConfig(val, indent+1)
                else:
                    if n==0 and n!=(len(value)-1):
                        print(f'{base_str}: [{val}, ', end='')
                    elif n==0 and n==(len(value)-1):
                        print(f'{base_str}: [{val}]')
                    elif n==(len(value)-1):
                        print(f"{val}]")
                    else:
                        print(f"{val}, ", end='')
        else:
            print(f'{base_str}: {value}')

# decoder pydantic classes

class UnetConfig(BaseModel):
    dim:              int
    dim_mults:        ListOrTuple[int]
    image_embed_dim:  Optional[int]        = None
    text_embed_dim:   Optional[int]        = None
    cond_on_text_encodings: Optional[bool] = None
    cond_dim:         Optional[int]        = None
    channels:         int  = 3
    self_attn:        SingularOrIterable[bool] = False
    attn_dim_head:    int  = 32
    attn_heads:       int  = 16
    init_cross_embed: bool = True

    class Config:
        extra = "allow"


class DecoderConfig(BaseModel):
    unets:                ListOrTuple[UnetConfig]
    image_size:           Optional[int]           = None
    image_sizes:          ListOrTuple[int]        = None
    clip:                 Optional[AdapterConfig] = None   # The clip model to use if embeddings are not provided
    channels:             int   = 3
    timesteps:            int   = 1000
    sample_timesteps:     Optional[SingularOrIterable[Optional[int]]] = None
    loss_type:            str   = 'l2'
    beta_schedule:        Optional[ListOrTuple[str]] = None  # None means all cosine
    learned_variance:     SingularOrIterable[bool]   = True
    image_cond_drop_prob: float = 0.1
    text_cond_drop_prob:  float = 0.5

    def create(self):
        decoder_kwargs = self.dict()

        unet_configs   = decoder_kwargs.pop('unets')

        unets          = [Unet(**config) for config in unet_configs]

        has_clip = exists(decoder_kwargs.pop('clip'))
        clip     = None
        if has_clip:
            clip = self.clip.create()

        return Decoder(unets, clip=clip, **decoder_kwargs)

    @validator('image_sizes')
    def check_image_sizes(cls, image_sizes, values):
        if exists(values.get('image_size')) ^ exists(image_sizes):
            return image_sizes
        raise ValueError('either image_size or image_sizes is required, but not both')

    class Config:
        extra = "allow"


class DecoderDataConfig(BaseModel):
    webdataset_base_url: str                     # path to a webdataset with jpg images
    img_embeddings_url:  Optional[str] = None    # path to .npy files with embeddings
    text_embeddings_url: Optional[str] = None    # path to .npy files with embeddings
    num_workers:         int  = 4
    start_shard:         int  = 0
    end_shard:           int  = 99999
    shard_width:         int  = 5
    index_width:         int  = 4
    splits:              TrainSplitConfig
    shuffle_train:       bool = True
    resample_train:      bool = False
    preprocessing:       Dict[str, Any] = {'ToTensor': True}

    @property
    def img_preproc(self):
        def _get_transformation(transformation_name, **kwargs):
            if transformation_name == "RandomResizedCrop":
                return T.RandomResizedCrop(**kwargs)
            elif transformation_name == "RandomHorizontalFlip":
                return T.RandomHorizontalFlip()
            elif transformation_name == "ToTensor":
                return T.ToTensor()

        transforms = []
        for transform_name, transform_kwargs_or_bool in self.preprocessing.items():
            transform_kwargs = {} if not isinstance(transform_kwargs_or_bool, dict) else transform_kwargs_or_bool
            transforms.append(_get_transformation(transform_name, **transform_kwargs))
        return T.Compose(transforms)


class DecoderTrainConfig(BaseModel):
    epochs:                 int = 20
    batch_size:             int = 32
    lr:                     SingularOrIterable[float] = 0.0001
    wd:                     SingularOrIterable[float] = 0.01
    warmup_steps:           Optional[SingularOrIterable[int]] = None
    find_unused_parameters: bool = True
    static_graph:           bool = True
    max_grad_norm:          SingularOrIterable[float] = 0.5
    save_every_n_samples:   int = 100000
    n_sample_images:        int = 6   # The number of example images to produce when sampling the train and test dataset
    cond_scale:             Union[float, List[float]] = 1.0
    epoch_samples:          Optional[int] = None   # Limits the number of samples per epoch. None means no limit. Required if resample_train is true as otherwise the number of samples per epoch is infinite.
    validation_samples:     Optional[int] = None   # Same as above but for validation.
    save_immediately:       bool          = False
    use_ema:                bool          = True
    ema_beta:               float         = 0.999
    amp:                    bool          = False
    unet_training_mask:     Optional[ListOrTuple[bool]] = None   # If None, use all unets


class DecoderEvaluateConfig(BaseModel):
    n_evaluation_samples: int       = 1000
    FID:   Optional[Dict[str, Any]] = None
    IS:    Optional[Dict[str, Any]] = None
    KID:   Optional[Dict[str, Any]] = None
    LPIPS: Optional[Dict[str, Any]] = None


class TrainDecoderConfig(BaseModel):
    decoder:  DecoderConfig
    data:     DecoderDataConfig
    train:    DecoderTrainConfig
    evaluate: DecoderEvaluateConfig
    tracker:  TrackerConfig
    seed:     int = 0

    @classmethod
    def from_json_path(cls, json_path):
        with open(json_path) as f:
            config = json.load(f)
            print(f'[INFO] Prior training configuration:')
            printConfig(config, indent=0)
        return cls(**config)

    @model_validator(mode = 'after')
    def check_has_embeddings(self, m):
        # Makes sure that enough information is provided to 
        # get the embeddings specified for training
        values = dict(self)

        data_config, decoder_config = values.get('data'), values.get('decoder')

        if not exists(data_config) or not exists(decoder_config):
            # Then some error occurred and we should just pass through
            return values

        using_text_embeddings = any([unet.cond_on_text_encodings for unet in decoder_config.unets])
        using_clip   = exists(decoder_config.clip)
        img_emb_url  = data_config.img_embeddings_url
        text_emb_url = data_config.text_embeddings_url

        if using_text_embeddings:
            # Then we need some way to get the embeddings
            assert using_clip or exists(text_emb_url), 'If text conditioning, either clip or text_embeddings_url must be provided'

        if using_clip:
            if using_text_embeddings:
                assert not exists(text_emb_url) or not exists(img_emb_url), 'Loaded clip, but also provided text_embeddings_url and img_embeddings_url. This is redundant. Remove the clip model or the text embeddings'
            else:
                assert not exists(img_emb_url), 'Loaded clip, but also provided img_embeddings_url. This is redundant. Remove the clip model or the embeddings'

        if text_emb_url:
            assert using_text_embeddings, "Text embeddings are being loaded, but text embeddings are not being conditioned on. This will slow down the dataloader for no reason."

        return m

## Trainer Classes

### Helper functions

In [ ]:
def pick_and_pop(keys, d):
    values = list(map(lambda key: d.pop(key), keys))
    return dict(zip(keys, values))


def group_dict_by_key(cond, d):
    return_val = [dict(),dict()]
    for key in d.keys():
        match = bool(cond(key))
        ind   = int(not match)
        return_val[ind][key] = d[key]
    return (*return_val,)


def string_begins_with(prefix, str):
    return str.startswith(prefix)


def group_by_key_prefix(prefix, d):
    return group_dict_by_key(partial(string_begins_with, prefix), d)


def groupby_prefix_and_trim(prefix, d):
    kwargs_with_prefix, kwargs = group_dict_by_key(partial(string_begins_with, prefix), d)
    kwargs_without_prefix      = dict(map(lambda x: (x[0][len(prefix):], x[1]), tuple(kwargs_with_prefix.items())))
    return kwargs_without_prefix, kwargs


def num_to_groups(num, divisor):
    groups    = num // divisor
    remainder = num % divisor
    arr       = [divisor] * groups
    if remainder > 0:
        arr.append(remainder)
    return arr

### Gradient accumulation functions

In [ ]:
def split_iterable(it, split_size):
    accum = []
    for ind in range(ceil(len(it) / split_size)):
        start_index = ind * split_size
        accum.append(it[start_index: (start_index + split_size)])
    return accum

def split(t, split_size = None):
    if not exists(split_size):
        return t

    if isinstance(t, torch.Tensor):
        return t.split(split_size, dim = 0)

    if isinstance(t, Iterable):
        return split_iterable(t, split_size)

    return TypeError

def find_first(cond, arr):
    for el in arr:
        if cond(el):
            return el
    return None

def split_args_and_kwargs(*args, split_size = None, **kwargs):
    all_args     = (*args, *kwargs.values())
    len_all_args = len(all_args)
    first_tensor = find_first(lambda t: isinstance(t, torch.Tensor), all_args)
    assert exists(first_tensor)

    batch_size = len(first_tensor)
    split_size = default(split_size, batch_size)
    num_chunks = ceil(batch_size / split_size)

    dict_len   = len(kwargs)
    dict_keys  = kwargs.keys()
    split_kwargs_index = len_all_args - dict_len

    split_all_args = [split(arg, split_size = split_size) if exists(arg) and isinstance(arg, (torch.Tensor, Iterable)) else ((arg,) * num_chunks) for arg in all_args]
    chunk_sizes    = tuple(map(len, split_all_args[0]))

    for (chunk_size, *chunked_all_args) in tuple(zip(chunk_sizes, *split_all_args)):
        chunked_args, chunked_kwargs_values = chunked_all_args[:split_kwargs_index], chunked_all_args[split_kwargs_index:]
        chunked_kwargs  = dict(tuple(zip(dict_keys, chunked_kwargs_values)))
        chunk_size_frac = chunk_size / batch_size
        yield chunk_size_frac, (chunked_args, chunked_kwargs)

## Prior training dataset class

In [ ]:
class PriorEmbeddingDataset(IterableDataset):
    """
    PriorEmbeddingDataset is a wrapper of EmbeddingReader.

    It enables one to simplify the logic necessary to yield samples from
    the different EmbeddingReader configurations available.
    """
    def __init__(
            self,
            text_conditioned: bool,
            batch_size:       int,
            start:            int,
            stop:             int,
            image_reader,
            text_reader:      EmbeddingReader = None,
        ) -> None:
        super(PriorEmbeddingDataset).__init__()

        self.text_conditioned = text_conditioned

        if not self.text_conditioned:
            self.text_reader = text_reader

        self.image_reader = image_reader
        self.start        = start
        self.stop         = stop
        self.batch_size   = batch_size

    def __len__(self):
        return self.stop - self.start

    def __iter__(self):
        # D.R.Y loader args
        loader_args = dict(
            batch_size    = self.batch_size,
            start         = self.start,
            end           = self.stop,
            show_progress = False,
        )

        # if the data requested is text conditioned, only load images
        if self.text_conditioned:
            self.loader = self.image_reader(**loader_args)
        # otherwise, include text embeddings and bypass metadata
        else:
            self.loader = zip(
                self.image_reader(**loader_args), self.text_reader(**loader_args)
            )

        # return the data loader in its formatted state
        return self

    def __next__(self):
        try:
            return self.get_sample()
        except StopIteration:
            raise StopIteration

    def __str__(self):
        return f"<PriorEmbeddingDataset: start: {self.start}, stop: {self.stop}, len: {self.__len__()}>"

    def set_start(self, start):
        """
        Adjust the starting point within the reader, useful for resuming an epoch
        """
        self.start = start

    def get_start(self):
        return self.start

    def get_sample(self):
        """
        pre-proocess data from either reader into a common format
        """
        if self.text_conditioned:
            image_embedding, caption = next(self.loader)

            image_embedding   = torch.from_numpy(image_embedding)
            tokenized_caption = tokenize(caption["caption"].to_list(), truncate=True)

            return image_embedding, tokenized_caption

        else:
            (image_embedding, _), (text_embedding, _) = next(self.loader)

            image_embedding = torch.from_numpy(image_embedding)
            text_embedding  = torch.from_numpy(text_embedding)

            return image_embedding, text_embedding

### Training prior helper functions

## Decoder training dataset

In [ ]:
def distribute_to_rank(start, stop, rank, world_size):
    """
    Distribute data to each rank given the world size.

    Return:
        - New start and stop points for this rank.
    """
    num_samples = int(stop - start)

    per_rank = int(ceil((num_samples) / float(world_size)))

    assert (
        per_rank > 0
    ), f"Number of samples per rank must be larger than 0, (found: {per_rank})"

    rank_start = start + rank * per_rank

    rank_stop = min(rank_start + per_rank, stop)

    new_length = rank_stop - rank_start

    assert (
        new_length > 0
    ), "Calculated start and stop points result in a length of zero for this rank."

    return rank_start, rank_stop


def get_reader(
    text_conditioned: bool, img_url: str, meta_url: str = None, txt_url: str = None
):
    """
    Create an EmbeddingReader object from the specified URLs

    get_reader() will always expect a url to image embeddings.

    If text-conditioned, it will also expect a meta_url for the captions.
    Otherwise, it will need txt_url for the matching text embeddings.

    Returns an image_reader object if text-conditioned.
    Otherwise it returns both an image_reader and a text_reader
    """

    assert img_url is not None, "Must supply a image url"

    if text_conditioned:
        assert meta_url is not None, "Must supply meta url if text-conditioned"

        image_reader = EmbeddingReader(
            embeddings_folder=img_url,
            file_format="parquet_npy",
            # will assume the caption column exists and is the only one requested
            meta_columns=["caption"],
            metadata_folder=meta_url,
        )

        print(f'[INFO] created an image embeddings reader')

        return image_reader, None

    # otherwise we will require text embeddings as well and return two readers
    assert (
        txt_url is not None
    ), "Must supply text embedding url if not text-conditioning"

    image_reader = EmbeddingReader(img_url, file_format="npy")
    text_reader  = EmbeddingReader(txt_url, file_format="npy")

    print(f'[INFO] created image and text embeddings readers')

    return image_reader, text_reader


def make_splits(
        text_conditioned: bool,
        batch_size:       int,
        num_data_points:  int,
        train_split:      float,
        eval_split:       float,
        image_reader:     EmbeddingReader,
        text_reader:      EmbeddingReader,
        start             = 0,
        rank              = 0,
        world_size        = 1,
    ):
    """
    Split an embedding reader object as needed.

    NOTE: make_splits() will infer the test set size from your train and eval.

    Arguments:
    - text_conditioned: whether to prepare text-conditioned training data
    - batch_size: the batch size for a single gpu
    - num_data_points: the total number of data points you wish to train on
    - train_split: the percentage of data you wish to train on
    - eval_split: the percentage of data you wish to validate on
    - image_reader: the image_reader you wish to split
    - text_reader: the text_reader you want to split (if !text_conditioned)
    - start: the starting point within your dataset
    - rank: the rank of your worker
    - world_size: the total world size of your distributed training run

    Returns:
        - PyTorch Dataloaders that yield tuples of (img, txt) data.
    """

    dset_count = image_reader.count
    print(f'[INFO] dataset samples: {dset_count}')

    assert start < dset_count, "start position cannot exceed reader count."

    # verify that the num_data_points does not exceed the max points
    if num_data_points > (dset_count - start):
        print(
            "Specified count is larger than what is available...defaulting to reader's count."
        )
        num_data_points = dset_count

    # compute split points
    train_set_size = int(train_split * num_data_points)
    eval_set_size  = int(eval_split * num_data_points)
    eval_start     = train_set_size
    eval_stop      = int(eval_start + eval_set_size)

    print(f'[INFO] training   dataset size: {train_set_size}')
    print(f'[INFO] validation dataset size: {eval_set_size}')
    print(f'[INFO] testing    dataset size: {num_data_points-(train_set_size+eval_set_size)}')

    assert (
        train_split + eval_split
    ) < 1.0, "Specified train and eval split is too large to infer a test split."

    # distribute to rank
    rank_train_start, rank_train_stop = distribute_to_rank(
        start,
        train_set_size,
        rank,
        world_size
    )
    rank_eval_start, rank_eval_stop = distribute_to_rank(
        train_set_size,
        eval_stop,
        rank,
        world_size
    )
    rank_test_start, rank_test_stop = distribute_to_rank(
        eval_stop,
        num_data_points,
        rank,
        world_size
    )

    # wrap up splits into a dict
    train_split_args = dict(
        start      = rank_train_start, 
        stop       = rank_train_stop,
        batch_size = batch_size
    )
    eval_split_args = dict(
        start      = rank_eval_start,
        stop       = rank_eval_stop,
        batch_size = batch_size
    )
    test_split_args = dict(
        start      = rank_test_start,
        stop       = rank_test_stop,
        batch_size = batch_size
    )

    if text_conditioned:
        # add the text-conditioned args to a unified dict
        reader_args = dict(
            text_conditioned = text_conditioned,
            image_reader     = image_reader,
        )

        train_split_args = dict(**reader_args, **train_split_args)
        eval_split_args  = dict(**reader_args, **eval_split_args)
        test_split_args  = dict(**reader_args, **test_split_args)

        train = PriorEmbeddingDataset(**train_split_args)
        val   = PriorEmbeddingDataset(**eval_split_args)
        test  = PriorEmbeddingDataset(**test_split_args)

    else:
        # add the non-conditioned args to a unified dict
        reader_args = dict(
            text_conditioned = text_conditioned,
            image_reader     = image_reader,
            text_reader      = text_reader,
        )

        train_split_args = dict(**reader_args, **train_split_args)
        eval_split_args  = dict(**reader_args, **eval_split_args)
        test_split_args  = dict(**reader_args, **test_split_args)

        train = PriorEmbeddingDataset(**train_split_args)
        val   = PriorEmbeddingDataset(**eval_split_args)
        test  = PriorEmbeddingDataset(**test_split_args)

    # true batch size is specifed in the PriorEmbeddingDataset
    train_loader = DataLoader(train, batch_size=None)
    eval_loader  = DataLoader(val,   batch_size=None)
    test_loader  = DataLoader(test,  batch_size=None)

    return train_loader, eval_loader, test_loader

In [ ]:
def get_shard(filename):
    """
    Filenames with shards in them have a consistent structure that we can take 
    advantage of standard structure: path/to/file/prefix_string_00001.ext
    """
    try:
        return filename.split("_")[-1].split(".")[0]
    except ValueError:
        raise RuntimeError(f"Could not find shard for filename {filename}")


def get_example_file(fs, path, file_format):
    """
    Given a file system and a file extension, returns the example file.
    """
    return fs.glob(os.path.join(path, f"*.{file_format}"))[0]


def embedding_inserter(samples, embeddings_url, index_width, sample_key='npy', handler=wds.handlers.reraise_exception):
    """
    Given a datum of {"__key__": str, "__url__": str, ...} adds the cooresponding embedding and yields.
    """
    previous_tar_url   = None
    current_embeddings = None
    # Get a reference to an abstract file system where the embeddings are stored
    embeddings_fs, embeddings_path = fsspec.core.url_to_fs(embeddings_url)
    example_embedding_file  = get_example_file(embeddings_fs, embeddings_path, "npy")
    example_embedding_shard = get_shard(example_embedding_file)
    emb_shard_width = len(example_embedding_shard)
    # Easier to get the basename without the shard once than search through for the correct file every time
    embedding_file_basename = '_'.join(example_embedding_file.split("_")[:-1]) + "_"

    def load_corresponding_embeds(tar_url):
      """
      Finds and reads the npy files that contains embeddings for the given webdataset tar.
      """
      shard         = int(tar_url.split("/")[-1].split(".")[0])
      embedding_url = embedding_file_basename + str(shard).zfill(emb_shard_width) + '.npy'
      with embeddings_fs.open(embedding_url) as f:
        data = np.load(f)
      return torch.from_numpy(data)

    for sample in samples:
        try:
            tar_url = sample["__url__"]
            key = sample["__key__"]
            if tar_url != previous_tar_url:
                # If the tar changed, we need to download new embeddings
                # This means if we shuffle before inserting it will load many more files than we expect and be very inefficient.
                previous_tar_url = tar_url
                current_embeddings = load_corresponding_embeds(tar_url)
                
            embedding_index = int(key[-index_width:])
            embedding = current_embeddings[embedding_index]
            # We need to check if this sample is nonzero. If it is, this embedding is not valid and we should continue to the next loop
            if torch.count_nonzero(embedding) == 0:
                raise RuntimeError(f"Webdataset had a sample, but no embedding was found. ImgShard: {key[:-index_width]} - Index: {key[-index_width:]}")
            sample[sample_key] = embedding
            yield sample
        except Exception as exn:  # From wds implementation
            if handler(exn):
                continue
            else:
                break


def unassociated_shard_skipper(tarfiles, embeddings_url, handler=wds.handlers.reraise_exception):
    """
    Finds if the is a corresponding embedding for the tarfile at { url: [URL] }.
    """
    embeddings_fs, embeddings_path = fsspec.core.url_to_fs(embeddings_url)
    embedding_files     = embeddings_fs.ls(embeddings_path)
    get_embedding_shard = lambda embedding_file: int(embedding_file.split("_")[-1].split(".")[0])
    embedding_shards    = set([get_embedding_shard(filename) for filename in embedding_files])  # Sets have O(1) check for member

    get_tar_shard = lambda tar_file: int(tar_file.split("/")[-1].split(".")[0])
    for tarfile in tarfiles:
        try:
            webdataset_shard = get_tar_shard(tarfile["url"])
            # If this shard has an associated embeddings file, we pass it through. Otherwise we iterate until we do have one
            if webdataset_shard in embedding_shards:
                yield tarfile
        except Exception as exn:  # From wds implementation
            if handler(exn):
                continue
            else:
                break


def join_embeddings(samples, handler=wds.handlers.reraise_exception):
    """
    Takes the img_emb and text_emb keys and turns them into one key "emb": { "text": text_emb, "img": img_emb }
    either or both of text_emb and img_emb may not be in the sample so we only add the ones that exist
    """
    for sample in samples:
        try:
            sample['emb'] = {}
            if 'text_emb' in sample:
                sample['emb']['text'] = sample['text_emb']
            if 'img_emb' in sample:
                sample['emb']['img'] = sample['img_emb']
            yield sample
        except Exception as exn:  # From wds implementation
            if handler(exn):
                continue
            else:
                break


def verify_keys(samples, required_keys, handler=wds.handlers.reraise_exception):
    """
    Requires that both the image and embedding are present in the sample
    This is important to do as a user may forget they do not have embeddings in their webdataset and neglect to add them using the embedding_folder_url parameter.
    """
    for sample in samples:
        try:
            for key in required_keys:
                assert key in sample, f"Sample {sample['__key__']} missing {key}. Has keys {sample.keys()}"
            yield sample
        except Exception as exn:  # From wds implementation
            if handler(exn):
                continue
            else:
                break

In [ ]:
class ImageEmbeddingDataset(wds.DataPipeline, wds.compat.FluidInterface):
    """
    A interface wrapper for DataPipline that returns image embedding pairs.
    Reads embeddings as NPY files from the webdataset if they exist.
    If embedding_folder_url is set, embeddings will be read from an alternative folder.
    """

    def __init__(
            self,
            urls,
            img_embedding_folder_url  = None,
            text_embedding_folder_url = None,
            index_width               = None,
            img_preproc               = None,
            extra_keys                = [],
            handler                   = wds.handlers.reraise_exception,
            resample                  = False,
            shuffle_shards            = True
        ):
        """
        Modeled according to the WebDataset constructor.

        Arguments:
        * urls: A url pointing to the tar files of the webdataset formatted as /path/to/webdataset/{0000..9999}.tar
        * embedding_folder_url: Required if webdataset does not contain embeddings. A url pointing to the npy files of the embeddings. Should have the same number of shards as the webdataset.
                Webdataset image keys should align with the index of the embedding. This means missing image indices must have a corresponding embedding of all zeros.
        * index_width: The number of digits in the index. This is used to align the embedding index with the image index.
            For example, if a file in the webdataset shard 3 is named 0003039.jpg, we know the shard is 4 digits and the last 3 digits are the index_width.
        * img_preproc: This function is run on the img before it is batched and returned. Useful for data augmentation or converting to torch tensor.
        * handler:  A webdataset handler.
        * resample: If true, resample webdataset shards with replacement. You need to set your own epoch size if this is true since it will resample infinitely.
        * shuffle_shards: If true, shuffle the shards before resampling. This cannot be true if resample is true.
        """
        super().__init__()
        keys = ["jpg", "emb"] + extra_keys
        # if img_embedding_folder_url is not None:
        #     keys.append("img_emb")
        # if text_embedding_folder_url is not None:
        #     keys.append("text_emb")
        # keys.extend(extra_keys)
        self.key_map     = {key: i for i, key in enumerate(keys)}
        self.resampling  = resample
        self.img_preproc = img_preproc

        # If s3, check if s3fs is installed and s3cmd is installed and check if the data is piped instead of straight up
        if (isinstance(urls, str) and "s3:" in urls) or (isinstance(urls, list) and any(["s3:" in url for url in urls])):
            # Then this has an s3 link for the webdataset and we need extra packages
            if shutil.which("s3cmd") is None:
                raise RuntimeError("s3cmd is required for s3 webdataset")
        if (img_embedding_folder_url is not None and "s3:" in img_embedding_folder_url) or (text_embedding_folder_url is not None and "s3:" in text_embedding_folder_url):
            # Then the embeddings are being loaded from s3 and fsspec requires s3fs
            try:
                import s3fs
            except ImportError:
                raise RuntimeError("s3fs is required to load embeddings from s3")

        # Add the shardList and randomize or resample if requested
        if resample:
            assert not shuffle_shards, "Cannot both resample and shuffle"
            self.append(wds.ResampledShards(urls))
        else:
            self.append(wds.SimpleShardList(urls))
            if shuffle_shards:
                self.append(wds.filters.shuffle(1000))

        if img_embedding_folder_url is not None:
            # There may be webdataset shards that do not have a embedding shard associated with it.
            # If we do not skip these, they would cause issues.
            self.append(skip_unassociated_shards(embeddings_url=img_embedding_folder_url, handler=handler))

        if text_embedding_folder_url is not None:
            self.append(skip_unassociated_shards(embeddings_url=text_embedding_folder_url, handler=handler))

        self.append(wds.tarfile_to_samples(handler=handler))
        self.append(wds.decode("pilrgb", handler=handler))
        if img_embedding_folder_url is not None:
            # Then we are loading image embeddings for a remote source
            assert index_width is not None, "Reading embeddings separately requires index width length to be given"
            self.append(insert_embedding(embeddings_url=img_embedding_folder_url, index_width=index_width, sample_key='img_emb', handler=handler))

        if text_embedding_folder_url is not None:
            # Then we are loading image embeddings for a remote source
            assert index_width is not None, "Reading embeddings separately requires index width length to be given"
            self.append(insert_embedding(embeddings_url=text_embedding_folder_url, index_width=index_width, sample_key='text_emb', handler=handler))

        self.append(join_embeddings)
        self.append(key_verifier(required_keys=keys, handler=handler))

        # Apply preprocessing
        self.append(wds.map(self.preproc))
        self.append(wds.to_tuple(*keys))

    def preproc(self, sample):
        """
        Applies the preprocessing for images.
        """
        if self.img_preproc is not None:
            sample["jpg"] = self.img_preproc(sample["jpg"])
        return sample


def create_image_embedding_dataloader(
        tar_url,
        num_workers,
        batch_size,
        img_embeddings_url  = None,
        text_embeddings_url = None,
        index_width         = None,
        shuffle_num         = None,
        shuffle_shards      = True,
        resample_shards     = False, 
        img_preproc         = None,
        extra_keys          = [],
        handler             = wds.handlers.reraise_exception # warn_and_continue
    ):
    """
    Creates an image embedding dataset and a dataloader.

    Arguments:
    * tar_url:         A url pointing to the tar files of the webdataset
                       formatted as '/path/to/webdataset/{0000..9999}.tar'.
    * num_workers:     The number of workers to use for the dataloader.
    * batch_size:      The batch size to use for the dataloader.
    * embeddings_url:  Required if webdataset does not contain embeddings.
                       A URL pointing to the NPY files of the embeddings.
                       Should have the same number of shards as the webdataset.
                       Webdataset image keys should align with the index of the embedding.
                       This means missing image indices must have a corresponding embedding of all zeros.
    * index_width:     The number of digits in the index. This is used to align
                       the embedding index with the image index. For example, if a file in the
                       webdataset shard 3 is named 0003039.jpg, we know the shard is 4 digits
                       and the last 3 digits are the index_width.
    * shuffle_num:     If not None, shuffle the dataset with this size buffer after sampling.
    * shuffle_shards:  If true, shuffle the shards before sampling.
                       This cannot be true if resample is true.
    * resample_shards: If true, resample webdataset shards with replacement. We need to set 
                       our own epoch size if this is true since it will resample infinitely.
    * handler:         A webdataset handler.
    """
    ds = ImageEmbeddingDataset(
        tar_url,
        img_embedding_folder_url  = img_embeddings_url,
        text_embedding_folder_url = text_embeddings_url,
        index_width               = index_width,
        shuffle_shards            = shuffle_shards,
        resample                  = resample_shards,
        extra_keys                = extra_keys,
        img_preproc               = img_preproc,
        handler                   = handler
    )

    if shuffle_num is not None and shuffle_num > 0:
        ds.shuffle(1000)

    #print(f'[DEBUG] create_image_embedding_dataloader(): batch size is {batch_size}')

    return DataLoader(
        ds,
        num_workers     = num_workers,
        batch_size      = batch_size,
        prefetch_factor = 2,  # This might be good to have high so the next npy file is prefetched
        pin_memory      = True,
        shuffle         = False,
        drop_last       = True
    )

## Dataset and DataLoader for images in a folder

In [ ]:
class Dataset(data.Dataset):
    def __init__(
        self,
        folder,
        image_size,
        exts = ['jpg', 'jpeg', 'png']
    ):
        super().__init__()
        self.folder     = folder
        self.image_size = image_size
        self.paths      = [p for ext in exts for p in Path(f'{folder}').glob(f'**/*.{ext}')]

        self.transform = T.Compose([
            T.Resize(image_size),
            T.RandomHorizontalFlip(),
            T.CenterCrop(image_size),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        img  = Image.open(path)
        return self.transform(img)

def get_images_dataloader(
        folder,
        *,
        batch_size,
        image_size,
        shuffle    = True,
        cycle_dl   = True,
        pin_memory = True
    ):
    ds = Dataset(folder, image_size)
    dl = data.DataLoader(
        ds,
        batch_size = batch_size,
        shuffle    = shuffle,
        pin_memory = pin_memory,
        drop_last  = True
    )

    if cycle_dl:
        dl = cycle(dl)
    return dl


In [ ]:
insert_embedding         = wds.filters.pipelinefilter(embedding_inserter)

skip_unassociated_shards = wds.filters.pipelinefilter(unassociated_shard_skipper)

key_verifier             = wds.filters.pipelinefilter(verify_keys)

In [ ]:
def create_dataloaders(
        available_shards,
        webdataset_base_url,
        img_embeddings_url  = None,
        text_embeddings_url = None,
        shard_width         = 6,
        num_workers         = 4,
        batch_size          = 32,
        n_sample_images     = 6,
        shuffle_train       = True,
        resample_train      = False,
        img_preproc         = None,
        index_width         = 4,
        train_prop          = 0.75,
        val_prop            = 0.15,
        test_prop           = 0.10,
        seed                = 0,
        **kwargs
    ):
    """
    Randomly splits the available shards into train, val, and test sets.
    and returns a dataloader for each.
    """
    assert train_prop + test_prop + val_prop == 1
    num_train = round(train_prop*len(available_shards))
    num_test  = round(test_prop*len(available_shards))
    num_val   = len(available_shards) - num_train - num_test
    assert num_train + num_test + num_val == len(available_shards), f"{num_train} + {num_test} + {num_val} = {num_train + num_test + num_val} != {len(available_shards)}"

    train_split, test_split, val_split = torch.utils.data.random_split(
        available_shards,
        [num_train, num_test, num_val],
        generator = torch.Generator().manual_seed(seed)
    )

    print(f'[INFO] The number of training shards is {num_train}')
    for shard in train_split:
        print(f'{shard}', end='  ')
    print(f'\n[INFO] The number of validation shards is {num_val}')
    for shard in val_split:
        print(f'{shard}', end='  ')
    print(f'\n[INFO] The number of testing shards is {num_test}')
    for shard in test_split:
        print(f'{shard}', end='  ')

    #print(f'[DEBUG] create_dataloaders(): batch size is {batch_size}')
    #print(f'[DEBUG] create_dataloaders(): shard width is  {shard_width}')
    #print(f'[DEBUG] create_dataloaders(): webdataset base url is {webdataset_base_url}')

    # The shard number in the webdataset file names has a fixed width.
    # We zero pad the shard numbers so they correspond to a filename.
    train_urls = [webdataset_base_url.format(str(shard).zfill(shard_width)) for shard in train_split]
    test_urls  = [webdataset_base_url.format(str(shard).zfill(shard_width)) for shard in test_split]
    val_urls   = [webdataset_base_url.format(str(shard).zfill(shard_width)) for shard in val_split]

    #for url in train_urls: print(f'[DEBUG] training url:   {url}')
    #for url in val_urls:   print(f'[DEBUG] validation url: {url}')
    #for url in test_urls:  print(f'[DEBUG] test url:       {url}')

    create_dataloader = lambda tar_urls, shuffle=False, resample=False, for_sampling=False: \
        create_image_embedding_dataloader(
            tar_url             = tar_urls,
            num_workers         = num_workers,
            batch_size          = batch_size if not for_sampling else n_sample_images,
            img_embeddings_url  = img_embeddings_url,
            text_embeddings_url = text_embeddings_url,
            index_width         = index_width,
            shuffle_num         = None,
            extra_keys          = ["txt"],
            shuffle_shards      = shuffle,
            resample_shards     = resample, 
            img_preproc         = img_preproc,
            handler             = wds.handlers.warn_and_continue
        )

    train_dataloader          = create_dataloader(train_urls, shuffle=shuffle_train, resample=resample_train)
    train_sampling_dataloader = create_dataloader(train_urls, shuffle=False, for_sampling=True)
    val_dataloader            = create_dataloader(val_urls,   shuffle=False)
    test_dataloader           = create_dataloader(test_urls,  shuffle=False)
    test_sampling_dataloader  = create_dataloader(test_urls,  shuffle=False, for_sampling=True)

    return {
        "train":          train_dataloader,
        "train_sampling": train_sampling_dataloader,
        "val":            val_dataloader,
        "test":           test_dataloader,
        "test_sampling":  test_sampling_dataloader
    }

In [ ]:
def get_dataset_keys(dataloader):
    """
    It is sometimes neccesary to get the keys the dataloader is returning.
    Since the dataset is burried in the dataloader, we need to do a process to recover it.
    """
    # If the dataloader is actually a WebLoader, we need to extract the real dataloader
    if isinstance(dataloader, wds.WebLoader):
        dataloader = dataloader.pipeline[0]
    return dataloader.dataset.key_map

def get_example_data(dataloader, device, n=5):
    """
    Sample the dataloader and returns a zipped list of examples.
    """
    images          = []
    img_embeddings  = []
    text_embeddings = []
    captions        = []

    for img, emb, txt in dataloader:
        img_emb, text_emb = emb.get('img'), emb.get('text')

        # print(f'[DEBUG] get_example_data(): img_emb shape is {img_emb.shape}')

        if img_emb is not None:
            img_emb = img_emb.to(device=device, dtype=torch.float)
            img_embeddings.extend(list(img_emb))
        else:
            # Then we add None img.shape[0] times
            img_embeddings.extend([None]*img.shape[0])
        if text_emb is not None:
            text_emb = text_emb.to(device=device, dtype=torch.float)
            text_embeddings.extend(list(text_emb))
        else:
            # Then we add None img.shape[0] times
            text_embeddings.extend([None]*img.shape[0])

        img = img.to(device=device, dtype=torch.float)
        images.extend(list(img))
        captions.extend(list(txt))
        if len(images) >= n:
            break

    return list(zip(images[:n], img_embeddings[:n], text_embeddings[:n], captions[:n]))

In [ ]:
def generate_samples(
        trainer,
        example_data,
        clip                        = None,
        start_unet                  = 1,
        end_unet                    = None,
        condition_on_text_encodings = False,
        cond_scale                  = 1.0,
        device                      = None,
        text_prepend                = "",
        match_image_size            = True
    ):
    """
    Takes example data and generates images from the embeddings.
    Returns three lists: real images, generated images, and captions.
    """
    real_images, img_embeddings, text_embeddings, txts = zip(*example_data)
    sample_params = {}

    if img_embeddings[0] is None:
        # Generate image embeddings from clip
        imgs_tensor = torch.stack(real_images)
        assert clip is not None, "clip is None, but img_embeddings is None"
        imgs_tensor.to(device=device)
        img_embeddings, img_encoding = clip.embed_image(imgs_tensor)
        sample_params["image_embed"] = img_embeddings
    else:
        # Then we are using precomputed image embeddings
        img_embeddings               = torch.stack(img_embeddings)
        sample_params["image_embed"] = img_embeddings

    if condition_on_text_encodings:
        if text_embeddings[0] is None:
            # Generate text embeddings from text
            assert clip is not None, "clip is None, but text_embeddings is None"
            tokenized_texts                 = tokenize(txts, truncate=True).to(device=device)
            text_embed, text_encodings      = clip.embed_text(tokenized_texts)
            sample_params["text_encodings"] = text_encodings
        else:
            # Then we are using precomputed text embeddings
            text_embeddings                 = torch.stack(text_embeddings)
            sample_params["text_encodings"] = text_embeddings

    sample_params["start_at_unet_number"] = start_unet
    sample_params["stop_at_unet_number"]  = end_unet
    if start_unet > 1:
        # If we are only training upsamplers
        sample_params["image"] = torch.stack(real_images)

    if device is not None:
        sample_params["_device"] = device

    if start_unet > 1:                             ###################################### [DEBUG] RUN SAMPLING ON CPU
        sample_params["image"].to(device)          ###################################### [DEBUG] RUN SAMPLING ON CPU
    sample_params["image_embed"].to(device)        ###################################### [DEBUG] RUN SAMPLING ON CPU
    if condition_on_text_encodings:                ###################################### [DEBUG] RUN SAMPLING ON CPU
        sample_params["text_encodings"].to(device) ###################################### [DEBUG] RUN SAMPLING ON CPU

    samples          = trainer.sample(
        **sample_params,
        _cast_deepspeed_precision=False
    )  # At sampling time we do not want to cast to FP16

    generated_images = list(samples)
    captions         = [text_prepend + txt for txt in txts]

    if match_image_size:
        generated_image_size = generated_images[0].shape[-1]
        real_images = [resize_image_to(image, generated_image_size, clamp_range=(0, 1)) for image in real_images]
    return real_images, generated_images, captions

In [ ]:
def generate_grid_samples(
        trainer,
        examples,
        clip                        = None,
        start_unet                  = 1,
        end_unet                    = None,
        condition_on_text_encodings = False,
        cond_scale                  = 1.0,
        device                      = None,
        text_prepend                = "",
    ):
    """
    Generates samples and uses torchvision to put them in a side by side grid 
    for easy viewing.
    """
    real_images, generated_images, captions = generate_samples(
        trainer,
        examples,
        clip,
        start_unet,
        end_unet,
        condition_on_text_encodings,
        cond_scale,
        device,
        text_prepend
    )
    grid_images = [
        torchvision.utils.make_grid(
            [
                original_image,
                generated_image
            ]
        ) for original_image, generated_image in zip(real_images, generated_images)
    ]
    return grid_images, captions

## Prior training utilities

In [ ]:
def make_model(
        prior_config: DiffusionPriorConfig,
        train_config: DiffusionPriorTrainConfig,
        device:       str         = None,
        accelerator:  Accelerator = None,
    ):
    # Create the prior model given the configuration
    diffusion_prior = prior_config.create()

    # Instantiate the prior trainer
    trainer = DiffusionPriorTrainer(
        diffusion_prior = diffusion_prior,
        lr              = train_config.lr,
        wd              = train_config.wd,
        max_grad_norm   = train_config.max_grad_norm,
        amp             = train_config.amp,
        use_ema         = train_config.use_ema,
        device          = device,
        accelerator     = accelerator,
        warmup_steps    = train_config.warmup_steps,
    )

    return trainer

In [ ]:
def create_tracker(
        accelerator: Accelerator,
        config:      TrainDiffusionPriorConfig,
        config_path: str,
        dummy:       bool = False,
    ) -> Tracker:

    tracker_config = config.tracker

    accelerator_config = {
        "Distributed":     accelerator.distributed_type
                           != accelerate_dataclasses.DistributedType.NO,
        "DistributedType": accelerator.distributed_type,
        "NumProcesses":    accelerator.num_processes,
        "MixedPrecision":  accelerator.mixed_precision,
    }

    tracker: Tracker = tracker_config.create(
        config,
        accelerator_config,
        dummy_mode = dummy
    )

    tracker.save_config(
        config_path,
        config_name = "prior_config.json"
    )

    return tracker

In [ ]:
def pad_gather_reduce(trainer: DiffusionPriorTrainer, x, method="mean"):
    """
    Pad a value or tensor across all processes and gather.

    Arguments:
        * trainer: a trainer that carries an accelerator object
        * x:       a number or torch tensor to reduce
        * method:  "mean", "sum", "max", "min"

    Returns:
        * the average tensor after masking out 0's
        * 'None' if the gather resulted in an empty tensor
    """

    assert method in [
        "mean",
        "sum",
        "max",
        "min",
    ], "This function has limited capabilities [sum, mean, max, min]"

    assert type(x) is not None, "Cannot reduce a None type object"

    # wait for everyone to arrive here before gathering

    if type(x) is not torch.Tensor:
        x = torch.tensor([x])

    # verify that the tensor is on the proper device
    x = x.to(trainer.device)

    # pad across processes
    padded_x = trainer.accelerator.pad_across_processes(x, dim=0)

    # gather across all procesess
    gathered_x = trainer.accelerator.gather(padded_x)

    # mask out zeros
    masked_x = gathered_x[gathered_x != 0]

    # if the tensor is empty, warn and return None
    if len(masked_x) == 0:
        print(
            f"The call to this method resulted in an empty tensor after masking out zeros. The gathered tensor was this: {gathered_x} and the original value passed was: {x}."
        )
        return None

    if method == "mean":
        return torch.mean(masked_x)
    elif method == "sum":
        return torch.sum(masked_x)
    elif method == "max":
        return torch.max(masked_x)
    elif method == "min":
        return torch.min(masked_x)

In [ ]:
def save_trainer(
        tracker:             Tracker,
        trainer:              DiffusionPriorTrainer,
        is_latest:            bool,
        is_best:              bool,
        epoch:                int,
        samples_seen:         int,
        best_validation_loss: float,
    ):
    """
    Save the prior model with an appropriate method depending on the tracker.
    """
    trainer.accelerator.wait_for_everyone()

    if trainer.accelerator.is_main_process:
        print(
            f"RANK: {trainer.accelerator.process_index} | Saving model | Best={is_best} | Latest={is_latest}"
        )

    tracker.save(
        trainer              = trainer,
        is_best              = is_best,
        is_latest            = is_latest,
        epoch                = int(epoch),
        samples_seen         = int(samples_seen),
        best_validation_loss = best_validation_loss,
    )

In [ ]:
def recall_trainer(tracker: Tracker, trainer: DiffusionPriorTrainer):
    """
    Load the prior model with an appropriate method depending on the tracker.
    """

    if trainer.accelerator.is_main_process:
        print(f"[INFO] Loading prior model from {type(tracker.loader).__name__}")

    state_dict = tracker.recall()

    trainer.load(state_dict, strict=True)

    return (
        int(state_dict.get("epoch", 0)),
        state_dict.get("best_validation_loss", 0),
        int(state_dict.get("samples_seen", 0)),
    )

## Prior evaluation functions

In [ ]:
def report_validation_loss(
        trainer:          DiffusionPriorTrainer,
        dataloader:       DataLoader,
        iterations:       int,
        text_conditioned: bool,
        use_ema:          bool,
        type_model:       str,
        tracker:          Tracker,
        split:            str,
        tracker_folder:   str,
        loss_type:        str,
    ):
    """
    Calculate the validation loss on a given subset of data.
    """
    if trainer.accelerator.is_main_process:
        print(f"[INFO] Calculating average loss on {type_model}-{split} split ...")

    total_loss = torch.zeros(1, dtype=torch.float, device=trainer.device)

    it = 0
    for image_embeddings, text_data in dataloader:
        it += 1

        image_embeddings = image_embeddings.to(trainer.device)
        text_data        = text_data.to(trainer.device)

        input_args       = dict(image_embed=image_embeddings)

        if text_conditioned:
            input_args   = dict(**input_args, text=text_data)
        else:
            input_args   = dict(**input_args, text_embed=text_data)

        if use_ema:
            loss = trainer.ema_diffusion_prior(**input_args)

            is_nan = torch.isnan(loss)
            if(is_nan.item() == True):
                print(f'NaN ', end='')
                loss = torch.zeros(1, dtype=torch.float, device=trainer.device)
        else:
            loss = trainer(**input_args)

            is_nan = math.isnan(loss)
            if(is_nan == True):
                print(f'NaN ', end='')
                loss = 0.0

        total_loss += loss

    print('\n')
    if it !=  0:
        total_loss /= it

    # Calculate the average loss across all processes

    avg_loss = pad_gather_reduce(trainer, total_loss, method="mean")
    stats    = {f"{tracker_folder}/{loss_type}-loss": avg_loss}

    # Print and log results if this is the main process
    tracker.log(stats, step=trainer.step.item() + 1)

    print(f'[INFO] Average {loss_type}-loss: {avg_loss}')

    return avg_loss

In [ ]:
def report_cosine_similarities(
        trainer:          DiffusionPriorTrainer,
        dataloader:       DataLoader,
        iterations:       int,
        text_conditioned: bool,
        tracker:          Tracker,
        split:            str,
        timesteps:        int,
        tracker_folder:   str,
    ):
    '''
    Calculate cosine similarities.
    '''
    trainer.eval()
    if trainer.accelerator.is_main_process:
        print(
            f"[INFO] Calculating cosine similarity on {split} split with {timesteps} timesteps ..."
        )

    it = 0
    for test_image_embeddings, text_data in dataloader:

        it += 1
        test_image_embeddings = test_image_embeddings.to(trainer.device)
        text_data             = text_data.to(trainer.device)

        # If using text conditioning, produce a text embedding from the tokenized text
        if text_conditioned:
            text_embedding, text_encodings = trainer.embed_text(text_data)
            text_cond = dict(
                text_embed     = text_embedding,
                text_encodings = text_encodings
            )
        else:
            text_embedding = text_data
            text_cond      = dict(text_embed=text_embedding)

        # Make a copy of the text embeddings for shuffling
        text_embed_shuffled = text_embedding.clone()

        # Roll the text to simulate "unrelated" captions
        rolled_idx          = torch.roll(torch.arange(text_embedding.shape[0]), 1)
        text_embed_shuffled = text_embed_shuffled[rolled_idx]
        text_embed_shuffled = text_embed_shuffled / text_embed_shuffled.norm(
            dim=1,
            keepdim=True
        )

        if text_conditioned:
            text_encodings_shuffled = text_encodings[rolled_idx]
        else:
            text_encodings_shuffled = None

        text_cond_shuffled = dict(
            text_embed     = text_embed_shuffled,
            text_encodings = text_encodings_shuffled
        )

        # Prepare the text embedding
        text_embed = text_embedding / text_embedding.norm(dim=1, keepdim=True)

        # Prepare the image embeddings
        test_image_embeddings = test_image_embeddings / test_image_embeddings.norm(
            dim=1,
            keepdim=True
        )

        # Predict image embedding on the unshuffled text embeddings
        predicted_image_embeddings = trainer.p_sample_loop(
            test_image_embeddings.shape,
            text_cond,
            timesteps = timesteps,
        )

        predicted_image_embeddings = (
            predicted_image_embeddings
            / predicted_image_embeddings.norm(dim=1, keepdim=True)
        )

        # Predict image embedding on the shuffled embeddings
        predicted_unrelated_embeddings = trainer.p_sample_loop(
            test_image_embeddings.shape,
            text_cond_shuffled,
            timesteps = timesteps,
        )

        predicted_unrelated_embeddings = (
            predicted_unrelated_embeddings
            / predicted_unrelated_embeddings.norm(dim=1, keepdim=True)
        )

        # Calculate similarities
        orig_sim = pad_gather_reduce(
            trainer,
            cos(text_embed, test_image_embeddings),
            method = "mean"
        )
        pred_sim = pad_gather_reduce(
            trainer,
            cos(text_embed, predicted_image_embeddings),
            method = "mean"
        )
        unrel_sim = pad_gather_reduce(
            trainer,
            cos(text_embed, predicted_unrelated_embeddings),
            method = "mean"
        )
        pred_img_sim = pad_gather_reduce(
            trainer,
            cos(test_image_embeddings, predicted_image_embeddings),
            method = "mean",
        )

        stats = {
            f"{tracker_folder}/baseline similarity [steps={timesteps}]": orig_sim,
            f"{tracker_folder}/similarity with text [steps={timesteps}]": pred_sim,
            f"{tracker_folder}/similarity with original image [steps={timesteps}]": pred_img_sim,
            f"{tracker_folder}/similarity with unrelated caption [steps={timesteps}]": unrel_sim,
            f"{tracker_folder}/difference from baseline similarity [steps={timesteps}]": pred_sim - orig_sim,
        }

        tracker.log(stats, step=trainer.step.item() + 1)

        print(f'{it}/{iterations} similarities (steps={timesteps}): ', end='')
        print(f'base {orig_sim :.8f} | ', end = '')
        print(f'text {pred_sim :.8f} | ', end = '')
        print(f'image {pred_img_sim :.8f} | ', end = '')
        print(f'other caption {unrel_sim :.8f} | ', end = '')
        print(f'diff base {pred_sim - orig_sim :.8f}')


In [ ]:
def prior_eval(
        trainer:          DiffusionPriorTrainer,
        dataloader:       DataLoader,
        iterations:       int,
        text_conditioned: bool,
        split:            str,
        tracker:          Tracker,
        use_ema:          bool,
        report_cosine:    bool,
        report_loss:      bool,
        timesteps:        List[int],
        loss_type:        str = None,
    ):
    """
    Run evaluation of the prior model and track metrics.

    Returns: the loss, if requested.
    """
    trainer.eval()

    type_model     = "ema" if use_ema else "online"
    tracker_folder = f"metrics/{type_model}-{split}"

    # Determine if valid timesteps are passed

    min_timesteps = trainer.accelerator.unwrap_model(
        trainer.diffusion_prior
    ).sample_timesteps
    max_timesteps = trainer.accelerator.unwrap_model(
        trainer.diffusion_prior
    ).noise_scheduler.num_timesteps

    assert all_between(
        timesteps, lower_bound=min_timesteps, upper_bound=max_timesteps
    ), f"all timesteps values must be between {min_timesteps} and {max_timesteps}: got {timesteps}"

    # Calculate the cosine similarity for various eta's and timesteps

    if report_cosine:
        for timestep in timesteps:
            report_cosine_similarities(
                trainer,
                dataloader       = dataloader,
                iterations       = iterations,
                text_conditioned = text_conditioned,
                tracker          = tracker,
                split            = split,
                timesteps        = timestep,
                tracker_folder   = tracker_folder,
            )

    # Calculate the loss on a seperate split of data

    if report_loss:
        loss = report_validation_loss(
            trainer          = trainer,
            dataloader       = dataloader,
            iterations       = iterations,
            text_conditioned = text_conditioned,
            use_ema          = use_ema,
            type_model       = type_model,
            tracker          = tracker,
            split            = split,
            tracker_folder   = tracker_folder,
            loss_type        = loss_type,
        )

        return loss

In [ ]:
def prior_train(
        trainer:      DiffusionPriorTrainer,
        tracker:      Tracker,
        train_loader: DataLoader,
        eval_loader:  DataLoader,
        test_loader:  DataLoader,
        config:       DiffusionPriorTrainConfig,
    ):
    '''
    The main function for training the prior model.
    '''
    # Initialize the timers
    samples_timer        = Timer()  # samples/sec
    validation_profiler  = Timer()  # how long is validation taking

    # Keep track of the best validation loss

    best_validation_loss = config.train.best_validation_loss
    samples_seen         = config.train.num_samples_seen

    bsize                = config.data.batch_size
    data_points          = config.data.num_data_points
    train_split          = config.data.splits.train
    val_split            = config.data.splits.val
    train_points         = train_split * data_points
    train_iterations     = int(math.ceil(train_points/bsize))
    val_points           = val_split * data_points
    val_iterations       = int(math.ceil(val_points/bsize))
    test_points          = data_points - (int(train_points) + int(val_points))
    test_iterations      = int(math.ceil(test_points/bsize))

    print(f'[INFO] train samples:      {int(train_points)} | ', end='')
    print(f'train iterations:      {train_iterations}')
    print(f'[INFO] validation samples: {int(val_points)} | ', end='')
    print(f'validation iterations: {val_iterations}')
    print(f'[INFO] test samples:       {int(test_points)} | ', end='')
    print(f'test iterations:       {test_iterations}')

    # ...................... training cycle .............................

    start_epoch = config.train.current_epoch

    for epoch in range(start_epoch, config.train.epochs):

        # If we finished out an old epoch, reset the distribution to be a full epoch
        tracker.log({"tracking/epoch": epoch}, step=trainer.step.item())

        if train_loader.dataset.get_start() > 0 and epoch == start_epoch+1:
            if trainer.accelerator.is_main_process:
                print("[INFO] Finished a resumed epoch, so we reset the dataloader.")
            train_loader.dataset.set_start(0)

        # Iterate over image embedding, text embedding pairs
        for img, txt in train_loader:
            # Setup things at every step beginning

            trainer.train()
            current_step = trainer.step.item()
            samples_timer.reset()

            # ############################################################################## [DEBUG] BLOCK BEGIN
            #for i in range(4):
            #    aux_img = img[i]
            #    aux_txt = txt[i]
            #    print(f'[DEBUG] img[{i}] shape={}: {aux_img}')
            #    print(f'[DEBUG] txt[{i}]: {aux_txt}')
            #return
            # ############################################################################## [DEBUG] BLOCK END

            # Put the data on the computing device

            img = img.to(trainer.device)
            txt = txt.to(trainer.device)

            # Pass the sampled data to the model

            loss = trainer(text_embed=txt, image_embed=img)

            # Perform backpropagation and update the EMA model

            trainer.update()

            # Gather information about the current training step

            all_loss        = pad_gather_reduce(trainer, loss,     method="mean")
            num_samples     = pad_gather_reduce(trainer, len(txt), method="sum")
            samples_per_sec = num_samples / samples_timer.elapsed()
            samples_seen   += num_samples
            if config.train.use_ema == True:
                ema_decay       = trainer.ema_diffusion_prior.get_current_decay()

            # Log the gathered information

            if (current_step+1) % config.train.log_every_steps == 0:
                if config.train.use_ema == True:
                    tracker.log(
                        {
                            "tracking/samples-sec":                        samples_per_sec.item(),
                            "tracking/samples-seen":                       samples_seen.item(),
                            "tracking/ema-decay":                          ema_decay,
                            f"tracking/training-{config.prior.loss_type}": all_loss.item(),
                        },
                        step = current_step+1,
                    )
                else:
                    tracker.log(
                        {
                            "tracking/samples-sec":                        samples_per_sec.item(),
                            "tracking/samples-seen":                       samples_seen.item(),
                            f"tracking/training-{config.prior.loss_type}": all_loss.item(),
                        },
                        step = current_step+1,
                    )

                print(f'epoch {epoch+1} | step {str(current_step+1).zfill(6)} | ', end = '')
                print(f'samples/sec: {samples_per_sec.item() :.6f} | ', end = '')
                print(f'samples seen: {str(samples_seen.item()).zfill(7)} | ', end = '')
                if config.train.use_ema == True:
                    print(f'ema decay: {ema_decay :.6f} | ', end = '')
                print(f'L2 loss: {all_loss.item() :.6f}')

            # Save the latest model ......................................................

            if (current_step+1) % config.train.save_every_steps == 0:
                save_trainer(
                    trainer              = trainer,
                    tracker              = tracker,
                    is_best              = False,
                    is_latest            = True,
                    samples_seen         = samples_seen,
                    epoch                = epoch,
                    best_validation_loss = best_validation_loss,
                )

            # Track metrics at regular intervals through validation .....................

            if (current_step+1) % config.train.eval_every_steps == 0:

                # Begin measuring for how long validation lasts
                validation_profiler.reset()

                # Pack kwargs for running model  evaluation

                eval_kwargs = {
                    "trainer":          trainer,
                    "tracker":          tracker,
                    "text_conditioned": config.prior.condition_on_text_encodings,
                    "timesteps":        config.train.eval_timesteps,
                }

                # Evaluate the model being trained on the validation set

                val_loss = prior_eval(
                    dataloader    = eval_loader,
                    iterations    = val_iterations,
                    loss_type     = config.prior.loss_type,
                    split         = "validation",
                    use_ema       = False,
                    report_cosine = not config.train.use_ema,
                    report_loss   = True,
                    **eval_kwargs,
                )

                # Evaluate the EMA model on the validation set

                if config.train.use_ema == True:
                    val_loss = prior_eval(
                        dataloader    = eval_loader,
                        iterations    = val_iterations,
                        loss_type     = config.prior.loss_type,
                        split         = "validation",
                        use_ema       = True,
                        report_cosine = True,
                        report_loss   = True,
                        **eval_kwargs,
                    )

                val_duration = validation_profiler.elapsed() / 60
                tracker.log(
                    {
                    "tracking/validation duration (minutes)": val_duration
                    }
                )
                print(f'[INFO] validation duration: {val_duration} minutes')

                # Check if the EMA validation loss is the lowest loss seen yet

                if val_loss is not None:
                    if val_loss < best_validation_loss:
                        best_validation_loss = val_loss

                        # If it is the best loss, save the model as the best one found
                        save_trainer(
                            trainer              = trainer,
                            tracker              = tracker,
                            is_best              = True,
                            is_latest            = False,
                            samples_seen         = samples_seen,
                            epoch                = epoch,
                            best_validation_loss = best_validation_loss,
                        )
                # ...................... end of validation ......................

            # .......................... end of epoch ...........................

        # ............................. end of training .........................

    # Evaluate the prior model on the test set ..................................

    if trainer.accelerator.is_main_process:
        print(f"[INFO] Start testing")

    # Save the latest snapshot of the model before validation

    save_trainer(
        tracker              = tracker,
        trainer              = trainer,
        is_best              = False,
        is_latest            = True,
        samples_seen         = samples_seen,
        epoch                = epoch,
        best_validation_loss = best_validation_loss,
    )

    test_loss = prior_eval(
        trainer          = trainer,
        dataloader       = test_loader,
        iterations       = test_iterations,
        text_conditioned = config.prior.condition_on_text_encodings,
        split            = "test",
        tracker          = tracker,
        use_ema          = config.train.use_ema,
        report_cosine    = False,
        report_loss      = True,
        timesteps        = config.train.eval_timesteps,
        loss_type        = config.prior.loss_type,
    )

    if test_loss < best_validation_loss:
        best_validation_loss = test_loss

        #  If loss is the lowest one, save the model as the best found

        save_trainer(
            trainer              = trainer,
            tracker              = tracker,
            is_best              = True,
            is_latest            = False,
            samples_seen         = samples_seen,
            epoch                = epoch,
            best_validation_loss = test_loss,
        )

In [ ]:
def initialize_prior_training(config_file, accelerator, device, inference_device):
    """
    Parse the configuration file, and prepare everything necessary
    for prior training.
    """
    # load the configuration file
    if accelerator.is_main_process:
        print(f"[INFO] Loading configuration from {config_file}")

    config = TrainDiffusionPriorConfig.from_json_path(config_file)

    # seed

    set_seed(config.train.random_seed)

    # get a device

    device = accelerator.device

    # Instantiate the model trainer (will automatically distribute if possible & configured)

    trainer: DiffusionPriorTrainer = make_model(
        config.prior,
        config.train,
        device,
        accelerator,
    ).to(device)

    print(f'[INFO] Instantiated the model trainer')

    # create a tracker

    tracker = create_tracker(
        accelerator,
        config,
        config_file,
        dummy = accelerator.process_index != 0
    )

    print(f'[INFO] Created the training tracker')

    # reload from checkpoint

    if tracker.can_recall:
        current_epoch, best_validation_loss, samples_seen = recall_trainer(
            tracker = tracker,
            trainer = trainer
        )

        # display the best values
        if trainer.accelerator.is_main_process:
            print(f'[INFO] A checkpoint was loaded from file')
            print(f'[INFO] Checkpoint current epoch: {current_epoch} | best val loss: {best_validation_loss} | samples seen: {samples_seen}')

        # update config to reflect recalled values
        config.train.num_samples_seen     = samples_seen
        config.train.current_epoch        = current_epoch
        config.train.best_validation_loss = best_validation_loss
    else:
        print(f'[WARN] No checkpoint was loaded from file')

    # fetch and prepare data

    if trainer.accelerator.is_main_process:
        print('[INFO] Grabbing data...')

    trainer.accelerator.wait_for_everyone()
    img_reader, txt_reader = get_reader(
        text_conditioned = trainer.text_conditioned,
        img_url          = config.data.image_url,
        meta_url         = config.data.meta_url,
        txt_url          = config.data.txt_url,
    )

    # calculate the starting point within epoch

    trainer.accelerator.wait_for_everyone()

    train_loader, eval_loader, test_loader = make_splits(
        text_conditioned = trainer.text_conditioned,
        batch_size       = config.data.batch_size,
        num_data_points  = config.data.num_data_points,
        train_split      = config.data.splits.train,
        eval_split       = config.data.splits.val,
        image_reader     = img_reader,
        text_reader      = txt_reader,
        rank             = accelerator.state.process_index,
        world_size       = accelerator.state.num_processes,
        start            = 0,
    )

    # update the starting point to finish out the epoch on a resumed run

    if tracker.can_recall:
        samples_seen = config.train.num_samples_seen
        length = (
            config.data.num_data_points
            if samples_seen <= img_reader.count
            else img_reader.count
        )
        scaled_samples = length * config.train.current_epoch
        start_point = (
            scaled_samples - samples_seen if scaled_samples > samples_seen else samples_seen
        )

        if trainer.accelerator.is_main_process:
           print(f"[INFO] Resuming at sample {start_point}")

        train_loader.dataset.set_start(start_point)

    # start training

    if trainer.accelerator.is_main_process:
        print(
            f"[INFO] Beginning prior training with distributed={accelerator.state.distributed_type != accelerate_dataclasses.DistributedType.NO}")

    prior_train(
        trainer      = trainer,
        tracker      = tracker,
        train_loader = train_loader,
        eval_loader  = eval_loader,
        test_loader  = test_loader,
        config       = config,
    )

In [ ]:
def run_prior_train(config_file, device, inference_device):
    # Start Hugging Face Accelerate
    accelerator = Accelerator()

    # Setup prior training
    initialize_prior_training(config_file, accelerator, device, inference_device)

In [ ]:
# Run the prior training
run_prior_train(config_file, device, inference_device)